In [0]:
"""
id: source_1
template: source
templateVersion: 2.0.0
name: resgates
position:
  x: 0
  y: 310
description:
  text: Load data from a specified table.
  hash: ce30b81f
previewCodeHash: 5517cc29706e91a2
codegenHash: f9bd80fc93b51b62
config:
  table_source:
    tableName: vertem.workshop_dataprep.resgates
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "vertem.workshop_dataprep.resgates"
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_1.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_1" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_1.data"])
if "source_1" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "source_1", "counts": {"data": int(ctx["source_1.data"].count())}}}, raw=True)

In [0]:
"""
id: source_2
template: source
templateVersion: 2.0.0
name: clientes_b2b
position:
  x: 300
  y: 0
description:
  text: Load all data from the specified table in the workspace.
  hash: f822a183
previewCodeHash: afd06a8ddb2e9c0d
codegenHash: f9bd80fc93b51b62
config:
  table_source:
    tableName: vertem.workshop_dataprep.clientes_b2b
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "vertem.workshop_dataprep.clientes_b2b"
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_2.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_2" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_2.data"])
if "source_2" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "source_2", "counts": {"data": int(ctx["source_2.data"].count())}}}, raw=True)

In [0]:
"""
id: source_22
template: source
templateVersion: 2.0.0
name: excel-meta
position:
  x: 3900
  y: 155
description:
  text: Read data from an Excel file with headers.
  hash: 1f123a41
previewCodeHash: db28e37e4ef8e18f
codegenHash: f9bd80fc93b51b62
config:
  file_source:
    path: /Volumes/databricks/home/fc/fernando_custodio/my_files/metas_resgate_2025.xlsx
    format: '"excel"'
    schemaEvolutionMode: '"none"'
    headerRows: true
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "file_source": {
        "path": "/Volumes/databricks/home/fc/fernando_custodio/my_files/metas_resgate_2025.xlsx",
        "format": "\"excel\"",
        "schemaEvolutionMode": "\"none\"",
        "headerRows": True
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_22.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_22" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_22.data"])
if "source_22" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "source_22", "counts": {"data": int(ctx["source_22.data"].count())}}}, raw=True)

In [0]:
"""
id: source_4
template: source
templateVersion: 2.0.0
name: parceiros_resgate
position:
  x: 300
  y: 155
description:
  text: Load all data from the parceiros_resgate table.
  hash: ee6074c8
previewCodeHash: 589ddc3c8272e6e5
codegenHash: f9bd80fc93b51b62
config:
  table_source:
    tableName: vertem.workshop_dataprep.parceiros_resgate
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "vertem.workshop_dataprep.parceiros_resgate"
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_4.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_4" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_4.data"])
if "source_4" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "source_4", "counts": {"data": int(ctx["source_4.data"].count())}}}, raw=True)

In [0]:
"""
id: source_7
template: source
templateVersion: 2.0.0
name: campanhas
position:
  x: 0
  y: 465
description:
  text: Load data from the table 'vertem.workshop_dataprep.campanhas'.
  hash: 0da968f4
previewCodeHash: bc250afcb493c931
codegenHash: f9bd80fc93b51b62
config:
  table_source:
    tableName: vertem.workshop_dataprep.campanhas
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "vertem.workshop_dataprep.campanhas"
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_7.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_7" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_7.data"])
if "source_7" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "source_7", "counts": {"data": int(ctx["source_7.data"].count())}}}, raw=True)

In [0]:
"""
id: prepare_1
template: prepare
templateVersion: 2.0.0
name: data-resgate
position:
  x: 300
  y: 310
description:
  text: Convert 'data_resgate' to date, handling 'dd/MM/yyyy' format and other date castings.
  hash: 60c24344
previewCodeHash: dd79a9e16a8a3594
codegenHash: d318165b9d528af4
config:
  actions:
    - type: formula
      target: data_resgate
      expression: |
        CASE
          WHEN data_resgate LIKE '__/__/____'
          THEN to_date(data_resgate, 'dd/MM/yyyy')
          ELSE try_cast(data_resgate AS DATE)
        END
input:
  - node: source_1
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Any, Callable, Dict, List, Optional, Tuple
import json
import re
import pyspark.sql.functions as F
import pyspark.sql.types as T

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

CURRENT_COLUMN_PLACEHOLDER = "{current_column}"

JSON_PARSE_MODES = ("name_value", "typed_value", "flatten_array", "unnest")

JSON_PARSE_OUTPUT_SUFFIXES: Dict[str, List[str]] = {
    "name_value": ["_Name", "_ValueString"],
    "typed_value": [
        "_Name",
        "_ValueString",
        "_ValueInt",
        "_ValueFloat",
        "_ValueBool",
    ],
    "flatten_array": ["_flatten", "_idx"],
}

JSON_PARSE_SEPARATOR = "_"

JSON_VARIANT_DTYPE = "variant"

JSON_UNNEST_MAX_KEYS = 1024

INT64_MIN = -(2 ** 63)
INT64_MAX = 2 ** 63 - 1

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _action_columns(action: Dict[str, Any], action_type: str) -> List[str]:
    """Resolve an action's target columns from either accepted key.

    The "columns" list wins when it names anything, else the scalar
    "column". An empty list therefore can't mask a valid scalar. Empty
    names are dropped and repeats collapsed, since applying an action
    twice to a column is a no-op. Names are used verbatim (not
    stripped): a column identifier can legally carry edge spaces, so
    trimming would look up a different column than the one saved.
    Mirrors readColumnSelection in the notebook parser so the UI and
    the job resolve the same selection.
    """

    def _names(raw: Any) -> List[str]:
        if isinstance(raw, str):
            raw = [raw]
        if not isinstance(raw, (list, tuple)):
            raise ValueError(
                f"{action_type}: 'columns' must be a list of column names, "
                f"got {type(raw).__name__}"
            )
        names: List[str] = []
        for entry in raw:
            if not isinstance(entry, str):
                raise ValueError(
                    f"{action_type}: column names must be strings, got {entry!r}"
                )
            if entry and entry not in names:
                names.append(entry)
        return names

    columns = _names(action.get("columns", []))
    if not columns:
        columns = _names(action.get("column", ""))
    if not columns:
        raise ValueError(f"{action_type}: 'column' or 'columns' is required")
    return columns

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _quoted_or_comment_end(expression: str, start: int) -> int:
    end = len(expression)
    opening = expression[start]
    if opening in ("'", '"', "`"):
        index = start + 1
        while index < end:
            if expression[index] == "\\" and opening != "`":
                index += 2
                continue
            if expression[index] == opening:
                if index + 1 < end and expression[index + 1] == opening:
                    index += 2
                    continue
                return index + 1
            index += 1
        return end
    if expression.startswith("--", start):
        line_end = expression.find("\n", start)
        return end if line_end == -1 else line_end
    if expression.startswith("/*", start):
        comment_end = expression.find("*/", start + 2)
        return end if comment_end == -1 else comment_end + 2
    return start

def _placeholder_positions(expression: str) -> List[int]:
    positions: List[int] = []
    index = 0
    end = len(expression)
    while index < end:
        if expression.startswith(CURRENT_COLUMN_PLACEHOLDER, index):
            positions.append(index)
            index += len(CURRENT_COLUMN_PLACEHOLDER)
            continue
        skip_to = _quoted_or_comment_end(expression, index)
        index = skip_to if skip_to > index else index + 1
    return positions

def _expand_placeholders(
    expression: str, positions: List[int], quoted_column: str
) -> str:
    rendered: List[str] = []
    cursor = 0
    for position in positions:
        rendered.append(expression[cursor:position])
        rendered.append(quoted_column)
        cursor = position + len(CURRENT_COLUMN_PLACEHOLDER)
    rendered.append(expression[cursor:])
    return "".join(rendered)

def _apply_multi_column_formula(df, action: Dict[str, Any]):
    columns = action.get("columns") or []
    expression = action.get("expression", "")
    prefix = str(action.get("prefix") or "").strip()
    suffix = str(action.get("suffix") or "").strip()
    if isinstance(columns, str) or not isinstance(columns, (list, tuple)):
        raise ValueError(
            "multi_column_formula: 'columns' must be a list of column names, "
            f"got {type(columns).__name__}"
        )
    if not columns:
        raise ValueError("multi_column_formula: 'columns' is required")
    if not expression:
        raise ValueError("multi_column_formula: 'expression' is required")
    positions = _placeholder_positions(expression)
    if not positions:
        raise ValueError(
            "multi_column_formula: 'expression' must contain the "
            + CURRENT_COLUMN_PLACEHOLDER
            + " placeholder outside of string literals, quoted identifiers "
            + "and comments, e.g. '"
            + CURRENT_COLUMN_PLACEHOLDER
            + " * 1.1'"
        )
    assignments: Dict[str, Any] = {}
    for raw_column in columns:
        column = "" if raw_column is None else str(raw_column).strip()
        if not column:
            raise ValueError(
                "multi_column_formula: 'columns' must not contain blank entries"
            )
        _require_column(df, column, "multi_column_formula")
        rendered = _expand_placeholders(
            expression, positions, _quoted_ident(column)
        )
        assignments[f"{prefix}{column}{suffix}"] = F.expr(rendered)
    return df.withColumns(assignments)

def _apply_cast(df, action: Dict[str, Any], column: str):
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return F.expr(f"try_cast({_quoted_ident(column)} as {to_type})")
    return _col(column).cast(to_type)

def _apply_replace_value(df, action: Dict[str, Any], column: str):
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return F.when(predicate, replacement).otherwise(_col(column))

def _apply_fill_null(df, action: Dict[str, Any], column: str):
    with_val = action.get("with", "")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return F.when(_col(column).isNull(), replacement).otherwise(_col(column))

def _apply_text_case(df, action: Dict[str, Any], column: str):
    case = action.get("case", "")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_trim(df, action: Dict[str, Any], column: str):
    side = action.get("side", "both")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_regex_replace(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    return F.regexp_replace(_col(column), pattern, replacement)

def _apply_extract(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    group_raw = action.get("group", 0)
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return F.regexp_extract(_col(column), pattern, group_idx)

def _extract_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    """Destination for one extract application.

    Several extractions cannot share one destination, so "target" is
    honoured for a single source and ignored beyond that, where every
    source is rewritten in place.
    """
    if len(columns) > 1:
        return column
    return action.get("target") or column

def _apply_parse_date(df, action: Dict[str, Any], column: str):
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return F.try_to_timestamp(_col(column), F.lit(fmt))
            return F.try_to_timestamp(_col(column))
        if fmt:
            return F.to_date(_col(column), fmt)
        return F.expr(f"try_cast({_quoted_ident(column)} as date)")
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return fn(_col(column), fmt)
    return fn(_col(column))

def _unique_temp_name(df, base: str) -> str:
    """Name for an intermediate column that cannot shadow user data."""
    existing = set(df.columns)
    if base not in existing:
        return base
    suffix = 2
    while base + "_" + str(suffix) in existing:
        suffix += 1
    return base + "_" + str(suffix)

JSON_DECODE_ERRORS = (ValueError, TypeError, RecursionError)

JSON_PARSE_CONTAINERS = {
    "flatten_array": (list, "array"),
    "unnest": (dict, "object"),
}

def _json_container(text: Any, column: str, mode: str, strict: bool):
    """Decode one document that must be the container the mode expects.
    Returns None when the text is missing, undecodable or the wrong shape;
    raises for the latter two when strict."""
    expected, label = JSON_PARSE_CONTAINERS[mode]
    if _json_document_missing(text):
        return None
    try:
        document = json.loads(text)
    except JSON_DECODE_ERRORS as exc:
        if strict:
            raise ValueError(
                "parse_json: invalid JSON in column '" + column + "': " + str(exc)
            ) from exc
        return None
    if isinstance(document, expected):
        return document
    if strict:
        raise ValueError(
            "parse_json: mode '" + mode + "' expects a JSON " + label
            + " in column '" + column + "', got " + type(document).__name__
        )
    return None

def _json_leaf_pairs(text: str) -> List[Tuple[str, Any]]:
    """Depth-first (path, scalar) pairs for one JSON document.

    Paths use dotted notation with array positions as segments
    ("phoneNumber.0.type"). An empty object or array yields a single
    (path, None) pair so the row still records that the key existed.

    The walk is iterative rather than recursive so that traversal depth is
    bounded by memory, not by the interpreter's recursion limit. (json's own
    decoder still recurses, so documents nested deeper than that limit are
    reported as undecodable — see JSON_DECODE_ERRORS.)
    """
    pairs: List[Tuple[str, Any]] = []
    stack: List[Tuple[str, Any]] = [("", json.loads(text))]
    while stack:
        path, node = stack.pop()
        if isinstance(node, dict):
            if not node:
                pairs.append((path, None))
                continue
            for key in reversed(list(node.keys())):
                child = str(key) if not path else path + "." + str(key)
                stack.append((child, node[key]))
        elif isinstance(node, list):
            if not node:
                pairs.append((path, None))
                continue
            for index in range(len(node) - 1, -1, -1):
                child = str(index) if not path else path + "." + str(index)
                stack.append((child, node[index]))
        else:
            pairs.append((path, node))
    return pairs

def _json_text(value: Any) -> Optional[str]:
    """Render a JSON value as text: strings verbatim, everything else as
    compact JSON so booleans read "true"/"false" and containers keep their
    original form."""
    if value is None:
        return None
    if value is True:
        return "true"
    if value is False:
        return "false"
    if isinstance(value, str):
        return value
    return json.dumps(value, separators=(",", ":"), ensure_ascii=False)

def _reject_json_constant(name: str) -> Any:
    raise ValueError("parse_json: refusing to read " + name + " as a number")

JSON_SCALAR_LEADS = "-0123456789tf"

def _json_scalar_from_text(text: str) -> Any:
    """Reinterpret a quoted scalar as the value it spells, or None when it
    spells nothing.

    Plenty of documents quote everything ({"active": "true", "age": "25"}),
    which would otherwise pile every leaf into the string column and leave
    the typed ones permanently empty. The test is JSON's own literal
    grammar, applied to the whole text: "25" is a number because JSON says
    so, while "007", "+5", "1_000" and " 25 " are not, and so keep their
    exact text rather than being silently rewritten. Only bool/int/float
    count -- "null" stays the word, never a missing value, and Infinity/NaN
    are refused outright since JSON has no such literals.
    """
    if not text or text[0] not in JSON_SCALAR_LEADS or text[-1].isspace():
        return None
    try:
        parsed = json.loads(text, parse_constant=_reject_json_constant)
    except JSON_DECODE_ERRORS:
        return None
    if parsed is True or parsed is False or isinstance(parsed, (int, float)):
        return parsed
    return None

def _json_typed(value: Any) -> Tuple[Any, Any, Any, Any]:
    """Spread a JSON scalar across (string, int, float, bool).

    Exactly one slot is filled, picked from the value's type, so an integer
    never also appears as a float. Integers too wide for BIGINT fall back to
    the string slot instead of overflowing. A quoted scalar is routed by
    what it spells rather than by its quoting, so {"a": true} and
    {"a": "true"} land in the same column -- see _json_scalar_from_text.
    """
    if value is None:
        return (None, None, None, None)
    if value is True or value is False:
        return (None, None, None, bool(value))
    if isinstance(value, str):
        spelled = _json_scalar_from_text(value)
        if spelled is None:
            return (value, None, None, None)
        return _json_typed(spelled)
    if isinstance(value, int):
        if INT64_MIN <= value <= INT64_MAX:
            return (None, value, None, None)
        return (str(value), None, None, None)
    if isinstance(value, float):
        return (None, None, value, None)
    return (_json_text(value), None, None, None)

def _json_document_missing(text: Any) -> bool:
    """NULL and blank text mean "no document" rather than "bad document", so
    they yield NULL outputs even when on_error is "error"."""
    return text is None or (isinstance(text, str) and not text.strip())

def _prefixed_output(prefix: str, suffix: str) -> str:
    """Name a generated column. With no prefix the separator goes too, so
    the column is "Name" rather than "_Name"."""
    if prefix:
        return prefix + suffix
    if suffix.startswith(JSON_PARSE_SEPARATOR):
        return suffix[len(JSON_PARSE_SEPARATOR):]
    return suffix

def _json_source_column(df, column: str):
    """The source column, as the JSON text a mode's UDF expects.

    A VARIANT column holds an already-parsed document rather than its text,
    so render it back with to_json. Any other type is handed over untouched:
    string is text already, and nothing else is offered as a source.
    """
    if _column_dtype(df, column) == JSON_VARIANT_DTYPE:
        return F.to_json(_col(column))
    return _col(column)

def _require_free_columns(outputs: List[str], retained: List[str]) -> None:
    collisions = [name for name in outputs if name in retained]
    if collisions:
        raise ValueError(
            f"parse_json: generated column(s) {collisions} would overwrite existing "
            f"columns. Set a different 'prefix'."
        )

def _parse_json_pairs(df, column: str, source, mode: str, outputs: List[str],
                      retained: List[str], strict: bool):
    typed = mode == "typed_value"
    fields = [
        T.StructField("name", T.StringType()),
        T.StructField("value_string", T.StringType()),
    ]
    if typed:
        fields.extend([
            T.StructField("value_int", T.LongType()),
            T.StructField("value_float", T.DoubleType()),
            T.StructField("value_bool", T.BooleanType()),
        ])

    def to_pairs(text):
        if _json_document_missing(text):
            return None
        try:
            pairs = _json_leaf_pairs(text)
        except JSON_DECODE_ERRORS as exc:
            if strict:
                raise ValueError(
                    "parse_json: invalid JSON in column '" + column + "': " + str(exc)
                ) from exc
            return None
        if typed:
            return [(path,) + _json_typed(value) for path, value in pairs]
        return [(path, _json_text(value)) for path, value in pairs]

    pairs_udf = F.udf(to_pairs, T.ArrayType(T.StructType(fields)))
    exploded = _unique_temp_name(df, "__parse_json")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.explode_outer(pairs_udf(source)).alias(exploded),
    ).select(
        *kept,
        *[
            _col(exploded).getField(field.name).alias(output)
            for field, output in zip(fields, outputs)
        ],
    )

def _parse_json_flatten_array(df, column: str, source, outputs: List[str],
                              retained: List[str], strict: bool):
    def to_elements(text):
        document = _json_container(text, column, "flatten_array", strict)
        if document is None:
            return None
        return [_json_text(element) for element in document]

    elements_udf = F.udf(to_elements, T.ArrayType(T.StringType()))
    position = _unique_temp_name(df, "__parse_json_pos")
    element = _unique_temp_name(df, "__parse_json_val")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.posexplode_outer(elements_udf(source)).alias(position, element),
    ).select(
        *kept,
        _col(element).alias(outputs[0]),
        (_col(position) + F.lit(1)).cast("int").alias(outputs[1]),
    )

def _parse_json_unnest(df, column: str, source, prefix: str,
                       retained: List[str], strict: bool):
    """Alteryx 'Unnest JSON field': one column per top-level key, each value
    kept as JSON text. One level only -- unnest again to go deeper.

    Alone among the modes, the output columns come from the data, so this
    runs one extra Spark job to discover the key names before it can build
    the projection. Keys are ordered by their first position within a
    document (ties broken by name) so the resulting schema is document-order
    like Alteryx's, yet stable across runs and partitionings.
    """
    def to_keys(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [(str(key), index) for index, key in enumerate(document.keys())]

    keys_udf = F.udf(
        to_keys,
        T.ArrayType(
            T.StructType([
                T.StructField("key", T.StringType()),
                T.StructField("position", T.IntegerType()),
            ])
        ),
    )
    found = _unique_temp_name(df, "__parse_json_key")
    discovered = (
        df.select(F.explode(keys_udf(source)).alias(found))
        .select(
            _col(found).getField("key").alias("key"),
            _col(found).getField("position").alias("position"),
        )
        .groupBy("key")
        .agg(F.min("position").alias("position"))
        .limit(JSON_UNNEST_MAX_KEYS + 1)
        .collect()
    )
    if len(discovered) > JSON_UNNEST_MAX_KEYS:
        raise ValueError(
            f"parse_json: mode 'unnest' found more than {JSON_UNNEST_MAX_KEYS} "
            f"distinct keys in column '{column}'. Keys that vary per row (ids, "
            f"timestamps) belong in rows, not columns -- use mode 'name_value' "
            f"for those."
        )
    keys = [
        row["key"]
        for row in sorted(discovered, key=lambda row: (row["position"], row["key"]))
    ]
    if not keys:
        raise ValueError(
            f"parse_json: mode 'unnest' found no keys to unnest in column '{column}'. "
            f"Every value was NULL, blank, or not a JSON object."
        )
    outputs = [prefix + "." + key if prefix else key for key in keys]
    _require_free_columns(outputs, retained)

    def to_values(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [_json_text(document.get(key)) for key in keys]

    values_udf = F.udf(to_values, T.ArrayType(T.StringType()))
    values = _unique_temp_name(df, "__parse_json_values")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        values_udf(source).alias(values),
    ).select(
        *kept,
        *[
            _col(values).getItem(index).alias(output)
            for index, output in enumerate(outputs)
        ],
    )

def _apply_parse_json(df, action: Dict[str, Any]):
    """Explode JSON into rows or columns, mirroring Alteryx's JSON Parse
    tool.

    Python UDFs on purpose: from_json / schema_of_json would need the
    document's schema at plan time, and inferring it means eagerly scanning
    the data, which would both cost a job per action and make the operator's
    output schema depend on the current data. The "unnest" mode accepts
    exactly that cost because its column names ARE the data; every other
    mode stays lazy. VARIANT does not change this -- see the parse_json
    design notes at the top of this file for why each mode reads text.
    """
    column = action.get("column", "")
    mode = action.get("mode", "name_value")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_json")
    if mode not in JSON_PARSE_MODES:
        raise ValueError(
            f"parse_json: unsupported mode '{mode}'. "
            f"Choose one of: {sorted(JSON_PARSE_MODES)}"
        )
    if on_error not in ("null", "error"):
        raise ValueError(
            f"parse_json: unsupported on_error '{on_error}'. Choose null or error."
        )
    prefix = (action.get("prefix") or "").strip()
    keep_source = action.get("keep_source", True) is not False
    retained = [name for name in df.columns if keep_source or name != column]
    strict = on_error == "error"
    source = _json_source_column(df, column)
    if mode == "unnest":
        return _parse_json_unnest(df, column, source, prefix, retained, strict)
    outputs = [
        _prefixed_output(prefix, suffix)
        for suffix in JSON_PARSE_OUTPUT_SUFFIXES[mode]
    ]
    _require_free_columns(outputs, retained)
    if mode == "flatten_array":
        return _parse_json_flatten_array(df, column, source, outputs, retained, strict)
    return _parse_json_pairs(df, column, source, mode, outputs, retained, strict)

VALID_RENAME_TRANSFORMS = {
    "add_prefix",
    "add_suffix",
    "lower",
    "upper",
    "trim",
    "remove_whitespace",
    "snake_case",
    "title_case",
    "sentence_case",
    "camel_case",
    "replace",
}

def _rename_words(name: str) -> List[str]:
    return [word for word in re.split(r"[^A-Za-z0-9]+", name) if word]

def _capitalize_word(word: str) -> str:
    return word[:1].upper() + word[1:].lower()

def _renamed(name: str, action: Dict[str, Any]) -> str:
    transform = action.get("transform", "")
    if transform == "add_prefix":
        return action.get("text", "") + name
    if transform == "add_suffix":
        return name + action.get("text", "")
    if transform == "lower":
        return name.lower()
    if transform == "upper":
        return name.upper()
    if transform == "trim":
        return name.strip()
    if transform == "remove_whitespace":
        return re.sub(r"\s+", "", name)
    if transform == "snake_case":
        return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    if transform == "title_case":
        return " ".join(_capitalize_word(word) for word in _rename_words(name))
    if transform == "sentence_case":
        words = _rename_words(name)
        if not words:
            return ""
        return " ".join([_capitalize_word(words[0])] + [word.lower() for word in words[1:]])
    if transform == "camel_case":
        words = _rename_words(name)
        if not words:
            return ""
        return words[0].lower() + "".join(_capitalize_word(word) for word in words[1:])
    if transform == "replace":
        find = action.get("find", "")
        if not find:
            return name
        return name.replace(find, action.get("replace_with", ""))
    return name

def _require_no_merged_columns(
    old_names: List[str], new_names: List[str]
) -> None:
    sources_by_name: Dict[str, List[str]] = {}
    display_by_name: Dict[str, str] = {}
    for old, new in zip(old_names, new_names):
        key = new.lower()
        sources_by_name.setdefault(key, []).append(old)
        display_by_name.setdefault(key, new)
    merged = [
        (display_by_name[key], sorted(sources))
        for key, sources in sources_by_name.items()
        if len({source.lower() for source in sources}) > 1
    ]
    if merged:
        details = "; ".join(
            f"{sources} -> {new!r}" for new, sources in sorted(merged)
        )
        raise ValueError(
            "rename_columns: the rename would give distinct columns the "
            f"same name ({details}). Spark matches column names "
            "case-insensitively. Narrow 'columns' or pick another transform."
        )

def _apply_rename_columns(df, action: Dict[str, Any]):
    transform = action.get("transform", "")
    if transform not in VALID_RENAME_TRANSFORMS:
        raise ValueError(
            f"rename_columns: unsupported transform '{transform}'. "
            f"Choose one of: {sorted(VALID_RENAME_TRANSFORMS)}"
        )
    targets = action.get("columns") or []
    if not isinstance(targets, list):
        raise ValueError(
            "rename_columns: 'columns' must be a list of column names"
        )
    old_names = list(df.columns)
    missing = [name for name in targets if name not in old_names]
    if missing:
        raise ValueError(
            f"rename_columns: column(s) {missing} not found in input data. "
            f"Available columns: {old_names}"
        )
    selected = set(targets)
    new_names = [
        _renamed(name, action) if not selected or name in selected else name
        for name in old_names
    ]
    blank = [old for old, new in zip(old_names, new_names) if not new]
    if blank:
        raise ValueError(
            f"rename_columns: transform '{transform}' leaves column(s) "
            f"{blank} with an empty name."
        )
    _require_no_merged_columns(old_names, new_names)
    if new_names == old_names:
        return df
    return df.toDF(*new_names)

COLUMN_ACTION_DISPATCH: Dict[str, Callable] = {
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def _in_place_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    return column

ACTION_TARGET_RESOLVERS: Dict[str, Callable] = {
    "extract": _extract_target,
}

# formula names its own output, multi_column_formula writes one output per
# selected column, parse_json reshapes rows/columns, and rename_columns
# rewrites the column names, so all four sit outside the single-expression,
# in-place column dispatch and are routed on their own; every other action
# targets one column or several via the shared column dispatch below.
SUPPORTED_ACTION_TYPES = sorted(
    [
        "formula",
        "multi_column_formula",
        "parse_json",
        "rename_columns",
        *COLUMN_ACTION_DISPATCH,
    ]
)

def _apply_column_action(df, action: Dict[str, Any], action_type: str):
    """Apply one column-scoped action across all of its target columns.

    Every expression is built against the same input DataFrame and
    applied in one withColumns pass. Equivalent to chaining a withColumn
    per column, since no builder reads a column another target in the
    same action rewrites, but it adds one Project instead of N.
    """
    columns = _action_columns(action, action_type)
    build_expression = COLUMN_ACTION_DISPATCH[action_type]
    resolve_target = ACTION_TARGET_RESOLVERS.get(action_type, _in_place_target)
    updates: Dict[str, Any] = {}
    for column in columns:
        _require_column(df, column, action_type)
        target = resolve_target(action, column, columns)
        updates[target] = build_expression(df, action, column)
    return df.withColumns(updates)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        if action_type == "formula":
            df = _apply_formula(df, action)
        elif action_type == "multi_column_formula":
            df = _apply_multi_column_formula(df, action)
        elif action_type == "parse_json":
            df = _apply_parse_json(df, action)
        elif action_type == "rename_columns":
            df = _apply_rename_columns(df, action)
        elif action_type in COLUMN_ACTION_DISPATCH:
            df = _apply_column_action(df, action, action_type)
        else:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {SUPPORTED_ACTION_TYPES}"
            )
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "target": "data_resgate",
            "expression": "CASE\n  WHEN data_resgate LIKE '__/__/____'\n  THEN to_date(data_resgate, 'dd/MM/yyyy')\n  ELSE try_cast(data_resgate AS DATE)\nEND\n"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["source_1.data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_1.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_1" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_1.prepared_data"])
if "prepare_1" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "prepare_1", "counts": {"prepared_data": int(ctx["prepare_1.prepared_data"].count())}}}, raw=True)

In [0]:
"""
id: unique_8
template: unique
templateVersion: 1.0.0
name: distinct
position:
  x: 300
  y: 465
previewCodeHash: 97e6d4960e5176b4
codegenHash: f37e4cbd06314ad6
config:
  unique_by_all_columns: true
  columns: []
  sort_expressions: []
input:
  - node: source_7
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F
from pyspark.sql import Window

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    if df is None:
        raise ValueError("Unique operator requires input 'data'")
    unique_by_all_columns = config.get("unique_by_all_columns", True)
    columns = config.get("columns", [])
    if not isinstance(columns, list):
        columns = []
    else:
        columns = [c for c in columns if isinstance(c, str)]
    sort_expressions = config.get("sort_expressions", [])

    if unique_by_all_columns:
        columns = df.columns
    elif not columns:
        return {"unique_data": df}

    order_idx = "__lb_orig_order__"
    while order_idx in df.columns:
        order_idx = order_idx + "_"
    df = df.withColumn(order_idx, F.monotonically_increasing_id())

    if unique_by_all_columns or not sort_expressions:
        deduped = df.dropDuplicates(columns)
        return {"unique_data": deduped.orderBy(order_idx).drop(order_idx)}

    order_cols = []
    for sort_def in sort_expressions:
        if not isinstance(sort_def, dict):
            continue
        col_expr = sort_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        if not raw_expr:
            continue
        direction = sort_def.get("sortBy", "ASC")
        col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        if direction == "DESC":
            col = col.desc_nulls_last()
        elif direction == "ASC":
            col = col.asc_nulls_last()
        order_cols.append(col)
    
    if not order_cols:
        deduped = df.dropDuplicates(columns)
        return {"unique_data": deduped.orderBy(order_idx).drop(order_idx)}

    rn = "__lb_row_number__"
    while rn in df.columns:
        rn = rn + "_"

    window = Window.partitionBy(*[F.col(c) for c in columns]).orderBy(*order_cols, F.col(order_idx))
    tagged = df.withColumn(rn, F.row_number().over(window))
    unique = tagged.filter(F.col(rn) == 1).orderBy(order_idx).drop(rn, order_idx)
    return {"unique_data": unique}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "unique_by_all_columns": True,
    "columns": [],
    "sort_expressions": []
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["source_7.data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["unique_8.unique_data"] = out["unique_data"]
if globals().get("ld_display_outputs", False) or "unique_8" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["unique_8.unique_data"])
if "unique_8" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "unique_8", "counts": {"unique_data": int(ctx["unique_8.unique_data"].count())}}}, raw=True)

In [0]:
"""
id: join_3
template: join
templateVersion: 3.0.0
name: j-resgate
position:
  x: 600
  y: 155
description:
  text: Join multiple tables on specified keys with option to keep unmatched rows from the left or right.
  hash: ec12623b
previewCodeHash: 23c912478c7549b7
codegenHash: ea5821d113ebbb3b
config:
  join_type: left
  match_case: false
  multi_table: true
  conditions:
    - left_input: 1
      right_input: 0
      join_keys:
        - left: id_cliente
          right: id_cliente
      join_conditions: ""
    - left_input: 2
      right_input: 1
      join_keys:
        - left: id_parceiro
          right: id_parceiro
      join_conditions: ""
  columns:
    - edits: []
      ordered: []
    - edits: []
      ordered: []
    - edits: []
      ordered: []
input:
  - node: source_2
    input_port: data
    output_port: data
  - node: prepare_1
    input_port: data
    output_port: prepared_data
  - node: source_4
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Any, Dict, List, Optional
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _qualified_col(alias: str, name: str):
    escaped = name.replace("`", "``")
    return F.col("`" + alias + "`.`" + escaped + "`")

def _ordered_names(df_cols: List[str], cfg: Dict[str, Any]) -> List[str]:
    ordered: List[str] = cfg.get("ordered") or []
    col_set = set(df_cols)
    placed = set()
    result: List[str] = []
    for name in ordered:
        if name in placed or name not in col_set:
            continue
        placed.add(name)
        result.append(name)
    for name in df_cols:
        if name in placed:
            continue
        placed.add(name)
        result.append(name)
    return result

def _edits_by_column(cfg: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    by_col: Dict[str, Dict[str, Any]] = {}
    for item in cfg.get("edits", []) or []:
        col = item.get("column")
        if col is not None and col not in by_col:
            by_col[col] = item
    return by_col

def _projection(frames: List[Any]):
    out = []
    seen: Dict[str, int] = {}
    n = len(frames)
    for idx, (alias, df, cfg) in enumerate(frames):
        edits = _edits_by_column(cfg)
        for name in _ordered_names(list(df.columns), cfg):
            edit = edits.get(name)
            if edit is not None and not _is_checked(edit):
                continue
            col = _qualified_col(alias, name)
            user_alias = edit.get("alias") if edit else None
            if user_alias:
                out.append(col.alias(user_alias))
                seen[user_alias.lower()] = seen.get(user_alias.lower(), 0) + 1
                continue
            key = name.lower()
            if key not in seen:
                seen[key] = 1
                out.append(col.alias(name))
                continue
            new_name = ("right_" + name) if (n == 2 and idx == 1) else (name + "_" + str(idx + 1))
            while new_name.lower() in seen:
                new_name = new_name + "_dup"
            seen[new_name.lower()] = 1
            out.append(col.alias(new_name))
    return out

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    dfs: List[Any] = []
    for port in ("left", "right"):
        df = inputs.get(port)
        if df is not None:
            dfs.append(df)
    dfs.extend(inputs.get("data") or [])
    if len(dfs) < 2:
        raise ValueError(
            f"Join requires at least 2 inputs, got {len(dfs)}. "
            "Wire 2 or more upstream operators into this Join."
        )

    n = len(dfs)
    join_type = config.get("join_type") or "split_join"
    match_case = bool(config.get("match_case", False))
    conditions: List[Dict[str, Any]] = config.get("conditions") or []
    columns: List[Dict[str, Any]] = config.get("columns") or []

    aliased = [df.alias("t" + str(i)) for i, df in enumerate(dfs)]
    types = [{f.name.lower(): f.dataType for f in df.schema} for df in aliased]

    def key_col(idx: int, name: str):
        col = _qualified_col("t" + str(idx), name)
        if not match_case and isinstance(types[idx].get(name.lower()), StringType):
            return F.upper(F.trim(col))
        return col

    def edge_predicate(i: int, j: int, edge: Dict[str, Any]) -> Optional[Any]:
        predicates = []
        for key in edge.get("join_keys") or []:
            predicates.append(key_col(i, key["left"]) == key_col(j, key["right"]))
        condition = edge.get("join_conditions") or ""
        if condition:
            predicates.append(F.expr(condition))
        join_expr = None
        for predicate in predicates:
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    edges: List[Any] = [
        (int(edge.get("left_input", 0)), int(edge.get("right_input", 0)), edge)
        for edge in conditions
    ]

    predicates_by_step: Dict[int, List[Any]] = {}
    for i, j, edge in edges:
        predicate = edge_predicate(i, j, edge)
        if predicate is not None:
            predicates_by_step.setdefault(max(i, j), []).append(predicate)

    def predicate_at(step: int) -> Optional[Any]:
        join_expr = None
        for predicate in predicates_by_step.get(step, []):
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    def cfg(i: int) -> Dict[str, Any]:
        return columns[i] if i < len(columns) else {}

    is_two = n == 2
    is_split = is_two and join_type == "split_join"

    if is_split:
        join_expr = predicate_at(1)
        left_df, right_df = aliased[0], aliased[1]
        if join_expr is None:
            matched = left_df.join(right_df, how="inner")
            left_unmatched = left_df.join(right_df, how="left_anti")
            right_unmatched = right_df.join(left_df, how="left_anti")
        else:
            matched = left_df.join(right_df, join_expr, how="inner")
            left_unmatched = left_df.join(right_df, join_expr, how="left_anti")
            right_unmatched = right_df.join(left_df, join_expr, how="left_anti")
        projection = _projection([("t0", left_df, cfg(0)), ("t1", right_df, cfg(1))])
        if projection:
            matched = matched.select(*projection)
        return {
            "joined_data": matched,
            "left_unmatched": left_unmatched,
            "right_unmatched": right_unmatched,
        }

    how = join_type if is_two else "inner"
    if is_two:
        acc = aliased[0]
        predicate = predicate_at(1)
        if predicate is None:
            acc = acc.join(aliased[1], how=how)
        else:
            acc = acc.join(aliased[1], predicate, how=how)
    else:
        adjacency: Dict[int, List[int]] = {}
        for idx, (i, j, _edge) in enumerate(edges):
            adjacency.setdefault(i, []).append(idx)
            adjacency.setdefault(j, []).append(idx)
        visited = [False] * n
        visited[0] = True
        acc = aliased[0]
        used_edges = set()
        queue: List[int] = [0]
        while queue:
            cur = queue.pop(0)
            for idx in adjacency.get(cur, []):
                i, j, edge = edges[idx]
                nxt = j if i == cur else i
                if visited[nxt]:
                    continue
                visited[nxt] = True
                used_edges.add(idx)
                predicate = edge_predicate(i, j, edge)
                if predicate is None:
                    acc = acc.join(aliased[nxt], how=how)
                else:
                    acc = acc.join(aliased[nxt], predicate, how=how)
                queue.append(nxt)
        for idx, (i, j, edge) in enumerate(edges):
            if idx in used_edges:
                continue
            predicate = edge_predicate(i, j, edge)
            if predicate is not None:
                acc = acc.where(predicate)

    projection = _projection([("t" + str(i), aliased[i], cfg(i)) for i in range(n)])
    if projection:
        acc = acc.select(*projection)

    return {
        "joined_data": acc,
        "left_unmatched": spark.createDataFrame([], aliased[0].schema),
        "right_unmatched": spark.createDataFrame([], aliased[n - 1].schema),
    }

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "match_case": False,
    "multi_table": True,
    "conditions": [
        {
            "left_input": 1,
            "right_input": 0,
            "join_keys": [
                {
                    "left": "id_cliente",
                    "right": "id_cliente"
                }
            ],
            "join_conditions": ""
        },
        {
            "left_input": 2,
            "right_input": 1,
            "join_keys": [
                {
                    "left": "id_parceiro",
                    "right": "id_parceiro"
                }
            ],
            "join_conditions": ""
        }
    ],
    "columns": [
        {
            "edits": [],
            "ordered": []
        },
        {
            "edits": [],
            "ordered": []
        },
        {
            "edits": [],
            "ordered": []
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": [
        ctx["source_2.data"],
        ctx["prepare_1.prepared_data"],
        ctx["source_4.data"]
    ],
    "data__sources": [
        {
            "node": "source_2",
            "output_port": "data",
            "name": "clientes_b2b",
            "df_name": "clientes_b2b"
        },
        {
            "node": "prepare_1",
            "output_port": "prepared_data",
            "name": "data-resgate",
            "df_name": "data_resgate"
        },
        {
            "node": "source_4",
            "output_port": "data",
            "name": "parceiros_resgate",
            "df_name": "parceiros_resgate"
        }
    ]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["join_3.joined_data"] = out["joined_data"]
ctx["join_3.left_unmatched"] = out["left_unmatched"]
ctx["join_3.right_unmatched"] = out["right_unmatched"]
if globals().get("ld_display_outputs", False) or "join_3" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["join_3.joined_data"])
    display(ctx["join_3.left_unmatched"])
    display(ctx["join_3.right_unmatched"])
if "join_3" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "join_3", "counts": {"joined_data": int(ctx["join_3.joined_data"].count()), "left_unmatched": int(ctx["join_3.left_unmatched"].count()), "right_unmatched": int(ctx["join_3.right_unmatched"].count())}}}, raw=True)

In [0]:
"""
id: prepare_8
template: prepare
templateVersion: 2.0.0
name: p-datas-campanha
position:
  x: 600
  y: 465
description:
  text: Convert 'data_inicio' and 'data_fim' columns to date format handling two different date patterns.
  hash: e6f9858e
previewCodeHash: 52b3b635fb9d2a6a
codegenHash: d318165b9d528af4
config:
  actions:
    - type: formula
      target: data_inicio
      expression: |
        CASE
          WHEN data_inicio LIKE '__/__/____'
          THEN to_date(data_inicio, 'dd/MM/yyyy')
          ELSE try_cast(data_inicio AS DATE)
        END
    - type: formula
      target: data_fim
      expression: |
        CASE
          WHEN data_fim LIKE '__/__/____'
          THEN to_date(data_fim, 'dd/MM/yyyy')
          ELSE try_cast(data_fim AS DATE)
        END
input:
  - node: unique_8
    input_port: data
    output_port: unique_data
"""

# generated from the system
from typing import Any, Callable, Dict, List, Optional, Tuple
import json
import re
import pyspark.sql.functions as F
import pyspark.sql.types as T

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

CURRENT_COLUMN_PLACEHOLDER = "{current_column}"

JSON_PARSE_MODES = ("name_value", "typed_value", "flatten_array", "unnest")

JSON_PARSE_OUTPUT_SUFFIXES: Dict[str, List[str]] = {
    "name_value": ["_Name", "_ValueString"],
    "typed_value": [
        "_Name",
        "_ValueString",
        "_ValueInt",
        "_ValueFloat",
        "_ValueBool",
    ],
    "flatten_array": ["_flatten", "_idx"],
}

JSON_PARSE_SEPARATOR = "_"

JSON_VARIANT_DTYPE = "variant"

JSON_UNNEST_MAX_KEYS = 1024

INT64_MIN = -(2 ** 63)
INT64_MAX = 2 ** 63 - 1

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _action_columns(action: Dict[str, Any], action_type: str) -> List[str]:
    """Resolve an action's target columns from either accepted key.

    The "columns" list wins when it names anything, else the scalar
    "column". An empty list therefore can't mask a valid scalar. Empty
    names are dropped and repeats collapsed, since applying an action
    twice to a column is a no-op. Names are used verbatim (not
    stripped): a column identifier can legally carry edge spaces, so
    trimming would look up a different column than the one saved.
    Mirrors readColumnSelection in the notebook parser so the UI and
    the job resolve the same selection.
    """

    def _names(raw: Any) -> List[str]:
        if isinstance(raw, str):
            raw = [raw]
        if not isinstance(raw, (list, tuple)):
            raise ValueError(
                f"{action_type}: 'columns' must be a list of column names, "
                f"got {type(raw).__name__}"
            )
        names: List[str] = []
        for entry in raw:
            if not isinstance(entry, str):
                raise ValueError(
                    f"{action_type}: column names must be strings, got {entry!r}"
                )
            if entry and entry not in names:
                names.append(entry)
        return names

    columns = _names(action.get("columns", []))
    if not columns:
        columns = _names(action.get("column", ""))
    if not columns:
        raise ValueError(f"{action_type}: 'column' or 'columns' is required")
    return columns

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _quoted_or_comment_end(expression: str, start: int) -> int:
    end = len(expression)
    opening = expression[start]
    if opening in ("'", '"', "`"):
        index = start + 1
        while index < end:
            if expression[index] == "\\" and opening != "`":
                index += 2
                continue
            if expression[index] == opening:
                if index + 1 < end and expression[index + 1] == opening:
                    index += 2
                    continue
                return index + 1
            index += 1
        return end
    if expression.startswith("--", start):
        line_end = expression.find("\n", start)
        return end if line_end == -1 else line_end
    if expression.startswith("/*", start):
        comment_end = expression.find("*/", start + 2)
        return end if comment_end == -1 else comment_end + 2
    return start

def _placeholder_positions(expression: str) -> List[int]:
    positions: List[int] = []
    index = 0
    end = len(expression)
    while index < end:
        if expression.startswith(CURRENT_COLUMN_PLACEHOLDER, index):
            positions.append(index)
            index += len(CURRENT_COLUMN_PLACEHOLDER)
            continue
        skip_to = _quoted_or_comment_end(expression, index)
        index = skip_to if skip_to > index else index + 1
    return positions

def _expand_placeholders(
    expression: str, positions: List[int], quoted_column: str
) -> str:
    rendered: List[str] = []
    cursor = 0
    for position in positions:
        rendered.append(expression[cursor:position])
        rendered.append(quoted_column)
        cursor = position + len(CURRENT_COLUMN_PLACEHOLDER)
    rendered.append(expression[cursor:])
    return "".join(rendered)

def _apply_multi_column_formula(df, action: Dict[str, Any]):
    columns = action.get("columns") or []
    expression = action.get("expression", "")
    prefix = str(action.get("prefix") or "").strip()
    suffix = str(action.get("suffix") or "").strip()
    if isinstance(columns, str) or not isinstance(columns, (list, tuple)):
        raise ValueError(
            "multi_column_formula: 'columns' must be a list of column names, "
            f"got {type(columns).__name__}"
        )
    if not columns:
        raise ValueError("multi_column_formula: 'columns' is required")
    if not expression:
        raise ValueError("multi_column_formula: 'expression' is required")
    positions = _placeholder_positions(expression)
    if not positions:
        raise ValueError(
            "multi_column_formula: 'expression' must contain the "
            + CURRENT_COLUMN_PLACEHOLDER
            + " placeholder outside of string literals, quoted identifiers "
            + "and comments, e.g. '"
            + CURRENT_COLUMN_PLACEHOLDER
            + " * 1.1'"
        )
    assignments: Dict[str, Any] = {}
    for raw_column in columns:
        column = "" if raw_column is None else str(raw_column).strip()
        if not column:
            raise ValueError(
                "multi_column_formula: 'columns' must not contain blank entries"
            )
        _require_column(df, column, "multi_column_formula")
        rendered = _expand_placeholders(
            expression, positions, _quoted_ident(column)
        )
        assignments[f"{prefix}{column}{suffix}"] = F.expr(rendered)
    return df.withColumns(assignments)

def _apply_cast(df, action: Dict[str, Any], column: str):
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return F.expr(f"try_cast({_quoted_ident(column)} as {to_type})")
    return _col(column).cast(to_type)

def _apply_replace_value(df, action: Dict[str, Any], column: str):
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return F.when(predicate, replacement).otherwise(_col(column))

def _apply_fill_null(df, action: Dict[str, Any], column: str):
    with_val = action.get("with", "")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return F.when(_col(column).isNull(), replacement).otherwise(_col(column))

def _apply_text_case(df, action: Dict[str, Any], column: str):
    case = action.get("case", "")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_trim(df, action: Dict[str, Any], column: str):
    side = action.get("side", "both")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_regex_replace(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    return F.regexp_replace(_col(column), pattern, replacement)

def _apply_extract(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    group_raw = action.get("group", 0)
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return F.regexp_extract(_col(column), pattern, group_idx)

def _extract_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    """Destination for one extract application.

    Several extractions cannot share one destination, so "target" is
    honoured for a single source and ignored beyond that, where every
    source is rewritten in place.
    """
    if len(columns) > 1:
        return column
    return action.get("target") or column

def _apply_parse_date(df, action: Dict[str, Any], column: str):
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return F.try_to_timestamp(_col(column), F.lit(fmt))
            return F.try_to_timestamp(_col(column))
        if fmt:
            return F.to_date(_col(column), fmt)
        return F.expr(f"try_cast({_quoted_ident(column)} as date)")
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return fn(_col(column), fmt)
    return fn(_col(column))

def _unique_temp_name(df, base: str) -> str:
    """Name for an intermediate column that cannot shadow user data."""
    existing = set(df.columns)
    if base not in existing:
        return base
    suffix = 2
    while base + "_" + str(suffix) in existing:
        suffix += 1
    return base + "_" + str(suffix)

JSON_DECODE_ERRORS = (ValueError, TypeError, RecursionError)

JSON_PARSE_CONTAINERS = {
    "flatten_array": (list, "array"),
    "unnest": (dict, "object"),
}

def _json_container(text: Any, column: str, mode: str, strict: bool):
    """Decode one document that must be the container the mode expects.
    Returns None when the text is missing, undecodable or the wrong shape;
    raises for the latter two when strict."""
    expected, label = JSON_PARSE_CONTAINERS[mode]
    if _json_document_missing(text):
        return None
    try:
        document = json.loads(text)
    except JSON_DECODE_ERRORS as exc:
        if strict:
            raise ValueError(
                "parse_json: invalid JSON in column '" + column + "': " + str(exc)
            ) from exc
        return None
    if isinstance(document, expected):
        return document
    if strict:
        raise ValueError(
            "parse_json: mode '" + mode + "' expects a JSON " + label
            + " in column '" + column + "', got " + type(document).__name__
        )
    return None

def _json_leaf_pairs(text: str) -> List[Tuple[str, Any]]:
    """Depth-first (path, scalar) pairs for one JSON document.

    Paths use dotted notation with array positions as segments
    ("phoneNumber.0.type"). An empty object or array yields a single
    (path, None) pair so the row still records that the key existed.

    The walk is iterative rather than recursive so that traversal depth is
    bounded by memory, not by the interpreter's recursion limit. (json's own
    decoder still recurses, so documents nested deeper than that limit are
    reported as undecodable — see JSON_DECODE_ERRORS.)
    """
    pairs: List[Tuple[str, Any]] = []
    stack: List[Tuple[str, Any]] = [("", json.loads(text))]
    while stack:
        path, node = stack.pop()
        if isinstance(node, dict):
            if not node:
                pairs.append((path, None))
                continue
            for key in reversed(list(node.keys())):
                child = str(key) if not path else path + "." + str(key)
                stack.append((child, node[key]))
        elif isinstance(node, list):
            if not node:
                pairs.append((path, None))
                continue
            for index in range(len(node) - 1, -1, -1):
                child = str(index) if not path else path + "." + str(index)
                stack.append((child, node[index]))
        else:
            pairs.append((path, node))
    return pairs

def _json_text(value: Any) -> Optional[str]:
    """Render a JSON value as text: strings verbatim, everything else as
    compact JSON so booleans read "true"/"false" and containers keep their
    original form."""
    if value is None:
        return None
    if value is True:
        return "true"
    if value is False:
        return "false"
    if isinstance(value, str):
        return value
    return json.dumps(value, separators=(",", ":"), ensure_ascii=False)

def _reject_json_constant(name: str) -> Any:
    raise ValueError("parse_json: refusing to read " + name + " as a number")

JSON_SCALAR_LEADS = "-0123456789tf"

def _json_scalar_from_text(text: str) -> Any:
    """Reinterpret a quoted scalar as the value it spells, or None when it
    spells nothing.

    Plenty of documents quote everything ({"active": "true", "age": "25"}),
    which would otherwise pile every leaf into the string column and leave
    the typed ones permanently empty. The test is JSON's own literal
    grammar, applied to the whole text: "25" is a number because JSON says
    so, while "007", "+5", "1_000" and " 25 " are not, and so keep their
    exact text rather than being silently rewritten. Only bool/int/float
    count -- "null" stays the word, never a missing value, and Infinity/NaN
    are refused outright since JSON has no such literals.
    """
    if not text or text[0] not in JSON_SCALAR_LEADS or text[-1].isspace():
        return None
    try:
        parsed = json.loads(text, parse_constant=_reject_json_constant)
    except JSON_DECODE_ERRORS:
        return None
    if parsed is True or parsed is False or isinstance(parsed, (int, float)):
        return parsed
    return None

def _json_typed(value: Any) -> Tuple[Any, Any, Any, Any]:
    """Spread a JSON scalar across (string, int, float, bool).

    Exactly one slot is filled, picked from the value's type, so an integer
    never also appears as a float. Integers too wide for BIGINT fall back to
    the string slot instead of overflowing. A quoted scalar is routed by
    what it spells rather than by its quoting, so {"a": true} and
    {"a": "true"} land in the same column -- see _json_scalar_from_text.
    """
    if value is None:
        return (None, None, None, None)
    if value is True or value is False:
        return (None, None, None, bool(value))
    if isinstance(value, str):
        spelled = _json_scalar_from_text(value)
        if spelled is None:
            return (value, None, None, None)
        return _json_typed(spelled)
    if isinstance(value, int):
        if INT64_MIN <= value <= INT64_MAX:
            return (None, value, None, None)
        return (str(value), None, None, None)
    if isinstance(value, float):
        return (None, None, value, None)
    return (_json_text(value), None, None, None)

def _json_document_missing(text: Any) -> bool:
    """NULL and blank text mean "no document" rather than "bad document", so
    they yield NULL outputs even when on_error is "error"."""
    return text is None or (isinstance(text, str) and not text.strip())

def _prefixed_output(prefix: str, suffix: str) -> str:
    """Name a generated column. With no prefix the separator goes too, so
    the column is "Name" rather than "_Name"."""
    if prefix:
        return prefix + suffix
    if suffix.startswith(JSON_PARSE_SEPARATOR):
        return suffix[len(JSON_PARSE_SEPARATOR):]
    return suffix

def _json_source_column(df, column: str):
    """The source column, as the JSON text a mode's UDF expects.

    A VARIANT column holds an already-parsed document rather than its text,
    so render it back with to_json. Any other type is handed over untouched:
    string is text already, and nothing else is offered as a source.
    """
    if _column_dtype(df, column) == JSON_VARIANT_DTYPE:
        return F.to_json(_col(column))
    return _col(column)

def _require_free_columns(outputs: List[str], retained: List[str]) -> None:
    collisions = [name for name in outputs if name in retained]
    if collisions:
        raise ValueError(
            f"parse_json: generated column(s) {collisions} would overwrite existing "
            f"columns. Set a different 'prefix'."
        )

def _parse_json_pairs(df, column: str, source, mode: str, outputs: List[str],
                      retained: List[str], strict: bool):
    typed = mode == "typed_value"
    fields = [
        T.StructField("name", T.StringType()),
        T.StructField("value_string", T.StringType()),
    ]
    if typed:
        fields.extend([
            T.StructField("value_int", T.LongType()),
            T.StructField("value_float", T.DoubleType()),
            T.StructField("value_bool", T.BooleanType()),
        ])

    def to_pairs(text):
        if _json_document_missing(text):
            return None
        try:
            pairs = _json_leaf_pairs(text)
        except JSON_DECODE_ERRORS as exc:
            if strict:
                raise ValueError(
                    "parse_json: invalid JSON in column '" + column + "': " + str(exc)
                ) from exc
            return None
        if typed:
            return [(path,) + _json_typed(value) for path, value in pairs]
        return [(path, _json_text(value)) for path, value in pairs]

    pairs_udf = F.udf(to_pairs, T.ArrayType(T.StructType(fields)))
    exploded = _unique_temp_name(df, "__parse_json")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.explode_outer(pairs_udf(source)).alias(exploded),
    ).select(
        *kept,
        *[
            _col(exploded).getField(field.name).alias(output)
            for field, output in zip(fields, outputs)
        ],
    )

def _parse_json_flatten_array(df, column: str, source, outputs: List[str],
                              retained: List[str], strict: bool):
    def to_elements(text):
        document = _json_container(text, column, "flatten_array", strict)
        if document is None:
            return None
        return [_json_text(element) for element in document]

    elements_udf = F.udf(to_elements, T.ArrayType(T.StringType()))
    position = _unique_temp_name(df, "__parse_json_pos")
    element = _unique_temp_name(df, "__parse_json_val")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.posexplode_outer(elements_udf(source)).alias(position, element),
    ).select(
        *kept,
        _col(element).alias(outputs[0]),
        (_col(position) + F.lit(1)).cast("int").alias(outputs[1]),
    )

def _parse_json_unnest(df, column: str, source, prefix: str,
                       retained: List[str], strict: bool):
    """Alteryx 'Unnest JSON field': one column per top-level key, each value
    kept as JSON text. One level only -- unnest again to go deeper.

    Alone among the modes, the output columns come from the data, so this
    runs one extra Spark job to discover the key names before it can build
    the projection. Keys are ordered by their first position within a
    document (ties broken by name) so the resulting schema is document-order
    like Alteryx's, yet stable across runs and partitionings.
    """
    def to_keys(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [(str(key), index) for index, key in enumerate(document.keys())]

    keys_udf = F.udf(
        to_keys,
        T.ArrayType(
            T.StructType([
                T.StructField("key", T.StringType()),
                T.StructField("position", T.IntegerType()),
            ])
        ),
    )
    found = _unique_temp_name(df, "__parse_json_key")
    discovered = (
        df.select(F.explode(keys_udf(source)).alias(found))
        .select(
            _col(found).getField("key").alias("key"),
            _col(found).getField("position").alias("position"),
        )
        .groupBy("key")
        .agg(F.min("position").alias("position"))
        .limit(JSON_UNNEST_MAX_KEYS + 1)
        .collect()
    )
    if len(discovered) > JSON_UNNEST_MAX_KEYS:
        raise ValueError(
            f"parse_json: mode 'unnest' found more than {JSON_UNNEST_MAX_KEYS} "
            f"distinct keys in column '{column}'. Keys that vary per row (ids, "
            f"timestamps) belong in rows, not columns -- use mode 'name_value' "
            f"for those."
        )
    keys = [
        row["key"]
        for row in sorted(discovered, key=lambda row: (row["position"], row["key"]))
    ]
    if not keys:
        raise ValueError(
            f"parse_json: mode 'unnest' found no keys to unnest in column '{column}'. "
            f"Every value was NULL, blank, or not a JSON object."
        )
    outputs = [prefix + "." + key if prefix else key for key in keys]
    _require_free_columns(outputs, retained)

    def to_values(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [_json_text(document.get(key)) for key in keys]

    values_udf = F.udf(to_values, T.ArrayType(T.StringType()))
    values = _unique_temp_name(df, "__parse_json_values")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        values_udf(source).alias(values),
    ).select(
        *kept,
        *[
            _col(values).getItem(index).alias(output)
            for index, output in enumerate(outputs)
        ],
    )

def _apply_parse_json(df, action: Dict[str, Any]):
    """Explode JSON into rows or columns, mirroring Alteryx's JSON Parse
    tool.

    Python UDFs on purpose: from_json / schema_of_json would need the
    document's schema at plan time, and inferring it means eagerly scanning
    the data, which would both cost a job per action and make the operator's
    output schema depend on the current data. The "unnest" mode accepts
    exactly that cost because its column names ARE the data; every other
    mode stays lazy. VARIANT does not change this -- see the parse_json
    design notes at the top of this file for why each mode reads text.
    """
    column = action.get("column", "")
    mode = action.get("mode", "name_value")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_json")
    if mode not in JSON_PARSE_MODES:
        raise ValueError(
            f"parse_json: unsupported mode '{mode}'. "
            f"Choose one of: {sorted(JSON_PARSE_MODES)}"
        )
    if on_error not in ("null", "error"):
        raise ValueError(
            f"parse_json: unsupported on_error '{on_error}'. Choose null or error."
        )
    prefix = (action.get("prefix") or "").strip()
    keep_source = action.get("keep_source", True) is not False
    retained = [name for name in df.columns if keep_source or name != column]
    strict = on_error == "error"
    source = _json_source_column(df, column)
    if mode == "unnest":
        return _parse_json_unnest(df, column, source, prefix, retained, strict)
    outputs = [
        _prefixed_output(prefix, suffix)
        for suffix in JSON_PARSE_OUTPUT_SUFFIXES[mode]
    ]
    _require_free_columns(outputs, retained)
    if mode == "flatten_array":
        return _parse_json_flatten_array(df, column, source, outputs, retained, strict)
    return _parse_json_pairs(df, column, source, mode, outputs, retained, strict)

VALID_RENAME_TRANSFORMS = {
    "add_prefix",
    "add_suffix",
    "lower",
    "upper",
    "trim",
    "remove_whitespace",
    "snake_case",
    "title_case",
    "sentence_case",
    "camel_case",
    "replace",
}

def _rename_words(name: str) -> List[str]:
    return [word for word in re.split(r"[^A-Za-z0-9]+", name) if word]

def _capitalize_word(word: str) -> str:
    return word[:1].upper() + word[1:].lower()

def _renamed(name: str, action: Dict[str, Any]) -> str:
    transform = action.get("transform", "")
    if transform == "add_prefix":
        return action.get("text", "") + name
    if transform == "add_suffix":
        return name + action.get("text", "")
    if transform == "lower":
        return name.lower()
    if transform == "upper":
        return name.upper()
    if transform == "trim":
        return name.strip()
    if transform == "remove_whitespace":
        return re.sub(r"\s+", "", name)
    if transform == "snake_case":
        return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    if transform == "title_case":
        return " ".join(_capitalize_word(word) for word in _rename_words(name))
    if transform == "sentence_case":
        words = _rename_words(name)
        if not words:
            return ""
        return " ".join([_capitalize_word(words[0])] + [word.lower() for word in words[1:]])
    if transform == "camel_case":
        words = _rename_words(name)
        if not words:
            return ""
        return words[0].lower() + "".join(_capitalize_word(word) for word in words[1:])
    if transform == "replace":
        find = action.get("find", "")
        if not find:
            return name
        return name.replace(find, action.get("replace_with", ""))
    return name

def _require_no_merged_columns(
    old_names: List[str], new_names: List[str]
) -> None:
    sources_by_name: Dict[str, List[str]] = {}
    display_by_name: Dict[str, str] = {}
    for old, new in zip(old_names, new_names):
        key = new.lower()
        sources_by_name.setdefault(key, []).append(old)
        display_by_name.setdefault(key, new)
    merged = [
        (display_by_name[key], sorted(sources))
        for key, sources in sources_by_name.items()
        if len({source.lower() for source in sources}) > 1
    ]
    if merged:
        details = "; ".join(
            f"{sources} -> {new!r}" for new, sources in sorted(merged)
        )
        raise ValueError(
            "rename_columns: the rename would give distinct columns the "
            f"same name ({details}). Spark matches column names "
            "case-insensitively. Narrow 'columns' or pick another transform."
        )

def _apply_rename_columns(df, action: Dict[str, Any]):
    transform = action.get("transform", "")
    if transform not in VALID_RENAME_TRANSFORMS:
        raise ValueError(
            f"rename_columns: unsupported transform '{transform}'. "
            f"Choose one of: {sorted(VALID_RENAME_TRANSFORMS)}"
        )
    targets = action.get("columns") or []
    if not isinstance(targets, list):
        raise ValueError(
            "rename_columns: 'columns' must be a list of column names"
        )
    old_names = list(df.columns)
    missing = [name for name in targets if name not in old_names]
    if missing:
        raise ValueError(
            f"rename_columns: column(s) {missing} not found in input data. "
            f"Available columns: {old_names}"
        )
    selected = set(targets)
    new_names = [
        _renamed(name, action) if not selected or name in selected else name
        for name in old_names
    ]
    blank = [old for old, new in zip(old_names, new_names) if not new]
    if blank:
        raise ValueError(
            f"rename_columns: transform '{transform}' leaves column(s) "
            f"{blank} with an empty name."
        )
    _require_no_merged_columns(old_names, new_names)
    if new_names == old_names:
        return df
    return df.toDF(*new_names)

COLUMN_ACTION_DISPATCH: Dict[str, Callable] = {
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def _in_place_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    return column

ACTION_TARGET_RESOLVERS: Dict[str, Callable] = {
    "extract": _extract_target,
}

# formula names its own output, multi_column_formula writes one output per
# selected column, parse_json reshapes rows/columns, and rename_columns
# rewrites the column names, so all four sit outside the single-expression,
# in-place column dispatch and are routed on their own; every other action
# targets one column or several via the shared column dispatch below.
SUPPORTED_ACTION_TYPES = sorted(
    [
        "formula",
        "multi_column_formula",
        "parse_json",
        "rename_columns",
        *COLUMN_ACTION_DISPATCH,
    ]
)

def _apply_column_action(df, action: Dict[str, Any], action_type: str):
    """Apply one column-scoped action across all of its target columns.

    Every expression is built against the same input DataFrame and
    applied in one withColumns pass. Equivalent to chaining a withColumn
    per column, since no builder reads a column another target in the
    same action rewrites, but it adds one Project instead of N.
    """
    columns = _action_columns(action, action_type)
    build_expression = COLUMN_ACTION_DISPATCH[action_type]
    resolve_target = ACTION_TARGET_RESOLVERS.get(action_type, _in_place_target)
    updates: Dict[str, Any] = {}
    for column in columns:
        _require_column(df, column, action_type)
        target = resolve_target(action, column, columns)
        updates[target] = build_expression(df, action, column)
    return df.withColumns(updates)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        if action_type == "formula":
            df = _apply_formula(df, action)
        elif action_type == "multi_column_formula":
            df = _apply_multi_column_formula(df, action)
        elif action_type == "parse_json":
            df = _apply_parse_json(df, action)
        elif action_type == "rename_columns":
            df = _apply_rename_columns(df, action)
        elif action_type in COLUMN_ACTION_DISPATCH:
            df = _apply_column_action(df, action, action_type)
        else:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {SUPPORTED_ACTION_TYPES}"
            )
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "target": "data_inicio",
            "expression": "CASE\n  WHEN data_inicio LIKE '__/__/____'\n  THEN to_date(data_inicio, 'dd/MM/yyyy')\n  ELSE try_cast(data_inicio AS DATE)\nEND\n"
        },
        {
            "type": "formula",
            "target": "data_fim",
            "expression": "CASE\n  WHEN data_fim LIKE '__/__/____'\n  THEN to_date(data_fim, 'dd/MM/yyyy')\n  ELSE try_cast(data_fim AS DATE)\nEND\n"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["unique_8.unique_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_8.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_8" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_8.prepared_data"])
if "prepare_8" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "prepare_8", "counts": {"prepared_data": int(ctx["prepare_8.prepared_data"].count())}}}, raw=True)

In [0]:
"""
id: join_8
template: join
templateVersion: 3.0.0
name: j-resgate-campanhas
position:
  x: 900
  y: 310
description:
  text: Join two datasets on client ID where redemption date is between start and end dates, keeping all from the left and matching from the right.
  hash: ddfad492
previewCodeHash: 0ab8dcf2e93fd20f
codegenHash: ea5821d113ebbb3b
config:
  join_type: left
  match_case: false
  multi_table: false
  conditions:
    - left_input: 0
      right_input: 1
      join_keys:
        - left: id_cliente
          right: id_cliente
      join_conditions: data_resgate BETWEEN data_inicio AND data_fim
  columns:
    - edits: []
      ordered: []
    - edits: []
      ordered: []
input:
  - node: join_3
    input_port: left
    output_port: joined_data
  - node: prepare_8
    input_port: right
    output_port: prepared_data
"""

# generated from the system
from typing import Any, Dict, List, Optional
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _qualified_col(alias: str, name: str):
    escaped = name.replace("`", "``")
    return F.col("`" + alias + "`.`" + escaped + "`")

def _ordered_names(df_cols: List[str], cfg: Dict[str, Any]) -> List[str]:
    ordered: List[str] = cfg.get("ordered") or []
    col_set = set(df_cols)
    placed = set()
    result: List[str] = []
    for name in ordered:
        if name in placed or name not in col_set:
            continue
        placed.add(name)
        result.append(name)
    for name in df_cols:
        if name in placed:
            continue
        placed.add(name)
        result.append(name)
    return result

def _edits_by_column(cfg: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    by_col: Dict[str, Dict[str, Any]] = {}
    for item in cfg.get("edits", []) or []:
        col = item.get("column")
        if col is not None and col not in by_col:
            by_col[col] = item
    return by_col

def _projection(frames: List[Any]):
    out = []
    seen: Dict[str, int] = {}
    n = len(frames)
    for idx, (alias, df, cfg) in enumerate(frames):
        edits = _edits_by_column(cfg)
        for name in _ordered_names(list(df.columns), cfg):
            edit = edits.get(name)
            if edit is not None and not _is_checked(edit):
                continue
            col = _qualified_col(alias, name)
            user_alias = edit.get("alias") if edit else None
            if user_alias:
                out.append(col.alias(user_alias))
                seen[user_alias.lower()] = seen.get(user_alias.lower(), 0) + 1
                continue
            key = name.lower()
            if key not in seen:
                seen[key] = 1
                out.append(col.alias(name))
                continue
            new_name = ("right_" + name) if (n == 2 and idx == 1) else (name + "_" + str(idx + 1))
            while new_name.lower() in seen:
                new_name = new_name + "_dup"
            seen[new_name.lower()] = 1
            out.append(col.alias(new_name))
    return out

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    dfs: List[Any] = []
    for port in ("left", "right"):
        df = inputs.get(port)
        if df is not None:
            dfs.append(df)
    dfs.extend(inputs.get("data") or [])
    if len(dfs) < 2:
        raise ValueError(
            f"Join requires at least 2 inputs, got {len(dfs)}. "
            "Wire 2 or more upstream operators into this Join."
        )

    n = len(dfs)
    join_type = config.get("join_type") or "split_join"
    match_case = bool(config.get("match_case", False))
    conditions: List[Dict[str, Any]] = config.get("conditions") or []
    columns: List[Dict[str, Any]] = config.get("columns") or []

    aliased = [df.alias("t" + str(i)) for i, df in enumerate(dfs)]
    types = [{f.name.lower(): f.dataType for f in df.schema} for df in aliased]

    def key_col(idx: int, name: str):
        col = _qualified_col("t" + str(idx), name)
        if not match_case and isinstance(types[idx].get(name.lower()), StringType):
            return F.upper(F.trim(col))
        return col

    def edge_predicate(i: int, j: int, edge: Dict[str, Any]) -> Optional[Any]:
        predicates = []
        for key in edge.get("join_keys") or []:
            predicates.append(key_col(i, key["left"]) == key_col(j, key["right"]))
        condition = edge.get("join_conditions") or ""
        if condition:
            predicates.append(F.expr(condition))
        join_expr = None
        for predicate in predicates:
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    edges: List[Any] = [
        (int(edge.get("left_input", 0)), int(edge.get("right_input", 0)), edge)
        for edge in conditions
    ]

    predicates_by_step: Dict[int, List[Any]] = {}
    for i, j, edge in edges:
        predicate = edge_predicate(i, j, edge)
        if predicate is not None:
            predicates_by_step.setdefault(max(i, j), []).append(predicate)

    def predicate_at(step: int) -> Optional[Any]:
        join_expr = None
        for predicate in predicates_by_step.get(step, []):
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    def cfg(i: int) -> Dict[str, Any]:
        return columns[i] if i < len(columns) else {}

    is_two = n == 2
    is_split = is_two and join_type == "split_join"

    if is_split:
        join_expr = predicate_at(1)
        left_df, right_df = aliased[0], aliased[1]
        if join_expr is None:
            matched = left_df.join(right_df, how="inner")
            left_unmatched = left_df.join(right_df, how="left_anti")
            right_unmatched = right_df.join(left_df, how="left_anti")
        else:
            matched = left_df.join(right_df, join_expr, how="inner")
            left_unmatched = left_df.join(right_df, join_expr, how="left_anti")
            right_unmatched = right_df.join(left_df, join_expr, how="left_anti")
        projection = _projection([("t0", left_df, cfg(0)), ("t1", right_df, cfg(1))])
        if projection:
            matched = matched.select(*projection)
        return {
            "joined_data": matched,
            "left_unmatched": left_unmatched,
            "right_unmatched": right_unmatched,
        }

    how = join_type if is_two else "inner"
    if is_two:
        acc = aliased[0]
        predicate = predicate_at(1)
        if predicate is None:
            acc = acc.join(aliased[1], how=how)
        else:
            acc = acc.join(aliased[1], predicate, how=how)
    else:
        adjacency: Dict[int, List[int]] = {}
        for idx, (i, j, _edge) in enumerate(edges):
            adjacency.setdefault(i, []).append(idx)
            adjacency.setdefault(j, []).append(idx)
        visited = [False] * n
        visited[0] = True
        acc = aliased[0]
        used_edges = set()
        queue: List[int] = [0]
        while queue:
            cur = queue.pop(0)
            for idx in adjacency.get(cur, []):
                i, j, edge = edges[idx]
                nxt = j if i == cur else i
                if visited[nxt]:
                    continue
                visited[nxt] = True
                used_edges.add(idx)
                predicate = edge_predicate(i, j, edge)
                if predicate is None:
                    acc = acc.join(aliased[nxt], how=how)
                else:
                    acc = acc.join(aliased[nxt], predicate, how=how)
                queue.append(nxt)
        for idx, (i, j, edge) in enumerate(edges):
            if idx in used_edges:
                continue
            predicate = edge_predicate(i, j, edge)
            if predicate is not None:
                acc = acc.where(predicate)

    projection = _projection([("t" + str(i), aliased[i], cfg(i)) for i in range(n)])
    if projection:
        acc = acc.select(*projection)

    return {
        "joined_data": acc,
        "left_unmatched": spark.createDataFrame([], aliased[0].schema),
        "right_unmatched": spark.createDataFrame([], aliased[n - 1].schema),
    }

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "match_case": False,
    "multi_table": False,
    "conditions": [
        {
            "left_input": 0,
            "right_input": 1,
            "join_keys": [
                {
                    "left": "id_cliente",
                    "right": "id_cliente"
                }
            ],
            "join_conditions": "data_resgate BETWEEN data_inicio AND data_fim"
        }
    ],
    "columns": [
        {
            "edits": [],
            "ordered": []
        },
        {
            "edits": [],
            "ordered": []
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "left": ctx["join_3.joined_data"],
    "right": ctx["prepare_8.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["join_8.joined_data"] = out["joined_data"]
ctx["join_8.left_unmatched"] = out["left_unmatched"]
ctx["join_8.right_unmatched"] = out["right_unmatched"]
if globals().get("ld_display_outputs", False) or "join_8" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["join_8.joined_data"])
    display(ctx["join_8.left_unmatched"])
    display(ctx["join_8.right_unmatched"])
if "join_8" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "join_8", "counts": {"joined_data": int(ctx["join_8.joined_data"].count()), "left_unmatched": int(ctx["join_8.left_unmatched"].count()), "right_unmatched": int(ctx["join_8.right_unmatched"].count())}}}, raw=True)

In [0]:
"""
id: select_9
template: transform
templateVersion: 3.0.0
name: select
position:
  x: 1200
  y: 310
description:
  text: Select specified columns in a specific order.
  hash: bda62fc0
previewCodeHash: 2bf2c8e2e0d4897f
codegenHash: 1a3c17c6f2a1f062
config:
  mode: select
  edits:
    - column: data_resgate
    - column: id_cliente
    - column: nome_cliente
    - column: setor
    - column: cidade_sede
    - column: estado_sede
    - column: tipo_programa
    - column: status_cliente
    - column: pontos_resgatados
    - column: status_resgate
    - column: canal_resgate
    - column: tag_campanha
    - column: feedback_participante
    - column: nome_parceiro
    - column: categoria_parceiro
  ordered:
    - data_resgate
    - id_cliente
    - nome_cliente
    - setor
    - email_contato
    - telefone_contato
    - cidade_sede
    - estado_sede
    - data_inicio_contrato
    - tipo_programa
    - status_cliente
    - descricao_empresa
    - id_resgate
    - id_cliente_2
    - id_campanha
    - id_parceiro
    - id_recompensa
    - id_participante
    - pontos_resgatados
    - quantidade
    - status_resgate
    - canal_resgate
    - tag_campanha
    - feedback_participante
    - id_parceiro_3
    - nome_parceiro
    - categoria_parceiro
    - canal_liquidacao
    - sla_resgate
    - status_parceiro
    - right_id_campanha
    - right_id_cliente
    - nome_campanha
    - tipo_campanha
    - data_inicio
    - data_fim
    - orcamento_incentivo
    - canal_principal
    - status_campanha
    - briefing_campanha
input:
  - node: join_8
    input_port: data
    output_port: joined_data
"""

# generated from the system
from typing import Any, Dict, List, Optional
from pyspark.sql import functions as F
from pyspark.sql import types as _spark_types
from pyspark.sql.types import (
    BinaryType,
    BooleanType,
    DateType,
    DecimalType,
    FractionalType,
    IntegerType,
    IntegralType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

_TIMESTAMP_TYPES = tuple(
    t
    for t in (TimestampType, getattr(_spark_types, "TimestampNTZType", None))
    if t is not None
)

def _col_ref(name: str):
    if "." in name and not (name.startswith("`") and name.endswith("`")):
        return F.col("`" + name.replace("`", "``") + "`")
    return F.col(name)

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _column_for(item: Dict[str, Any]):
    col = _col_ref(item.get("column", ""))
    alias = item.get("alias")
    if alias:
        col = col.alias(alias)
    return col

def _passthrough_column(name: str, rename_map: Dict[str, Dict[str, Any]]):
    entry = rename_map.get(name)
    col = _col_ref(name)
    if entry and entry.get("alias"):
        col = col.alias(entry["alias"])
    return col

def _type_category(data_type) -> Optional[str]:
    if isinstance(data_type, BooleanType):
        return "boolean"
    if isinstance(data_type, DecimalType):
        return "decimal"
    if isinstance(data_type, IntegralType):
        return "integer"
    if isinstance(data_type, FractionalType):
        return "float"
    if isinstance(data_type, StringType):
        return "string"
    if isinstance(data_type, DateType):
        return "date"
    if isinstance(data_type, _TIMESTAMP_TYPES):
        return "timestamp"
    if isinstance(data_type, BinaryType):
        return "binary"
    return None

_METADATA_SCHEMA = StructType(
    [
        StructField("Name", StringType(), False),
        StructField("Type", StringType(), False),
        StructField("Category", StringType(), True),
        StructField("FieldNumber", IntegerType(), False),
        StructField("IsNumeric", BooleanType(), False),
        StructField("IsInteger", BooleanType(), False),
        StructField("IsFloat", BooleanType(), False),
        StructField("IsString", BooleanType(), False),
        StructField("IsDateOrTime", BooleanType(), False),
        StructField("IsBinary", BooleanType(), False),
    ]
)

def _metadata_row(index: int, field):
    data_type = field.dataType
    category = _type_category(data_type)
    return (
        field.name,
        data_type.simpleString(),
        category,
        index + 1,
        category in ("integer", "float", "decimal"),
        isinstance(data_type, IntegralType),
        category == "float",
        isinstance(data_type, StringType),
        isinstance(data_type, (DateType,) + _TIMESTAMP_TYPES),
        isinstance(data_type, BinaryType),
    )

def _sql_string_literal(value: str) -> str:
    return "'" + value.replace("\\", "\\\\").replace("'", "\\'") + "'"

def _predicate_for(dynamic: Dict[str, Any]) -> str:
    kind = dynamic.get("kind")
    if kind == "byType":
        selected = dynamic.get("types") or []
        if not selected:
            return "false"
        quoted = ", ".join(_sql_string_literal(t) for t in selected)
        return "Category IN (" + quoted + ")"
    if kind == "byName":
        pattern = dynamic.get("namePattern")
        if not pattern or not pattern.get("op"):
            raise ValueError(
                "Select: a byName dynamic rule requires namePattern with an 'op' and 'value'"
            )
        op = pattern.get("op")
        value = pattern.get("value", "")
        case_sensitive = pattern.get("caseSensitive", False) is True
        if op == "regex":
            regex = value if case_sensitive else "(?i)" + value
            return "Name rlike " + _sql_string_literal(regex)
        name_expr = "Name" if case_sensitive else "lower(Name)"
        needle = value if case_sensitive else value.lower()
        escaped = needle.replace("\\", "\\\\").replace("%", "\\%").replace("_", "\\_")
        if op == "startsWith":
            like = escaped + "%"
        elif op == "endsWith":
            like = "%" + escaped
        else:
            like = "%" + escaped + "%"
        return name_expr + " like " + _sql_string_literal(like)
    if kind == "byExpression":
        expression = dynamic.get("expression") or ""
        return expression if expression.strip() else "false"
    return "false"

def _resolve_dynamic(df, spark, dynamic: Dict[str, Any]):
    predicate = _predicate_for(dynamic)
    fields = df.schema.fields
    rows = [_metadata_row(i, f) for i, f in enumerate(fields)]
    meta_df = spark.createDataFrame(rows, _METADATA_SCHEMA)
    matched_ordinals = {
        row["FieldNumber"] for row in meta_df.filter(predicate).select("FieldNumber").collect()
    }
    action = dynamic.get("action", "keep")
    return [
        index
        for index in range(len(fields))
        if ((index + 1) in matched_ordinals) != (action == "remove")
    ]

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]

    dynamic = config.get("dynamic")
    if dynamic:
        kept_indexes = _resolve_dynamic(df, spark, dynamic)
        original_names = [field.name for field in df.schema.fields]
        placeholders = ["_c" + str(i) for i in range(len(original_names))]
        projected = df.toDF(*placeholders).select(*(F.col(placeholders[i]) for i in kept_indexes))
        return {
            "transformed_data": projected.toDF(*(original_names[i] for i in kept_indexes))
        }

    mode = config.get("mode", "passthrough")
    edits: List[Dict[str, Any]] = config.get("edits", [])
    ordered: List[str] = config.get("ordered") or []

    if mode == "select":
        checked_edits = [item for item in edits if _is_checked(item)]
        if not checked_edits:
            return {"transformed_data": df}
        order_index = {name: i for i, name in enumerate(ordered)}
        tail = len(order_index)
        ordered_edits = sorted(
            checked_edits,
            key=lambda item: order_index.get(item.get("column", ""), tail),
        )
        return {"transformed_data": df.select(*(_column_for(item) for item in ordered_edits))}

    unchecked_cols = {
        item.get("column", "")
        for item in edits
        if not _is_checked(item)
    }
    rename_map = {
        item.get("column", ""): item
        for item in edits
        if item.get("alias") and _is_checked(item)
    }
    upstream_cols = list(df.columns)
    upstream_set = set(upstream_cols)

    effective_ordered = ordered if ordered else list(upstream_cols)

    placed = set()
    out = []
    for token in effective_ordered:
        if token in placed or token not in upstream_set:
            continue
        placed.add(token)
        if token in unchecked_cols:
            continue
        out.append(_passthrough_column(token, rename_map))

    for col in upstream_cols:
        if col in placed or col in unchecked_cols:
            continue
        out.append(_passthrough_column(col, rename_map))

    if not out:
        return {"transformed_data": df}
    return {"transformed_data": df.select(*out)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "mode": "select",
    "edits": [
        {
            "column": "data_resgate"
        },
        {
            "column": "id_cliente"
        },
        {
            "column": "nome_cliente"
        },
        {
            "column": "setor"
        },
        {
            "column": "cidade_sede"
        },
        {
            "column": "estado_sede"
        },
        {
            "column": "tipo_programa"
        },
        {
            "column": "status_cliente"
        },
        {
            "column": "pontos_resgatados"
        },
        {
            "column": "status_resgate"
        },
        {
            "column": "canal_resgate"
        },
        {
            "column": "tag_campanha"
        },
        {
            "column": "feedback_participante"
        },
        {
            "column": "nome_parceiro"
        },
        {
            "column": "categoria_parceiro"
        }
    ],
    "ordered": [
        "data_resgate",
        "id_cliente",
        "nome_cliente",
        "setor",
        "email_contato",
        "telefone_contato",
        "cidade_sede",
        "estado_sede",
        "data_inicio_contrato",
        "tipo_programa",
        "status_cliente",
        "descricao_empresa",
        "id_resgate",
        "id_cliente_2",
        "id_campanha",
        "id_parceiro",
        "id_recompensa",
        "id_participante",
        "pontos_resgatados",
        "quantidade",
        "status_resgate",
        "canal_resgate",
        "tag_campanha",
        "feedback_participante",
        "id_parceiro_3",
        "nome_parceiro",
        "categoria_parceiro",
        "canal_liquidacao",
        "sla_resgate",
        "status_parceiro",
        "right_id_campanha",
        "right_id_cliente",
        "nome_campanha",
        "tipo_campanha",
        "data_inicio",
        "data_fim",
        "orcamento_incentivo",
        "canal_principal",
        "status_campanha",
        "briefing_campanha"
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["join_8.joined_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["select_9.transformed_data"] = out["transformed_data"]
if globals().get("ld_display_outputs", False) or "select_9" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["select_9.transformed_data"])
if "select_9" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "select_9", "counts": {"transformed_data": int(ctx["select_9.transformed_data"].count())}}}, raw=True)

In [0]:
"""
id: limit_26
template: limit
templateVersion: 1.0.0
name: l-1k
position:
  x: 1500
  y: 310
previewCodeHash: f50c525f7927f74e
codegenHash: 74dc4d520a2a810c
config:
  limit: "1000"
input:
  - node: select_9
    input_port: data
    output_port: transformed_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    n = int(config.get("limit", 1000))

    return {"limited_data": df.limit(n)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "limit": "1000"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["select_9.transformed_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["limit_26.limited_data"] = out["limited_data"]
if globals().get("ld_display_outputs", False) or "limit_26" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["limit_26.limited_data"])
if "limit_26" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "limit_26", "counts": {"limited_data": int(ctx["limit_26.limited_data"].count())}}}, raw=True)

In [0]:
"""
id: ai_function_10
template: ai_function
templateVersion: 3.0.0
name: ai-sentimento
position:
  x: 1800
  y: 310
description:
  text: Add sentiment analysis score for feedback and keep all original columns.
  hash: 701d78d1
codegenHash: 2cbe24e3809a0411
config:
  expressions:
    - ai_analyze_sentiment(feedback_participante) `feedback_classificacao`
  keep_all_columns: true
input:
  - node: limit_26
    input_port: data
    output_port: limited_data
"""

# generated from the system
from typing import Dict, Any, List
import hashlib

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]

    # Table-valued AI functions (ai_forecast, ...) live in the FROM
    # clause and have no SELECT-list shape, so we run them via
    # spark.sql and bind the upstream DataFrame as a session-scoped
    # temp view substituted in for the
    # __lakebuilder_ai_function_input__ placeholder identifier
    # (which is a real SQL identifier so the persisted statement
    # round-trips through the SQL parser when the cell reloads).
    #
    # spark.sql returns a lazy DataFrame: the view name is resolved
    # by Spark at action time (preview limit/collect), not when
    # spark.sql is called. We therefore deliberately do NOT drop
    # the temp view here — dropping it would leave the lazy plan
    # pointing at a missing relation and trigger TABLE_OR_VIEW_NOT
    # _FOUND when the downstream action fires.
    #
    # The view name is derived deterministically from (tvf_sql,
    # id(df)) so reruns of the same cell reuse the same name and
    # createOrReplaceTempView keeps the session catalog bounded at
    # one entry per (cell × upstream) instead of growing one entry
    # per run. id(df) is included to keep two cells that happen to
    # have identical tvf_sql but distinct upstream DataFrames from
    # clobbering each other's bindings.
    tvf_sql: str = config.get("tvf_sql") or ""
    if tvf_sql:
        key = f"{tvf_sql}\x00{id(df)}".encode("utf-8")
        view_name = f"lakebuilder_ai_fn_{hashlib.sha256(key).hexdigest()[:12]}"
        df.createOrReplaceTempView(view_name)
        sql = tvf_sql.replace("__lakebuilder_ai_function_input__", view_name)
        return {"ai_data": spark.sql(sql)}

    expressions: List[str] = config.get("expressions", [])
    if not expressions:
        return {"ai_data": df}

    keep_all_columns: bool = config.get("keep_all_columns", True)
    if keep_all_columns:
        return {"ai_data": df.selectExpr(*expressions, "*")}
    return {"ai_data": df.selectExpr(*expressions)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "ai_analyze_sentiment(feedback_participante) `feedback_classificacao`"
    ],
    "keep_all_columns": True
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["limit_26.limited_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["ai_function_10.ai_data"] = out["ai_data"]
if globals().get("ld_display_outputs", False) or "ai_function_10" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["ai_function_10.ai_data"])
if "ai_function_10" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "ai_function_10", "counts": {"ai_data": int(ctx["ai_function_10.ai_data"].count())}}}, raw=True)

In [0]:
"""
id: filter_11
template: filter
templateVersion: 2.0.0
name: f-feedback-negativo
position:
  x: 2100
  y: 232.5
description:
  text: Filter data where feedback classification is negative and separate excluded data.
  hash: 10282d37
codegenHash: d8c91e247e7ed9ff
config:
  condition: feedback_classificacao = 'negative'
input:
  - node: ai_function_10
    input_port: data
    output_port: ai_data
"""

# generated from the system
from typing import Dict, Any

from pyspark.sql import functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df, "excluded_data": spark.createDataFrame([], df.schema)}

    keep = F.coalesce(F.expr(condition), F.lit(False))
    return {"filtered_data": df.filter(keep), "excluded_data": df.filter(~keep)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "feedback_classificacao = 'negative'"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["ai_function_10.ai_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["filter_11.filtered_data"] = out["filtered_data"]
ctx["filter_11.excluded_data"] = out["excluded_data"]
if globals().get("ld_display_outputs", False) or "filter_11" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["filter_11.filtered_data"])
    display(ctx["filter_11.excluded_data"])
if "filter_11" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "filter_11", "counts": {"filtered_data": int(ctx["filter_11.filtered_data"].count()), "excluded_data": int(ctx["filter_11.excluded_data"].count())}}}, raw=True)

In [0]:
"""
id: visualization_14
template: visualization
templateVersion: 1.0.0
name: g-classificacao-feedback
position:
  x: 2100
  y: 387.5
description:
  text: Summarize points by feedback classification.
  hash: 988c2248
codegenHash: 95fa3a0d85544146
config:
  editorSpec:
    queries:
      - queryName: main_query
        query:
          datasetName: lakeflow_designer_upstream
          fields:
            - expression: SUM(`pontos_resgatados`)
              fieldName: sum(pontos_resgatados)
            - expression: "`feedback_classificacao`"
              fieldName: feedback_classificacao
          disaggregatedData: false
    renderSpec:
      version: 3
      widgetType: pie
      encodings:
        angle:
          fieldName: sum(pontos_resgatados)
          scale:
            type: quantitative
        color:
          fieldName: feedback_classificacao
          scale:
            type: categorical
        label:
          show: true
        extra:
          - fieldName: sum(pontos_resgatados)
      data:
        queryName: main_query
    parameterQueries: []
    version: 2
  sql_query: |-
    WITH q AS (SELECT * FROM lakeflow_designer_upstream)
    SELECT SUM(`pontos_resgatados`) AS `sum(pontos_resgatados)`, `feedback_classificacao` AS feedback_classificacao
    FROM q
    GROUP BY feedback_classificacao
input:
  - node: ai_function_10
    input_port: data
    output_port: ai_data
"""

# generated from the system
from typing import Any, Dict

_SOURCE_VIEW = "lakeflow_designer_upstream"

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    if df is None:
        return {"data": None}

    sql_query = config.get("sql_query")
    if not isinstance(sql_query, str) or not sql_query.strip():
        return {"data": df}

    try:
        df.createOrReplaceTempView(_SOURCE_VIEW)
        globals().setdefault("_lb_views", set()).add(_SOURCE_VIEW)
        return {"data": spark.sql(sql_query)}
    except Exception as exc:
        print("[visualization] compiled query failed, passing input through:", exc, sql_query)
        return {"data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "editorSpec": {
        "queries": [
            {
                "queryName": "main_query",
                "query": {
                    "datasetName": "lakeflow_designer_upstream",
                    "fields": [
                        {
                            "expression": "SUM(`pontos_resgatados`)",
                            "fieldName": "sum(pontos_resgatados)"
                        },
                        {
                            "expression": "`feedback_classificacao`",
                            "fieldName": "feedback_classificacao"
                        }
                    ],
                    "disaggregatedData": False
                }
            }
        ],
        "renderSpec": {
            "version": 3,
            "widgetType": "pie",
            "encodings": {
                "angle": {
                    "fieldName": "sum(pontos_resgatados)",
                    "scale": {
                        "type": "quantitative"
                    }
                },
                "color": {
                    "fieldName": "feedback_classificacao",
                    "scale": {
                        "type": "categorical"
                    }
                },
                "label": {
                    "show": True
                },
                "extra": [
                    {
                        "fieldName": "sum(pontos_resgatados)"
                    }
                ]
            },
            "data": {
                "queryName": "main_query"
            }
        },
        "parameterQueries": [],
        "version": 2
    },
    "sql_query": "WITH q AS (SELECT * FROM lakeflow_designer_upstream)\nSELECT SUM(`pontos_resgatados`) AS `sum(pontos_resgatados)`, `feedback_classificacao` AS feedback_classificacao\nFROM q\nGROUP BY feedback_classificacao"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["ai_function_10.ai_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["visualization_14.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "visualization_14" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["visualization_14.data"])
if "visualization_14" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "visualization_14", "counts": {"data": int(ctx["visualization_14.data"].count())}}}, raw=True)

In [0]:
"""
id: prepare_15
template: prepare
templateVersion: 2.0.0
name: p-campos-union
position:
  x: 3000
  y: 135
description:
  text: Create a new column 'recomendacao_anti_churn' with the value 'N/A' for all rows.
  hash: 20da6614
codegenHash: d318165b9d528af4
previewMode: "1000"
config:
  actions:
    - type: formula
      target: recomendacao_anti_churn
      expression: "'N/A'"
input:
  - node: filter_11
    input_port: data
    output_port: excluded_data
"""

# generated from the system
from typing import Any, Callable, Dict, List, Optional, Tuple
import json
import re
import pyspark.sql.functions as F
import pyspark.sql.types as T

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

CURRENT_COLUMN_PLACEHOLDER = "{current_column}"

JSON_PARSE_MODES = ("name_value", "typed_value", "flatten_array", "unnest")

JSON_PARSE_OUTPUT_SUFFIXES: Dict[str, List[str]] = {
    "name_value": ["_Name", "_ValueString"],
    "typed_value": [
        "_Name",
        "_ValueString",
        "_ValueInt",
        "_ValueFloat",
        "_ValueBool",
    ],
    "flatten_array": ["_flatten", "_idx"],
}

JSON_PARSE_SEPARATOR = "_"

JSON_VARIANT_DTYPE = "variant"

JSON_UNNEST_MAX_KEYS = 1024

INT64_MIN = -(2 ** 63)
INT64_MAX = 2 ** 63 - 1

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _action_columns(action: Dict[str, Any], action_type: str) -> List[str]:
    """Resolve an action's target columns from either accepted key.

    The "columns" list wins when it names anything, else the scalar
    "column". An empty list therefore can't mask a valid scalar. Empty
    names are dropped and repeats collapsed, since applying an action
    twice to a column is a no-op. Names are used verbatim (not
    stripped): a column identifier can legally carry edge spaces, so
    trimming would look up a different column than the one saved.
    Mirrors readColumnSelection in the notebook parser so the UI and
    the job resolve the same selection.
    """

    def _names(raw: Any) -> List[str]:
        if isinstance(raw, str):
            raw = [raw]
        if not isinstance(raw, (list, tuple)):
            raise ValueError(
                f"{action_type}: 'columns' must be a list of column names, "
                f"got {type(raw).__name__}"
            )
        names: List[str] = []
        for entry in raw:
            if not isinstance(entry, str):
                raise ValueError(
                    f"{action_type}: column names must be strings, got {entry!r}"
                )
            if entry and entry not in names:
                names.append(entry)
        return names

    columns = _names(action.get("columns", []))
    if not columns:
        columns = _names(action.get("column", ""))
    if not columns:
        raise ValueError(f"{action_type}: 'column' or 'columns' is required")
    return columns

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _quoted_or_comment_end(expression: str, start: int) -> int:
    end = len(expression)
    opening = expression[start]
    if opening in ("'", '"', "`"):
        index = start + 1
        while index < end:
            if expression[index] == "\\" and opening != "`":
                index += 2
                continue
            if expression[index] == opening:
                if index + 1 < end and expression[index + 1] == opening:
                    index += 2
                    continue
                return index + 1
            index += 1
        return end
    if expression.startswith("--", start):
        line_end = expression.find("\n", start)
        return end if line_end == -1 else line_end
    if expression.startswith("/*", start):
        comment_end = expression.find("*/", start + 2)
        return end if comment_end == -1 else comment_end + 2
    return start

def _placeholder_positions(expression: str) -> List[int]:
    positions: List[int] = []
    index = 0
    end = len(expression)
    while index < end:
        if expression.startswith(CURRENT_COLUMN_PLACEHOLDER, index):
            positions.append(index)
            index += len(CURRENT_COLUMN_PLACEHOLDER)
            continue
        skip_to = _quoted_or_comment_end(expression, index)
        index = skip_to if skip_to > index else index + 1
    return positions

def _expand_placeholders(
    expression: str, positions: List[int], quoted_column: str
) -> str:
    rendered: List[str] = []
    cursor = 0
    for position in positions:
        rendered.append(expression[cursor:position])
        rendered.append(quoted_column)
        cursor = position + len(CURRENT_COLUMN_PLACEHOLDER)
    rendered.append(expression[cursor:])
    return "".join(rendered)

def _apply_multi_column_formula(df, action: Dict[str, Any]):
    columns = action.get("columns") or []
    expression = action.get("expression", "")
    prefix = str(action.get("prefix") or "").strip()
    suffix = str(action.get("suffix") or "").strip()
    if isinstance(columns, str) or not isinstance(columns, (list, tuple)):
        raise ValueError(
            "multi_column_formula: 'columns' must be a list of column names, "
            f"got {type(columns).__name__}"
        )
    if not columns:
        raise ValueError("multi_column_formula: 'columns' is required")
    if not expression:
        raise ValueError("multi_column_formula: 'expression' is required")
    positions = _placeholder_positions(expression)
    if not positions:
        raise ValueError(
            "multi_column_formula: 'expression' must contain the "
            + CURRENT_COLUMN_PLACEHOLDER
            + " placeholder outside of string literals, quoted identifiers "
            + "and comments, e.g. '"
            + CURRENT_COLUMN_PLACEHOLDER
            + " * 1.1'"
        )
    assignments: Dict[str, Any] = {}
    for raw_column in columns:
        column = "" if raw_column is None else str(raw_column).strip()
        if not column:
            raise ValueError(
                "multi_column_formula: 'columns' must not contain blank entries"
            )
        _require_column(df, column, "multi_column_formula")
        rendered = _expand_placeholders(
            expression, positions, _quoted_ident(column)
        )
        assignments[f"{prefix}{column}{suffix}"] = F.expr(rendered)
    return df.withColumns(assignments)

def _apply_cast(df, action: Dict[str, Any], column: str):
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return F.expr(f"try_cast({_quoted_ident(column)} as {to_type})")
    return _col(column).cast(to_type)

def _apply_replace_value(df, action: Dict[str, Any], column: str):
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return F.when(predicate, replacement).otherwise(_col(column))

def _apply_fill_null(df, action: Dict[str, Any], column: str):
    with_val = action.get("with", "")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return F.when(_col(column).isNull(), replacement).otherwise(_col(column))

def _apply_text_case(df, action: Dict[str, Any], column: str):
    case = action.get("case", "")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_trim(df, action: Dict[str, Any], column: str):
    side = action.get("side", "both")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_regex_replace(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    return F.regexp_replace(_col(column), pattern, replacement)

def _apply_extract(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    group_raw = action.get("group", 0)
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return F.regexp_extract(_col(column), pattern, group_idx)

def _extract_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    """Destination for one extract application.

    Several extractions cannot share one destination, so "target" is
    honoured for a single source and ignored beyond that, where every
    source is rewritten in place.
    """
    if len(columns) > 1:
        return column
    return action.get("target") or column

def _apply_parse_date(df, action: Dict[str, Any], column: str):
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return F.try_to_timestamp(_col(column), F.lit(fmt))
            return F.try_to_timestamp(_col(column))
        if fmt:
            return F.to_date(_col(column), fmt)
        return F.expr(f"try_cast({_quoted_ident(column)} as date)")
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return fn(_col(column), fmt)
    return fn(_col(column))

def _unique_temp_name(df, base: str) -> str:
    """Name for an intermediate column that cannot shadow user data."""
    existing = set(df.columns)
    if base not in existing:
        return base
    suffix = 2
    while base + "_" + str(suffix) in existing:
        suffix += 1
    return base + "_" + str(suffix)

JSON_DECODE_ERRORS = (ValueError, TypeError, RecursionError)

JSON_PARSE_CONTAINERS = {
    "flatten_array": (list, "array"),
    "unnest": (dict, "object"),
}

def _json_container(text: Any, column: str, mode: str, strict: bool):
    """Decode one document that must be the container the mode expects.
    Returns None when the text is missing, undecodable or the wrong shape;
    raises for the latter two when strict."""
    expected, label = JSON_PARSE_CONTAINERS[mode]
    if _json_document_missing(text):
        return None
    try:
        document = json.loads(text)
    except JSON_DECODE_ERRORS as exc:
        if strict:
            raise ValueError(
                "parse_json: invalid JSON in column '" + column + "': " + str(exc)
            ) from exc
        return None
    if isinstance(document, expected):
        return document
    if strict:
        raise ValueError(
            "parse_json: mode '" + mode + "' expects a JSON " + label
            + " in column '" + column + "', got " + type(document).__name__
        )
    return None

def _json_leaf_pairs(text: str) -> List[Tuple[str, Any]]:
    """Depth-first (path, scalar) pairs for one JSON document.

    Paths use dotted notation with array positions as segments
    ("phoneNumber.0.type"). An empty object or array yields a single
    (path, None) pair so the row still records that the key existed.

    The walk is iterative rather than recursive so that traversal depth is
    bounded by memory, not by the interpreter's recursion limit. (json's own
    decoder still recurses, so documents nested deeper than that limit are
    reported as undecodable — see JSON_DECODE_ERRORS.)
    """
    pairs: List[Tuple[str, Any]] = []
    stack: List[Tuple[str, Any]] = [("", json.loads(text))]
    while stack:
        path, node = stack.pop()
        if isinstance(node, dict):
            if not node:
                pairs.append((path, None))
                continue
            for key in reversed(list(node.keys())):
                child = str(key) if not path else path + "." + str(key)
                stack.append((child, node[key]))
        elif isinstance(node, list):
            if not node:
                pairs.append((path, None))
                continue
            for index in range(len(node) - 1, -1, -1):
                child = str(index) if not path else path + "." + str(index)
                stack.append((child, node[index]))
        else:
            pairs.append((path, node))
    return pairs

def _json_text(value: Any) -> Optional[str]:
    """Render a JSON value as text: strings verbatim, everything else as
    compact JSON so booleans read "true"/"false" and containers keep their
    original form."""
    if value is None:
        return None
    if value is True:
        return "true"
    if value is False:
        return "false"
    if isinstance(value, str):
        return value
    return json.dumps(value, separators=(",", ":"), ensure_ascii=False)

def _reject_json_constant(name: str) -> Any:
    raise ValueError("parse_json: refusing to read " + name + " as a number")

JSON_SCALAR_LEADS = "-0123456789tf"

def _json_scalar_from_text(text: str) -> Any:
    """Reinterpret a quoted scalar as the value it spells, or None when it
    spells nothing.

    Plenty of documents quote everything ({"active": "true", "age": "25"}),
    which would otherwise pile every leaf into the string column and leave
    the typed ones permanently empty. The test is JSON's own literal
    grammar, applied to the whole text: "25" is a number because JSON says
    so, while "007", "+5", "1_000" and " 25 " are not, and so keep their
    exact text rather than being silently rewritten. Only bool/int/float
    count -- "null" stays the word, never a missing value, and Infinity/NaN
    are refused outright since JSON has no such literals.
    """
    if not text or text[0] not in JSON_SCALAR_LEADS or text[-1].isspace():
        return None
    try:
        parsed = json.loads(text, parse_constant=_reject_json_constant)
    except JSON_DECODE_ERRORS:
        return None
    if parsed is True or parsed is False or isinstance(parsed, (int, float)):
        return parsed
    return None

def _json_typed(value: Any) -> Tuple[Any, Any, Any, Any]:
    """Spread a JSON scalar across (string, int, float, bool).

    Exactly one slot is filled, picked from the value's type, so an integer
    never also appears as a float. Integers too wide for BIGINT fall back to
    the string slot instead of overflowing. A quoted scalar is routed by
    what it spells rather than by its quoting, so {"a": true} and
    {"a": "true"} land in the same column -- see _json_scalar_from_text.
    """
    if value is None:
        return (None, None, None, None)
    if value is True or value is False:
        return (None, None, None, bool(value))
    if isinstance(value, str):
        spelled = _json_scalar_from_text(value)
        if spelled is None:
            return (value, None, None, None)
        return _json_typed(spelled)
    if isinstance(value, int):
        if INT64_MIN <= value <= INT64_MAX:
            return (None, value, None, None)
        return (str(value), None, None, None)
    if isinstance(value, float):
        return (None, None, value, None)
    return (_json_text(value), None, None, None)

def _json_document_missing(text: Any) -> bool:
    """NULL and blank text mean "no document" rather than "bad document", so
    they yield NULL outputs even when on_error is "error"."""
    return text is None or (isinstance(text, str) and not text.strip())

def _prefixed_output(prefix: str, suffix: str) -> str:
    """Name a generated column. With no prefix the separator goes too, so
    the column is "Name" rather than "_Name"."""
    if prefix:
        return prefix + suffix
    if suffix.startswith(JSON_PARSE_SEPARATOR):
        return suffix[len(JSON_PARSE_SEPARATOR):]
    return suffix

def _json_source_column(df, column: str):
    """The source column, as the JSON text a mode's UDF expects.

    A VARIANT column holds an already-parsed document rather than its text,
    so render it back with to_json. Any other type is handed over untouched:
    string is text already, and nothing else is offered as a source.
    """
    if _column_dtype(df, column) == JSON_VARIANT_DTYPE:
        return F.to_json(_col(column))
    return _col(column)

def _require_free_columns(outputs: List[str], retained: List[str]) -> None:
    collisions = [name for name in outputs if name in retained]
    if collisions:
        raise ValueError(
            f"parse_json: generated column(s) {collisions} would overwrite existing "
            f"columns. Set a different 'prefix'."
        )

def _parse_json_pairs(df, column: str, source, mode: str, outputs: List[str],
                      retained: List[str], strict: bool):
    typed = mode == "typed_value"
    fields = [
        T.StructField("name", T.StringType()),
        T.StructField("value_string", T.StringType()),
    ]
    if typed:
        fields.extend([
            T.StructField("value_int", T.LongType()),
            T.StructField("value_float", T.DoubleType()),
            T.StructField("value_bool", T.BooleanType()),
        ])

    def to_pairs(text):
        if _json_document_missing(text):
            return None
        try:
            pairs = _json_leaf_pairs(text)
        except JSON_DECODE_ERRORS as exc:
            if strict:
                raise ValueError(
                    "parse_json: invalid JSON in column '" + column + "': " + str(exc)
                ) from exc
            return None
        if typed:
            return [(path,) + _json_typed(value) for path, value in pairs]
        return [(path, _json_text(value)) for path, value in pairs]

    pairs_udf = F.udf(to_pairs, T.ArrayType(T.StructType(fields)))
    exploded = _unique_temp_name(df, "__parse_json")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.explode_outer(pairs_udf(source)).alias(exploded),
    ).select(
        *kept,
        *[
            _col(exploded).getField(field.name).alias(output)
            for field, output in zip(fields, outputs)
        ],
    )

def _parse_json_flatten_array(df, column: str, source, outputs: List[str],
                              retained: List[str], strict: bool):
    def to_elements(text):
        document = _json_container(text, column, "flatten_array", strict)
        if document is None:
            return None
        return [_json_text(element) for element in document]

    elements_udf = F.udf(to_elements, T.ArrayType(T.StringType()))
    position = _unique_temp_name(df, "__parse_json_pos")
    element = _unique_temp_name(df, "__parse_json_val")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.posexplode_outer(elements_udf(source)).alias(position, element),
    ).select(
        *kept,
        _col(element).alias(outputs[0]),
        (_col(position) + F.lit(1)).cast("int").alias(outputs[1]),
    )

def _parse_json_unnest(df, column: str, source, prefix: str,
                       retained: List[str], strict: bool):
    """Alteryx 'Unnest JSON field': one column per top-level key, each value
    kept as JSON text. One level only -- unnest again to go deeper.

    Alone among the modes, the output columns come from the data, so this
    runs one extra Spark job to discover the key names before it can build
    the projection. Keys are ordered by their first position within a
    document (ties broken by name) so the resulting schema is document-order
    like Alteryx's, yet stable across runs and partitionings.
    """
    def to_keys(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [(str(key), index) for index, key in enumerate(document.keys())]

    keys_udf = F.udf(
        to_keys,
        T.ArrayType(
            T.StructType([
                T.StructField("key", T.StringType()),
                T.StructField("position", T.IntegerType()),
            ])
        ),
    )
    found = _unique_temp_name(df, "__parse_json_key")
    discovered = (
        df.select(F.explode(keys_udf(source)).alias(found))
        .select(
            _col(found).getField("key").alias("key"),
            _col(found).getField("position").alias("position"),
        )
        .groupBy("key")
        .agg(F.min("position").alias("position"))
        .limit(JSON_UNNEST_MAX_KEYS + 1)
        .collect()
    )
    if len(discovered) > JSON_UNNEST_MAX_KEYS:
        raise ValueError(
            f"parse_json: mode 'unnest' found more than {JSON_UNNEST_MAX_KEYS} "
            f"distinct keys in column '{column}'. Keys that vary per row (ids, "
            f"timestamps) belong in rows, not columns -- use mode 'name_value' "
            f"for those."
        )
    keys = [
        row["key"]
        for row in sorted(discovered, key=lambda row: (row["position"], row["key"]))
    ]
    if not keys:
        raise ValueError(
            f"parse_json: mode 'unnest' found no keys to unnest in column '{column}'. "
            f"Every value was NULL, blank, or not a JSON object."
        )
    outputs = [prefix + "." + key if prefix else key for key in keys]
    _require_free_columns(outputs, retained)

    def to_values(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [_json_text(document.get(key)) for key in keys]

    values_udf = F.udf(to_values, T.ArrayType(T.StringType()))
    values = _unique_temp_name(df, "__parse_json_values")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        values_udf(source).alias(values),
    ).select(
        *kept,
        *[
            _col(values).getItem(index).alias(output)
            for index, output in enumerate(outputs)
        ],
    )

def _apply_parse_json(df, action: Dict[str, Any]):
    """Explode JSON into rows or columns, mirroring Alteryx's JSON Parse
    tool.

    Python UDFs on purpose: from_json / schema_of_json would need the
    document's schema at plan time, and inferring it means eagerly scanning
    the data, which would both cost a job per action and make the operator's
    output schema depend on the current data. The "unnest" mode accepts
    exactly that cost because its column names ARE the data; every other
    mode stays lazy. VARIANT does not change this -- see the parse_json
    design notes at the top of this file for why each mode reads text.
    """
    column = action.get("column", "")
    mode = action.get("mode", "name_value")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_json")
    if mode not in JSON_PARSE_MODES:
        raise ValueError(
            f"parse_json: unsupported mode '{mode}'. "
            f"Choose one of: {sorted(JSON_PARSE_MODES)}"
        )
    if on_error not in ("null", "error"):
        raise ValueError(
            f"parse_json: unsupported on_error '{on_error}'. Choose null or error."
        )
    prefix = (action.get("prefix") or "").strip()
    keep_source = action.get("keep_source", True) is not False
    retained = [name for name in df.columns if keep_source or name != column]
    strict = on_error == "error"
    source = _json_source_column(df, column)
    if mode == "unnest":
        return _parse_json_unnest(df, column, source, prefix, retained, strict)
    outputs = [
        _prefixed_output(prefix, suffix)
        for suffix in JSON_PARSE_OUTPUT_SUFFIXES[mode]
    ]
    _require_free_columns(outputs, retained)
    if mode == "flatten_array":
        return _parse_json_flatten_array(df, column, source, outputs, retained, strict)
    return _parse_json_pairs(df, column, source, mode, outputs, retained, strict)

VALID_RENAME_TRANSFORMS = {
    "add_prefix",
    "add_suffix",
    "lower",
    "upper",
    "trim",
    "remove_whitespace",
    "snake_case",
    "title_case",
    "sentence_case",
    "camel_case",
    "replace",
}

def _rename_words(name: str) -> List[str]:
    return [word for word in re.split(r"[^A-Za-z0-9]+", name) if word]

def _capitalize_word(word: str) -> str:
    return word[:1].upper() + word[1:].lower()

def _renamed(name: str, action: Dict[str, Any]) -> str:
    transform = action.get("transform", "")
    if transform == "add_prefix":
        return action.get("text", "") + name
    if transform == "add_suffix":
        return name + action.get("text", "")
    if transform == "lower":
        return name.lower()
    if transform == "upper":
        return name.upper()
    if transform == "trim":
        return name.strip()
    if transform == "remove_whitespace":
        return re.sub(r"\s+", "", name)
    if transform == "snake_case":
        return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    if transform == "title_case":
        return " ".join(_capitalize_word(word) for word in _rename_words(name))
    if transform == "sentence_case":
        words = _rename_words(name)
        if not words:
            return ""
        return " ".join([_capitalize_word(words[0])] + [word.lower() for word in words[1:]])
    if transform == "camel_case":
        words = _rename_words(name)
        if not words:
            return ""
        return words[0].lower() + "".join(_capitalize_word(word) for word in words[1:])
    if transform == "replace":
        find = action.get("find", "")
        if not find:
            return name
        return name.replace(find, action.get("replace_with", ""))
    return name

def _require_no_merged_columns(
    old_names: List[str], new_names: List[str]
) -> None:
    sources_by_name: Dict[str, List[str]] = {}
    display_by_name: Dict[str, str] = {}
    for old, new in zip(old_names, new_names):
        key = new.lower()
        sources_by_name.setdefault(key, []).append(old)
        display_by_name.setdefault(key, new)
    merged = [
        (display_by_name[key], sorted(sources))
        for key, sources in sources_by_name.items()
        if len({source.lower() for source in sources}) > 1
    ]
    if merged:
        details = "; ".join(
            f"{sources} -> {new!r}" for new, sources in sorted(merged)
        )
        raise ValueError(
            "rename_columns: the rename would give distinct columns the "
            f"same name ({details}). Spark matches column names "
            "case-insensitively. Narrow 'columns' or pick another transform."
        )

def _apply_rename_columns(df, action: Dict[str, Any]):
    transform = action.get("transform", "")
    if transform not in VALID_RENAME_TRANSFORMS:
        raise ValueError(
            f"rename_columns: unsupported transform '{transform}'. "
            f"Choose one of: {sorted(VALID_RENAME_TRANSFORMS)}"
        )
    targets = action.get("columns") or []
    if not isinstance(targets, list):
        raise ValueError(
            "rename_columns: 'columns' must be a list of column names"
        )
    old_names = list(df.columns)
    missing = [name for name in targets if name not in old_names]
    if missing:
        raise ValueError(
            f"rename_columns: column(s) {missing} not found in input data. "
            f"Available columns: {old_names}"
        )
    selected = set(targets)
    new_names = [
        _renamed(name, action) if not selected or name in selected else name
        for name in old_names
    ]
    blank = [old for old, new in zip(old_names, new_names) if not new]
    if blank:
        raise ValueError(
            f"rename_columns: transform '{transform}' leaves column(s) "
            f"{blank} with an empty name."
        )
    _require_no_merged_columns(old_names, new_names)
    if new_names == old_names:
        return df
    return df.toDF(*new_names)

COLUMN_ACTION_DISPATCH: Dict[str, Callable] = {
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def _in_place_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    return column

ACTION_TARGET_RESOLVERS: Dict[str, Callable] = {
    "extract": _extract_target,
}

# formula names its own output, multi_column_formula writes one output per
# selected column, parse_json reshapes rows/columns, and rename_columns
# rewrites the column names, so all four sit outside the single-expression,
# in-place column dispatch and are routed on their own; every other action
# targets one column or several via the shared column dispatch below.
SUPPORTED_ACTION_TYPES = sorted(
    [
        "formula",
        "multi_column_formula",
        "parse_json",
        "rename_columns",
        *COLUMN_ACTION_DISPATCH,
    ]
)

def _apply_column_action(df, action: Dict[str, Any], action_type: str):
    """Apply one column-scoped action across all of its target columns.

    Every expression is built against the same input DataFrame and
    applied in one withColumns pass. Equivalent to chaining a withColumn
    per column, since no builder reads a column another target in the
    same action rewrites, but it adds one Project instead of N.
    """
    columns = _action_columns(action, action_type)
    build_expression = COLUMN_ACTION_DISPATCH[action_type]
    resolve_target = ACTION_TARGET_RESOLVERS.get(action_type, _in_place_target)
    updates: Dict[str, Any] = {}
    for column in columns:
        _require_column(df, column, action_type)
        target = resolve_target(action, column, columns)
        updates[target] = build_expression(df, action, column)
    return df.withColumns(updates)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        if action_type == "formula":
            df = _apply_formula(df, action)
        elif action_type == "multi_column_formula":
            df = _apply_multi_column_formula(df, action)
        elif action_type == "parse_json":
            df = _apply_parse_json(df, action)
        elif action_type == "rename_columns":
            df = _apply_rename_columns(df, action)
        elif action_type in COLUMN_ACTION_DISPATCH:
            df = _apply_column_action(df, action, action_type)
        else:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {SUPPORTED_ACTION_TYPES}"
            )
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "target": "recomendacao_anti_churn",
            "expression": "'N/A'"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["filter_11.excluded_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_15.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_15" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_15.prepared_data"])
if "prepare_15" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "prepare_15", "counts": {"prepared_data": int(ctx["prepare_15.prepared_data"].count())}}}, raw=True)

In [0]:
"""
id: sort_18
template: sort
templateVersion: 1.0.0
name: s-resgate
position:
  x: 2400
  y: 310
description:
  text: Sort data by pontos_resgatados in descending order.
  hash: c8b6444f
codegenHash: 8d8595d72d737d1d
previewMode: "1000"
config:
  sort_expressions:
    - columnExpr:
        expr: pontos_resgatados
        type: column
      sortBy: DESC
input:
  - node: filter_11
    input_port: data
    output_port: filtered_data
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    sort_expressions = config.get("sort_expressions", [])

    if not sort_expressions:
        return {"sorted_data": df}

    order_cols = []
    for sort_def in sort_expressions:
        col_expr = sort_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        direction = sort_def.get("sortBy", "UNSET")

        col = F.col(raw_expr)
        if direction == "DESC":
            col = col.desc()
        elif direction == "ASC":
            col = col.asc()

        order_cols.append(col)

    return {"sorted_data": df.orderBy(*order_cols)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "sort_expressions": [
        {
            "columnExpr": {
                "expr": "pontos_resgatados",
                "type": "column"
            },
            "sortBy": "DESC"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["filter_11.filtered_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["sort_18.sorted_data"] = out["sorted_data"]
if globals().get("ld_display_outputs", False) or "sort_18" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["sort_18.sorted_data"])
if "sort_18" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "sort_18", "counts": {"sorted_data": int(ctx["sort_18.sorted_data"].count())}}}, raw=True)

In [0]:
"""
id: limit_19
template: limit
templateVersion: 1.0.0
name: l-top-100
position:
  x: 2700
  y: 310
description:
  text: Keep only the first 100 rows.
  hash: c3f42cc0
codegenHash: 74dc4d520a2a810c
previewMode: "1000"
config:
  limit: "100"
input:
  - node: sort_18
    input_port: data
    output_port: sorted_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    n = int(config.get("limit", 1000))

    return {"limited_data": df.limit(n)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "limit": "100"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["sort_18.sorted_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["limit_19.limited_data"] = out["limited_data"]
if globals().get("ld_display_outputs", False) or "limit_19" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["limit_19.limited_data"])
if "limit_19" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "limit_19", "counts": {"limited_data": int(ctx["limit_19.limited_data"].count())}}}, raw=True)

In [0]:
"""
id: ai_function_12
template: ai_function
templateVersion: 3.0.0
name: ai-recomendacao-anti-churn
position:
  x: 3000
  y: 310
description:
  text: Generate a personalized anti-churn recommendation for each customer while keeping all original data columns.
  hash: 58c829a3
codegenHash: 2cbe24e3809a0411
previewMode: "1000"
config:
  expressions:
    - |-
      ai_gen('Você é um especialista em retenção de clientes de programas de fidelidade. '
        || 'Com base nas informações abaixo, gere UMA recomendação de ação concreta e personalizada para evitar o churn deste cliente. '
        || 'Responda em português, em no máximo 2 frases. '
        || 'Cliente: ' || nome_cliente
        || ' | Setor: ' || COALESCE(setor, 'N/A')
        || ' | Cidade: ' || COALESCE(cidade_sede, 'N/A') || '/' || COALESCE(estado_sede, 'N/A')
        || ' | Status resgate: ' || COALESCE(status_resgate, 'N/A')
        || ' | Canal: ' || COALESCE(canal_resgate, 'N/A')
        || ' | Pontos resgatados: ' || CAST(pontos_resgatados AS STRING)
        || ' | Parceiro: ' || COALESCE(nome_parceiro, 'N/A')
        || ' | Categoria parceiro: ' || COALESCE(categoria_parceiro, 'N/A')
        || ' | Feedback negativo: ' || feedback_participante) AS `recomendacao_anti_churn`
  keep_all_columns: true
input:
  - node: limit_19
    input_port: data
    output_port: limited_data
"""

# generated from the system
from typing import Dict, Any, List
import hashlib

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]

    # Table-valued AI functions (ai_forecast, ...) live in the FROM
    # clause and have no SELECT-list shape, so we run them via
    # spark.sql and bind the upstream DataFrame as a session-scoped
    # temp view substituted in for the
    # __lakebuilder_ai_function_input__ placeholder identifier
    # (which is a real SQL identifier so the persisted statement
    # round-trips through the SQL parser when the cell reloads).
    #
    # spark.sql returns a lazy DataFrame: the view name is resolved
    # by Spark at action time (preview limit/collect), not when
    # spark.sql is called. We therefore deliberately do NOT drop
    # the temp view here — dropping it would leave the lazy plan
    # pointing at a missing relation and trigger TABLE_OR_VIEW_NOT
    # _FOUND when the downstream action fires.
    #
    # The view name is derived deterministically from (tvf_sql,
    # id(df)) so reruns of the same cell reuse the same name and
    # createOrReplaceTempView keeps the session catalog bounded at
    # one entry per (cell × upstream) instead of growing one entry
    # per run. id(df) is included to keep two cells that happen to
    # have identical tvf_sql but distinct upstream DataFrames from
    # clobbering each other's bindings.
    tvf_sql: str = config.get("tvf_sql") or ""
    if tvf_sql:
        key = f"{tvf_sql}\x00{id(df)}".encode("utf-8")
        view_name = f"lakebuilder_ai_fn_{hashlib.sha256(key).hexdigest()[:12]}"
        df.createOrReplaceTempView(view_name)
        sql = tvf_sql.replace("__lakebuilder_ai_function_input__", view_name)
        return {"ai_data": spark.sql(sql)}

    expressions: List[str] = config.get("expressions", [])
    if not expressions:
        return {"ai_data": df}

    keep_all_columns: bool = config.get("keep_all_columns", True)
    if keep_all_columns:
        return {"ai_data": df.selectExpr(*expressions, "*")}
    return {"ai_data": df.selectExpr(*expressions)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "ai_gen('Você é um especialista em retenção de clientes de programas de fidelidade. '\n  || 'Com base nas informações abaixo, gere UMA recomendação de ação concreta e personalizada para evitar o churn deste cliente. '\n  || 'Responda em português, em no máximo 2 frases. '\n  || 'Cliente: ' || nome_cliente\n  || ' | Setor: ' || COALESCE(setor, 'N/A')\n  || ' | Cidade: ' || COALESCE(cidade_sede, 'N/A') || '/' || COALESCE(estado_sede, 'N/A')\n  || ' | Status resgate: ' || COALESCE(status_resgate, 'N/A')\n  || ' | Canal: ' || COALESCE(canal_resgate, 'N/A')\n  || ' | Pontos resgatados: ' || CAST(pontos_resgatados AS STRING)\n  || ' | Parceiro: ' || COALESCE(nome_parceiro, 'N/A')\n  || ' | Categoria parceiro: ' || COALESCE(categoria_parceiro, 'N/A')\n  || ' | Feedback negativo: ' || feedback_participante) AS `recomendacao_anti_churn`"
    ],
    "keep_all_columns": True
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["limit_19.limited_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["ai_function_12.ai_data"] = out["ai_data"]
if globals().get("ld_display_outputs", False) or "ai_function_12" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["ai_function_12.ai_data"])
if "ai_function_12" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "ai_function_12", "counts": {"ai_data": int(ctx["ai_function_12.ai_data"].count())}}}, raw=True)

In [0]:
"""
id: combine_14
template: combine
templateVersion: 2.0.0
name: union
position:
  x: 3300
  y: 155
description:
  text: Combine multiple tables by matching columns by name, including all rows from each.
  hash: 7bcbe3d8
previewCodeHash: 4fbba96f1d7d8e35
codegenHash: fcd9d3979c7d02f2
previewMode: "1000"
config:
  operator: UNION
  quantifier: ALL
  union_by_name: true
  allow_missing_columns: true
input:
  - node: ai_function_12
    input_port: data
    output_port: ai_data
  - node: prepare_15
    input_port: data
    output_port: prepared_data
"""

# generated from the system
from functools import reduce
from typing import Any, Callable, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    dfs: List[Any] = list(inputs.get("data") or [])
    if len(dfs) < 1:
        raise ValueError(
            f"Combine requires at least 1 input, got {len(dfs)}. "
            "Wire one or more upstream operators into the 'data' port."
        )

    operator = config.get("operator", "UNION")
    quantifier = config.get("quantifier", "DISTINCT")
    union_by_name = bool(config.get("union_by_name", False))
    allow_missing_columns = bool(config.get("allow_missing_columns", False))

    pair: Callable[[Any, Any], Any]
    if operator == "UNION":
        if union_by_name:
            def pair(a: Any, b: Any) -> Any:
                return a.unionByName(b, allowMissingColumns=allow_missing_columns)
        else:
            def pair(a: Any, b: Any) -> Any:
                return a.union(b)
    elif operator == "INTERSECT":
        if quantifier == "ALL":
            def pair(a: Any, b: Any) -> Any:
                return a.intersectAll(b)
        else:
            def pair(a: Any, b: Any) -> Any:
                return a.intersect(b)
    elif operator in ("EXCEPT", "MINUS"):
        if quantifier == "ALL":
            def pair(a: Any, b: Any) -> Any:
                return a.exceptAll(b)
        else:
            def pair(a: Any, b: Any) -> Any:
                return a.subtract(b)
    else:
        raise ValueError(f"Unsupported combine operator: {operator}")

    result = reduce(pair, dfs)
    if quantifier == "DISTINCT" and (operator == "UNION" or len(dfs) == 1):
        result = result.distinct()
    return {"combined_data": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "operator": "UNION",
    "quantifier": "ALL",
    "union_by_name": True,
    "allow_missing_columns": True
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": [
        ctx["ai_function_12.ai_data"],
        ctx["prepare_15.prepared_data"]
    ],
    "data__sources": [
        {
            "node": "ai_function_12",
            "output_port": "ai_data",
            "name": "ai-recomendacao-anti-churn",
            "df_name": "ai_recomendacao_anti_churn"
        },
        {
            "node": "prepare_15",
            "output_port": "prepared_data",
            "name": "p-campos-union",
            "df_name": "p_campos_union"
        }
    ]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["combine_14.combined_data"] = out["combined_data"]
if globals().get("ld_display_outputs", False) or "combine_14" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["combine_14.combined_data"])
if "combine_14" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "combine_14", "counts": {"combined_data": int(ctx["combine_14.combined_data"].count())}}}, raw=True)

In [0]:
"""
id: notifications_17
template: notifications
templateVersion: 1.0.0
name: email-recomendacao-acao
position:
  x: 3300
  y: 310
description:
  text: Send email notifications with optional attachments using configured recipients and dynamic content.
  hash: d51dc3fd
codegenHash: 9495d0ce4e48af6a
config:
  subscribers:
    - type: user
      id: "75102205510766"
      name: fernando.custodio@databricks.com
  show_email_input: false
  email_column: ""
  subject: "[Resgate] - Clientes com Experiencia Negativa"
  subject_from_column: false
  subject_column: ""
  body: ""
  body_from_column: false
  body_column: ""
  attach_data: true
  attachment_format: excel
input:
  - node: ai_function_12
    input_port: data
    output_port: ai_data
"""

# generated from the system
import uuid
from typing import Any, Dict, List, Optional

NOTIFICATION_CHANNEL_EMAIL = "NOTIFICATION_CHANNEL_EMAIL"
_USER_SUBSCRIBER_TYPES = ("user", "idp_user")
_EMAIL_ADDRESS_SUBSCRIBER_TYPE = "email"
_BODY_MAX_LENGTH = 10000
_ATTACHMENT_ROW_CAP = 10000
_ATTACHMENT_MAX_COUNT = 10
_ATTACHMENT_FORMATS = {"csv": ("FORMAT_CSV", "csv"), "excel": ("FORMAT_XLSX", "xlsx")}

def _notebook_id() -> str:
    context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    notebook_id = context.notebookId()
    resolved = notebook_id.get() if notebook_id.isDefined() else None
    if not resolved:
        raise ValueError(
            "Notifications: could not resolve the notebook id from the "
            "execution context."
        )
    return str(resolved)

def _user_ids(subscribers: List[Dict[str, Any]]) -> List[int]:
    ids: List[int] = []
    for subscriber in subscribers:
        if subscriber.get("type") in _USER_SUBSCRIBER_TYPES:
            raw_id = subscriber.get("id")
            try:
                ids.append(int(raw_id))
            except (TypeError, ValueError):
                raise ValueError(
                    "Notifications: user recipient "
                    f"{subscriber.get('name') or raw_id!r} has a non-numeric id and "
                    "cannot be resolved; remove and re-add the recipient."
                )
    return ids

def _destination_ids(subscribers: List[Dict[str, Any]]) -> List[str]:
    ids: List[str] = []
    for subscriber in subscribers:
        if (
            subscriber.get("type") == "destination"
            and subscriber.get("destinationType") == "email"
        ):
            destination_id = subscriber.get("id")
            if destination_id:
                ids.append(str(destination_id))
    return ids

def _typed_addresses(subscribers: List[Dict[str, Any]]) -> List[str]:
    addresses: List[str] = []
    for subscriber in subscribers:
        if subscriber.get("type") == _EMAIL_ADDRESS_SUBSCRIBER_TYPE:
            address = str(subscriber.get("id") or "").strip()
            if address:
                addresses.append(address)
    return addresses

def _column_emails(df, column: str) -> List[str]:
    if df is None or not column:
        return []
    fields = df.schema.fields
    index = next((i for i, field in enumerate(fields) if field.name == column), None)
    if index is None:
        return []
    aliases = [f"_lb_col_{i}" for i in range(len(fields))]
    target = aliases[index]
    selected = df.toDF(*aliases).select(target).where(f"`{target}` IS NOT NULL").distinct().collect()
    emails: List[str] = []
    for row in selected:
        value = row[0]
        if value is not None and str(value).strip():
            emails.append(str(value).strip())
    return emails

def _column_first_value(df, column: str) -> Optional[str]:
    if df is None or not column:
        return None
    fields = df.schema.fields
    index = next((i for i, field in enumerate(fields) if field.name == column), None)
    if index is None:
        return None
    rows = df.limit(1).collect()
    if not rows:
        return None
    value = rows[0][index]
    return None if value is None else str(value)

def _attachment_cell(value: Any) -> Any:
    if value is None:
        return ""
    if isinstance(value, (bool, int, float, str)):
        return value
    return str(value)

def _build_attachment(df, file_format: str, file_stem: str) -> Dict[str, Any]:
    if df.limit(_ATTACHMENT_ROW_CAP + 1).count() > _ATTACHMENT_ROW_CAP:
        raise ValueError(
            f"Notifications: attachment {file_stem!r} has more than "
            f"{_ATTACHMENT_ROW_CAP} rows; reduce the dataset (e.g. add a filter or "
            "limit upstream) before attaching."
        )
    proto_format, extension = _ATTACHMENT_FORMATS.get(file_format, _ATTACHMENT_FORMATS["csv"])
    fields = df.schema.fields
    columns = [field.name for field in fields]
    field_count = len(fields)
    collected = df.limit(_ATTACHMENT_ROW_CAP).collect()
    rows = [{"cells": [_attachment_cell(row[i]) for i in range(field_count)]} for row in collected]
    return {
        "file": f"{file_stem}.{extension}",
        "columns": columns,
        "rows": rows,
        "format": proto_format,
    }

def _build_attachments(
    dfs: List[Any], sources: List[Dict[str, Any]], file_format: str
) -> List[Dict[str, Any]]:
    candidates = [(index, df) for index, df in enumerate(dfs) if df is not None]
    if len(candidates) > _ATTACHMENT_MAX_COUNT:
        raise ValueError(
            f"Notifications: {len(candidates)} inputs are connected to attach, "
            f"exceeding the limit of {_ATTACHMENT_MAX_COUNT}; disconnect some inputs "
            "or combine them upstream before attaching."
        )
    single_attachment = len(candidates) == 1
    attachments: List[Dict[str, Any]] = []
    for position, (index, df) in enumerate(candidates):
        if single_attachment:
            file_stem = "results"
        else:
            source = sources[index] if index < len(sources) else {}
            file_stem = source.get("df_name") or f"results_{position + 1}"
        attachments.append(_build_attachment(df, file_format, file_stem))
    return attachments

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    subscribers = config.get("subscribers") or []
    email_input = inputs.get("email")

    subject = config.get("subject") or ""
    if config.get("subject_from_column"):
        resolved_subject = _column_first_value(email_input, config.get("subject_column") or "")
        if resolved_subject is not None:
            subject = resolved_subject
    body = config.get("body") or ""
    if config.get("body_from_column"):
        resolved_body = _column_first_value(email_input, config.get("body_column") or "")
        if resolved_body is not None:
            body = resolved_body
    body = body[:_BODY_MAX_LENGTH]
    content = {"subject": subject, "body": body}

    email_column = config.get("email_column") or ""
    recipients_configured = bool(subscribers) or bool(email_column and email_input is not None)

    receiver_variants: List[Dict[str, Any]] = []
    user_ids = _user_ids(subscribers)
    if user_ids:
        receiver_variants.append({"users": {"user_ids": user_ids}})
    destination_ids = _destination_ids(subscribers)
    if destination_ids:
        receiver_variants.append(
            {"destinations": {"destination_ids": destination_ids}}
        )
    column_emails = _column_emails(email_input, email_column)
    seen_keys = {address.lower() for address in column_emails}
    typed_addresses: List[str] = []
    for address in _typed_addresses(subscribers):
        if address.lower() not in seen_keys:
            seen_keys.add(address.lower())
            typed_addresses.append(address)
    email_addresses = typed_addresses + column_emails
    if email_addresses:
        receiver_variants.append({"emails": {"email_addresses": email_addresses}})

    if not receiver_variants:
        if recipients_configured:
            raise ValueError(
                "Notifications: the configured recipients resolved to no "
                "deliverable email addresses (check that the selected email "
                "column has non-empty values and that recipients have emails)."
            )
        return {}

    attachments: List[Dict[str, Any]] = []
    if config.get("attach_data", True):
        data_inputs = inputs.get("data")
        if data_inputs is None:
            data_inputs = []
        elif not isinstance(data_inputs, list):
            data_inputs = [data_inputs]
        attachments = _build_attachments(
            data_inputs,
            inputs.get("data__sources") or [],
            config.get("attachment_format") or "csv",
        )

    from databricks.sdk import WorkspaceClient

    api_client = WorkspaceClient().api_client
    name = f"notebooks/{_notebook_id()}"
    for receivers in receiver_variants:
        request_body = {
            "name": name,
            "idempotency_key": str(uuid.uuid4()),
            "content": content,
            "receivers": receivers,
            "channel": NOTIFICATION_CHANNEL_EMAIL,
        }
        if attachments:
            request_body["attachments"] = attachments
        api_client.do("POST", f"/api/2.0/{name}/notifications:send", body=request_body)
    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "subscribers": [
        {
            "type": "user",
            "id": "75102205510766",
            "name": "fernando.custodio@databricks.com"
        }
    ],
    "show_email_input": False,
    "email_column": "",
    "subject": "[Resgate] - Clientes com Experiencia Negativa",
    "subject_from_column": False,
    "subject_column": "",
    "body": "",
    "body_from_column": False,
    "body_column": "",
    "attach_data": True,
    "attachment_format": "excel"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": [
        ctx["ai_function_12.ai_data"]
    ],
    "data__sources": [
        {
            "node": "ai_function_12",
            "output_port": "ai_data",
            "name": "ai-recomendacao-anti-churn",
            "df_name": "ai_recomendacao_anti_churn"
        }
    ]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
if not config["meta_state"]["is_disabled"]:
    out = run(config, inputs, spark)
else:
    out = {}

In [0]:
"""
id: output_16
template: output
templateVersion: 4.0.0
name: vertem.workshop_dataprep.g_resgates
position:
  x: 3600
  y: 0
description:
  text: Create or refresh a materialized view from the input data, organizing it based on the specified catalog, schema, and view name.
  hash: e9b7ec89
codegenHash: e556653ac6e0f4e0
previewMode: "1000"
config:
  output_type: materialized_view
  catalog: vertem
  schema: workshop_dataprep
  view_name: g_resgates
  cluster_by_mode: auto
input:
  - node: combine_14
    input_port: data
    output_port: combined_data
"""

# generated from the system
import json
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_FAST_CSV_CELLS = 50_000_000

MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES = 2 * 1024 * 1024 * 1024

_FAST_CSV_BYTES_PER_CELL_OVERHEAD = 16

MAX_EXCEL_CELLS = 5_000_000

MAX_SPLIT_GROUPS = 50

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _pandas_line_terminator_parameter(pd) -> str:
    import inspect

    return (
        "lineterminator"
        if "lineterminator" in inspect.signature(pd.DataFrame.to_csv).parameters
        else "line_terminator"
    )

def _write_csv_overwrite(df, path: str, include_header: bool = True) -> bool:
    import os
    import shutil
    import tempfile

    from pyspark.sql import functions as F
    try:
        import pandas as pd
    except ModuleNotFoundError:
        return False

    supported_type_names = {
        "boolean",
        "byte",
        "char",
        "date",
        "decimal",
        "double",
        "float",
        "integer",
        "long",
        "short",
        "string",
        "timestamp",
        "timestamp_ntz",
        "varchar",
        "void",
    }
    normalized_names = [name.casefold() for name in df.columns]
    if (
        not df.columns
        or len(set(normalized_names)) != len(normalized_names)
        or any(field.dataType.typeName() not in supported_type_names for field in df.schema.fields)
    ):
        return False

    spark_output_directory = os.path.isdir(path) and os.path.isfile(
        os.path.join(path, "_SUCCESS")
    )
    if os.path.isdir(path) and not spark_output_directory:
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )

    from databricks.sdk import WorkspaceClient

    client = WorkspaceClient()

    expressions = []
    output_column_names = []
    nan_markers = []
    for index, field in enumerate(df.schema.fields):
        escaped_name = field.name.replace("`", "``")
        column = F.col(f"`{escaped_name}`")
        output_column_name = f"__lb_csv_column_{index}"
        output_column_names.append(field.name)
        type_name = field.dataType.typeName()
        if type_name in {"char", "string", "varchar"}:
            first_character = F.substring(column, 1, 1)
            column = F.when(
                first_character.isin(*_FORMULA_TRIGGERS),
                F.concat(F.lit("'"), column),
            ).otherwise(column)
        elif type_name in {"byte", "integer", "long", "short"}:
            column = column.cast("string")
        elif type_name in {"double", "float"}:
            marker = f"__lb_csv_nan_{index}"
            nan_markers.append((output_column_name, marker))
            expressions.append(F.isnan(column).alias(marker))
        elif type_name == "boolean":
            column = F.when(column, F.lit("True")).when(~column, F.lit("False"))
        elif type_name in {"date", "decimal"}:
            column = column.cast("string")
        elif type_name in {"timestamp", "timestamp_ntz"}:
            column = F.regexp_replace(
                F.date_format(column, "yyyy-MM-dd HH:mm:ss.SSSSSS"),
                r"[.]000000$",
                "",
            )
        expressions.append(column.alias(output_column_name))

    fast_csv_row_cap = min(
        MAX_SINGLE_FILE_ROWS,
        max(1, MAX_FAST_CSV_CELLS // len(df.columns)),
    )
    prepared_df = df.select(*expressions).limit(fast_csv_row_cap + 1).persist()
    try:
        payload_size = F.lit(0)
        for name in prepared_df.columns:
            escaped_name = name.replace("`", "``")
            payload_size = payload_size + F.length(
                F.encode(
                    F.coalesce(F.col(f"`{escaped_name}`").cast("string"), F.lit("")),
                    "UTF-8",
                )
            )
        stats = prepared_df.agg(
            F.count(F.lit(1)).alias("row_count"),
            F.sum(payload_size).alias("payload_bytes"),
        ).collect()[0]
        row_count = stats["row_count"]
        if row_count > fast_csv_row_cap:
            if fast_csv_row_cap < MAX_SINGLE_FILE_ROWS:
                raise ValueError(
                    f"Output: result too wide for safe single-file CSV export "
                    f"(over {MAX_FAST_CSV_CELLS:,} cells) — write to a table instead."
                )
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
        estimated_driver_bytes = (stats["payload_bytes"] or 0) + (
            row_count * len(prepared_df.columns) * _FAST_CSV_BYTES_PER_CELL_OVERHEAD
        )
        if estimated_driver_bytes > MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES:
            raise ValueError(
                f"Output: result too large for safe single-file CSV export "
                f"(estimated driver memory exceeds "
                f"{MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES // (1024 * 1024):,} MiB) — "
                f"write to a table instead."
            )
        pdf = prepared_df.toPandas()
    finally:
        prepared_df.unpersist()
    for column_name, marker in nan_markers:
        nan_mask = pdf[marker].fillna(False)
        finite_mask = ~pdf[column_name].isna()
        pdf[column_name] = pdf[column_name].astype(object)
        pdf.loc[finite_mask, column_name] = pdf.loc[finite_mask, column_name].map(
            lambda value: repr(float(value))
        )
        pdf.loc[nan_mask, column_name] = "nan"
        pdf.loc[~nan_mask & ~finite_mask, column_name] = None
        pdf.drop(columns=[marker], inplace=True)
    pdf.columns = output_column_names
    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        pdf.to_csv(
            local_tmp,
            index=False,
            header=include_header,
            **{_pandas_line_terminator_parameter(pd): "\r\n"},
        )
        if spark_output_directory:
            shutil.rmtree(path)
        with open(local_tmp, "rb") as f:
            client.files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)
    return True

def _file_path(
    catalog: str, schema: str, volume: str, file_name: str, directory_path: str = ""
) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        if directory_path:
            parts = directory_path.split("/")
            if (
                directory_path.startswith("/")
                or directory_path.endswith("/")
                or chr(92) in directory_path
                or any(part in ("", ".", "..") for part in parts)
            ):
                raise ValueError(
                    f"Output: invalid 'directory_path' {directory_path!r}: it must be "
                    "a relative path beneath the selected volume."
                )
        schema_segment = schema.replace(".", "/")
        directory_segment = f"/{directory_path}" if directory_path else ""
        return f"/Volumes/{catalog}/{schema_segment}/{volume}{directory_segment}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _sanitize_split_value(value) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(c for c in text if c not in "/\\").strip()
    cleaned = cleaned.replace("..", ".")
    cleaned = cleaned.strip(". ")
    return cleaned or "none"

def _split_file_name(file_name: str, value, file_type: str, used: set) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = file_name
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    base = f"{stem}_{_sanitize_split_value(value)}"
    candidate = base
    n = 2
    while candidate.lower() in used:
        candidate = f"{base}_{n}"
        n += 1
    used.add(candidate.lower())
    return candidate

_INVALID_SHEET_CHARS = "[]:*?/\\"
_SPLIT_MAP_SHEET = "_lb_split_map"

def _normalize_user_sheet_name(sheet_name: str) -> str:
    if not sheet_name:
        return ""
    cleaned = "".join(
        c for c in sheet_name if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    return cleaned[:31]

def _sanitize_sheet_name(value, used: set) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(
        c for c in text if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    cleaned = cleaned[:31]
    candidate = cleaned
    n = 2
    while candidate.lower() in used:
        tag = f"_{n}"
        candidate = cleaned[: 31 - len(tag)] + tag
        n += 1
    used.add(candidate.lower())
    return candidate

def _split_value_key(value) -> str:
    if value is None:
        return "null"
    if isinstance(value, bool):
        return f"b:{int(value)}"
    if isinstance(value, int):
        return f"i:{value}"
    if isinstance(value, float):
        return f"f:{value}"
    return f"s:{value}"

def _assign_split_names(values, name_for, prior_names=None):
    used: set = set()
    assigned: Dict[int, str] = {}
    priors = list(prior_names) if prior_names is not None else [None] * len(values)
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        prior = priors[i] if i < len(priors) else None
        if prior:
            assigned[i] = prior
            used.add(prior.lower())
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        if i in assigned:
            continue
        assigned[i] = name_for(values[i], used)
    return [assigned[i] for i in range(len(values))]

def _split_sheet_names(values, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(values, _sanitize_sheet_name, prior_names)

def _split_file_names(file_name: str, values, file_type: str, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(
        values,
        lambda value, used: _split_file_name(file_name, value, file_type, used),
        prior_names,
    )

def _load_sheet_split_map(workbook):
    if _SPLIT_MAP_SHEET not in workbook.sheetnames:
        return {}
    mapping = {}
    for row in workbook[_SPLIT_MAP_SHEET].iter_rows(min_row=2, values_only=True):
        if not row or row[0] is None or row[1] is None:
            continue
        mapping[str(row[0])] = str(row[1])
    return mapping

def _write_sheet_split_map(workbook, mapping) -> None:
    if _SPLIT_MAP_SHEET in workbook.sheetnames:
        del workbook[_SPLIT_MAP_SHEET]
    worksheet = workbook.create_sheet(_SPLIT_MAP_SHEET)
    worksheet.sheet_state = "hidden"
    worksheet.append(["value_key", "sheet_name"])
    for key in sorted(mapping):
        worksheet.append([key, mapping[key]])

def _split_manifest_path(directory: str, base_file_name: str, file_type: str) -> str:
    import os

    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    return os.path.join(directory, f"{stem}.lb-split-manifest.json")

def _load_file_split_manifest(directory: str, base_file_name: str, file_type: str):
    import json
    import os

    path = _split_manifest_path(directory, base_file_name, file_type)
    if not os.path.isfile(path):
        return {}
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    groups = payload.get("groups") if isinstance(payload, dict) else None
    if not isinstance(groups, dict):
        return {}
    return {str(k): str(v) for k, v in groups.items()}

def _write_file_split_manifest(directory: str, base_file_name: str, file_type: str, mapping) -> None:
    import json
    import os

    if not directory:
        return
    os.makedirs(directory, exist_ok=True)
    path = _split_manifest_path(directory, base_file_name, file_type)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump({"version": 1, "groups": mapping}, handle, indent=2, sort_keys=True)

def _staged_publisher():
    import os
    import shutil
    import tempfile

    pending: List[Any] = []

    def _discard() -> None:
        for local_tmp, _ in pending:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    def stage_for(dest: str):
        def _stage(write_body) -> None:
            fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
            os.close(fd)
            try:
                write_body(local_tmp)
            except BaseException:
                if os.path.isfile(local_tmp):
                    os.remove(local_tmp)
                raise
            pending.append((local_tmp, dest))

        return _stage

    def publish():
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        remote_staging: List[Any] = []
        try:
            for local_tmp, dest in pending:
                staging_dest = f"{dest}.lb-publishing"
                if os.path.isdir(staging_dest):
                    shutil.rmtree(staging_dest)
                with open(local_tmp, "rb") as f:
                    client.files.upload(staging_dest, f, overwrite=True)
                remote_staging.append((staging_dest, dest))
            for staging_dest, dest in remote_staging:
                if os.path.isdir(dest):
                    shutil.rmtree(dest)
                with open(staging_dest, "rb") as f:
                    client.files.upload(dest, f, overwrite=True)
        finally:
            for staging_dest, _ in remote_staging:
                if os.path.isfile(staging_dest):
                    os.remove(staging_dest)
            _discard()
        return [dest for _, dest in pending]

    return stage_for, publish, _discard

def _remove_stale_split_files(directory: str, base_file_name: str, file_type: str, keep) -> None:
    import os

    if not directory or not os.path.isdir(directory):
        return
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    if not stem:
        return
    prefix = f"{stem}_"
    keep_names = {os.path.basename(p) for p in keep}
    for name in sorted(os.listdir(directory)):
        if name in keep_names:
            continue
        if not name.startswith(prefix) or not name.lower().endswith(suffix.lower()):
            continue
        if len(name) <= len(prefix) + len(suffix):
            continue
        stale = os.path.join(directory, name)
        if os.path.isfile(stale):
            os.remove(stale)

def _align_source_keys(existing_header, columns, safe, where: str):
    pending: Dict[str, List[Any]] = {}
    for c in columns:
        pending.setdefault(str(safe(c)), []).append(c)
    source_keys: List[Any] = []
    for h in existing_header:
        bucket = pending.get(str(h))
        if not bucket:
            raise ValueError(
                f"Output: cannot append to {where}: the existing column {str(h)!r} "
                f"occurs more times than the data provides."
            )
        source_keys.append(bucket.pop(0))
    return source_keys

def _group_rows(rows, split_column: str):
    groups: List[Any] = []
    index: Dict[Any, int] = {}
    for row in rows:
        key = row[split_column]
        if key not in index:
            if len(groups) >= MAX_SPLIT_GROUPS:
                raise ValueError(
                    f"Output: splitting by {split_column!r} would produce more than "
                    f"{MAX_SPLIT_GROUPS} sheets/files. Pick a lower-cardinality column "
                    f"or write to a table instead."
                )
            index[key] = len(groups)
            groups.append((key, []))
        groups[index[key]][1].append(row)
    return groups

def _cell_reference(ref: str):
    from openpyxl.utils.cell import coordinate_to_tuple

    try:
        return coordinate_to_tuple(ref.strip().upper())
    except Exception as exc:
        raise ValueError(
            f"Output: invalid cell reference {ref!r} in the range — use A1 notation."
        ) from exc

def _write_single_file(
    rows, columns, file_type: str, path: str, append: bool, include_header: bool, stage=None
) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    _stage = stage if stage is not None else _upload_stage

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                elif include_header:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        _write_excel_single(
            rows, columns, path, appending, include_header, "", _stage
        )
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _excel_illegal_re():
    import re as _re

    try:
        from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
    except ImportError:
        _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
    return _lb_illegal

def _require_openpyxl():
    try:
        import openpyxl
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Writing Excel files requires the 'openpyxl' package, which "
            "is not installed in this environment. Add "
            "openpyxl==3.1.5 to the notebook environment "
            "and apply it, then run again."
        ) from exc

def _write_excel_single(rows, columns, path, appending, include_header, sheet_name, stage) -> None:
    import io

    _require_openpyxl()
    import openpyxl
    import pandas as pd

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""

    if appending:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            if target_sheet in workbook.sheetnames:
                worksheet = workbook[target_sheet]
            else:
                worksheet = workbook.create_sheet(target_sheet)
        else:
            worksheet = workbook.active

        first = next(worksheet.iter_rows(values_only=True), None)
        if first is not None:
            header = list(first)
            if sorted(str(c) for c in header) != sorted(
                str(_excel_safe(c)) for c in columns
            ):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                    f"file's columns {[str(c) for c in header]}."
                )
            source_keys = _align_source_keys(header, columns, _excel_safe, path)
            existing_rows = max(0, (worksheet.max_row or 0) - 1)
            row_cap = _excel_row_cap(len(header))
            if existing_rows + len(rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(header):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[k])) for k in source_keys]
                )
        else:
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            row_cap = _excel_row_cap(len(columns))
            if len(rows) >= row_cap:
                raise ValueError(
                    f"Output: writing {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(columns):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[c])) for c in columns]
                )

        def _write_excel(tmp_path: str) -> None:
            workbook.save(tmp_path)

        stage(_write_excel)
        return

    header = list(columns)
    source_keys = list(columns)
    combined = [
        [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
    ]
    new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])
    to_excel_kwargs = {"index": False, "engine": "openpyxl", "header": include_header}
    if target_sheet:
        to_excel_kwargs["sheet_name"] = target_sheet

    def _write_excel(tmp_path: str) -> None:
        buffer = io.BytesIO()
        new_df.to_excel(buffer, **to_excel_kwargs)
        with open(tmp_path, "wb") as out:
            out.write(buffer.getvalue())

    stage(_write_excel)

def _write_excel_sheets(groups, columns, path, include_header, append) -> None:
    import os
    import shutil
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    if os.path.isdir(path):
        if os.path.isfile(os.path.join(path, "_SUCCESS")):
            shutil.rmtree(path)
        else:
            raise ValueError(
                f"Output: cannot write file to {path}: a directory already exists at that path."
            )

    appending = append and os.path.isfile(path)

    if appending:
        workbook = openpyxl.load_workbook(path)
    else:
        workbook = openpyxl.Workbook()
        workbook.remove(workbook.active)

    effective_groups = groups if groups else [(None, [])]
    values = [value for value, _ in effective_groups]
    prior_map = _load_sheet_split_map(workbook) if appending else {}
    prior_for_assign = {
        key: name
        for key, name in prior_map.items()
        if name in workbook.sheetnames
    }
    sheet_names = _split_sheet_names(values, prior_for_assign if appending else None)
    updated_map = dict(prior_map) if appending else {}
    for value, target_name in zip(values, sheet_names):
        updated_map[_split_value_key(value)] = target_name
    row_cap = _excel_row_cap(len(columns))
    for (value, group_rows), target_name in zip(effective_groups, sheet_names):
        if appending and target_name in workbook.sheetnames:
            worksheet = workbook[target_name]
            if include_header:
                first = next(worksheet.iter_rows(values_only=True), None)
                if first is not None:
                    existing_header = [str(c) for c in first]
                    if sorted(existing_header) != sorted(str(_excel_safe(c)) for c in columns):
                        raise ValueError(
                            f"Output: cannot append to sheet {target_name!r} in {path}: "
                            f"the data's columns {[str(_excel_safe(c)) for c in columns]} do not "
                            f"match the sheet's columns {existing_header}."
                        )
                    source_keys = _align_source_keys(
                        existing_header, columns, _excel_safe, f"sheet {target_name!r} in {path}"
                    )
                    existing_rows = max(0, (worksheet.max_row or 0) - 1)
                else:
                    source_keys = list(columns)
                    existing_rows = 0
            else:
                source_keys = list(columns)
                existing_rows = worksheet.max_row or 0
            if existing_rows + len(group_rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(group_rows):,} rows would grow sheet "
                    f"{target_name!r} in {path} past the single-file excel export limit "
                    f"({MAX_EXCEL_CELLS:,} cells at {len(columns):,} columns) — write to "
                    f"a table instead."
                )
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[k])) for k in source_keys])
        else:
            worksheet = workbook.create_sheet(target_name)
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[c])) for c in columns])

    if not workbook.sheetnames or workbook.sheetnames == [_SPLIT_MAP_SHEET]:
        workbook.create_sheet("Sheet")
    _write_sheet_split_map(workbook, updated_map)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _write_excel_range(rows, columns, path, sheet_name, range_start, range_end, include_header) -> None:
    import os
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    start_row, start_col = _cell_reference(range_start)
    needed_rows = len(rows) + (1 if include_header else 0)
    needed_cols = len(columns)
    if range_end:
        end_row, end_col = _cell_reference(range_end)
        if end_row < start_row or end_col < start_col:
            raise ValueError(
                f"Output: range end {range_end!r} must be at or after range start {range_start!r}."
            )
        available_rows = end_row - start_row + 1
        available_cols = end_col - start_col + 1
        if needed_rows > available_rows or needed_cols > available_cols:
            raise ValueError(
                f"Output: the data ({len(rows):,} rows x {needed_cols} columns"
                f"{' plus a header row' if include_header else ''}) does not fit the "
                f"range {range_start}:{range_end} ({available_rows} rows x {available_cols} "
                f"columns). Widen the range or write to a table instead."
            )
    else:
        end_row = start_row + max(needed_rows, 1) - 1
        end_col = start_col + max(needed_cols, 1) - 1

    if os.path.isdir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""
    exists = os.path.isfile(path)
    if exists:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            worksheet = (
                workbook[target_sheet]
                if target_sheet in workbook.sheetnames
                else workbook.create_sheet(target_sheet)
            )
        else:
            worksheet = workbook.active
    else:
        workbook = openpyxl.Workbook()
        worksheet = workbook.active
        if target_sheet:
            worksheet.title = target_sheet

    if range_end:
        clear_end_row, clear_end_col = end_row, end_col
    else:
        clear_end_row = max(end_row, worksheet.max_row)
        probe = start_col
        while probe < worksheet.max_column:
            if all(
                worksheet.cell(row=r, column=probe + 1).value is None
                for r in range(start_row, clear_end_row + 1)
            ):
                break
            probe += 1
        clear_end_col = max(end_col, probe)

    from openpyxl.utils.cell import range_boundaries

    removed_merges = []
    for merged in list(worksheet.merged_cells.ranges):
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(str(merged))
        overlaps = (
            m_min_row <= clear_end_row
            and m_max_row >= start_row
            and m_min_col <= clear_end_col
            and m_max_col >= start_col
        )
        if overlaps:
            removed_merges.append(str(merged))
            worksheet.unmerge_cells(str(merged))

    for r in range(start_row, clear_end_row + 1):
        for c in range(start_col, clear_end_col + 1):
            worksheet.cell(row=r, column=c).value = None

    target_row = start_row
    if include_header:
        for offset, column in enumerate(columns):
            worksheet.cell(row=target_row, column=start_col + offset, value=_excel_safe(column))
        target_row += 1
    for row in rows:
        for offset, column in enumerate(columns):
            worksheet.cell(
                row=target_row,
                column=start_col + offset,
                value=_neutralize_formula(_excel_safe(row[column])),
            )
        target_row += 1

    write_end_row = target_row - 1
    write_end_col = start_col + max(needed_cols, 1) - 1
    for merged in removed_merges:
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(merged)
        overlaps_write = (
            m_min_row <= write_end_row
            and m_max_row >= start_row
            and m_min_col <= write_end_col
            and m_max_col >= start_col
        )
        if not overlaps_write:
            worksheet.merge_cells(merged)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _collect_capped(df, file_type: str):
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
        return rows
    rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
    if len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: result too large for single-file export "
            f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
        )
    return rows

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    split_by_column = config.get("split_by_column", "") or ""
    split_mode = config.get("split_mode", "sheets")
    sheet_name = config.get("sheet_name", "") or ""
    include_header = config.get("include_header", True)
    range_start = (config.get("range_start", "") or "").strip()
    range_end = (config.get("range_end", "") or "").strip()
    write_mode = config.get("write_mode", "overwrite")
    columns = df.columns

    if range_end and not range_start:
        raise ValueError(
            "Output: 'range_start' is required to write a cell range "
            "('range_end' alone is not a valid range)."
        )
    if range_start:
        if file_type != "excel":
            raise ValueError("Output: a cell range is only supported for excel file output.")
        if split_by_column:
            raise ValueError(
                "Output: a cell range cannot be combined with split_by_column — "
                "clear one of them."
            )
        if write_mode == "append":
            raise ValueError(
                "Output: a cell range cannot be combined with append — a range writes "
                "into a fixed region, so use write_mode overwrite."
            )

    if split_by_column:
        if split_mode not in ("sheets", "files"):
            raise ValueError(
                f"Output: split_mode {split_mode!r} is not supported. Use 'sheets' or 'files'."
            )
        if split_by_column not in columns:
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} is not a column of the input."
            )
        from pyspark.sql.types import ArrayType, MapType, StructType

        split_type = df.schema[split_by_column].dataType
        if isinstance(split_type, (ArrayType, MapType, StructType)):
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} has type "
                f"{split_type.simpleString()}, which cannot be used to group rows. "
                f"Pick a column with a scalar type (string, number, boolean, date)."
            )
        if split_mode == "sheets" and file_type != "excel":
            raise ValueError(
                "Output: split_mode 'sheets' is only supported for excel file output; "
                "use 'files' for csv/json."
            )

    base_file_name = _with_extension(config.get("file_name", ""), file_type)
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        base_file_name,
        config.get("directory_path", ""),
    )

    if (
        file_type == "csv"
        and not split_by_column
        and write_mode == "overwrite"
        and config.get("__fast_csv_overwrite", False)
        and _write_csv_overwrite(df, path, include_header)
    ):
        return

    rows = _collect_capped(df, file_type)
    append = write_mode == "append"

    if range_start:
        _write_excel_range(
            rows, columns, path, sheet_name, range_start, range_end, include_header
        )
        return

    if split_by_column and split_mode == "sheets":
        groups = _group_rows(rows, split_by_column)
        _write_excel_sheets(groups, columns, path, include_header, append)
        return

    if split_by_column:
        import os

        groups = _group_rows(rows, split_by_column)
        directory = os.path.dirname(path)
        prior_map = (
            _load_file_split_manifest(directory, base_file_name, file_type)
            if append
            else {}
        )
        prior_for_assign = {}
        if append:
            for key, name in prior_map.items():
                candidate = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(name, file_type),
                    config.get("directory_path", ""),
                )
                if os.path.isfile(candidate):
                    prior_for_assign[key] = name
        group_names = _split_file_names(
            config.get("file_name", ""),
            [value for value, _ in groups],
            file_type,
            prior_for_assign if append else None,
        )
        updated_map = dict(prior_map) if append else {}
        for value, group_name in zip([value for value, _ in groups], group_names):
            updated_map[_split_value_key(value)] = group_name
        stage_for, publish, discard = _staged_publisher()
        try:
            for (_, group_rows), group_name in zip(groups, group_names):
                group_path = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(group_name, file_type),
                    config.get("directory_path", ""),
                )
                if file_type == "excel":
                    _write_excel_single_at(
                        group_rows,
                        columns,
                        group_path,
                        append,
                        include_header,
                        sheet_name,
                        stage_for(group_path),
                    )
                else:
                    _write_single_file(
                        group_rows,
                        columns,
                        file_type,
                        group_path,
                        append,
                        include_header,
                        stage_for(group_path),
                    )
        except BaseException:
            discard()
            raise
        written = publish()
        _write_file_split_manifest(directory, base_file_name, file_type, updated_map)
        if not append:
            _remove_stale_split_files(
                directory, base_file_name, file_type, written
            )
        return

    if file_type == "excel":
        _write_excel_single_at(rows, columns, path, append, include_header, sheet_name)
        return
    _write_single_file(rows, columns, file_type, path, append, include_header)

def _write_excel_single_at(rows, columns, path, append, include_header, sheet_name, stage=None) -> None:
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            if os.path.isdir(path) and _is_spark_output_dir(path):
                _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    appending = append and os.path.isfile(path)
    _write_excel_single(
        rows,
        columns,
        path,
        appending,
        include_header,
        sheet_name,
        stage if stage is not None else _upload_stage,
    )

def _read_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    path_literal = json.dumps(path)
    if file_type == "excel":
        sheet_name = (config.get("sheet_name", "") or "").strip()
        range_start = (config.get("range_start", "") or "").strip()
        range_end = (config.get("range_end", "") or "").strip()
        cell_range = (
            f"{range_start}:{range_end}" if range_start and range_end else range_start
        )
        data_address = (
            f"{sheet_name}!{cell_range}" if sheet_name and cell_range else sheet_name or cell_range
        )
        options = ['format=>"excel"']
        if data_address:
            options.append(f"dataAddress=>{json.dumps(data_address)}")
        options.extend(
            [
                f'headerRows=>{1 if config.get("include_header", True) else 0}',
                'schemaEvolutionMode=>"none"',
            ]
        )
        return spark.sql(f"SELECT * FROM read_files({path_literal}, {', '.join(options)})")
    if file_type == "csv":
        header = "true" if config.get("include_header", True) else "false"
        return spark.sql(
            f'SELECT * FROM read_files({path_literal}, format=>"csv", header=>{header})'
        )
    return spark.sql(f'SELECT * FROM read_files({path_literal}, format=>"{file_type}")')

def _display_write_result(write_result) -> None:
    if write_result is not None and globals().get("ld_display_outputs", False):
        display(write_result)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
):
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    return spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        file_type = config.get("file_type", "csv")
        split_col = (config.get("split_by_column") or "").strip()
        if split_col:
            return {"result": df}
        path = _file_path(
            config.get("catalog", ""),
            config.get("schema", ""),
            config.get("volume", ""),
            _with_extension(config.get("file_name", ""), file_type),
            config.get("directory_path", ""),
        )
        return {"result": _read_file_result(path, file_type, config, spark)}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _display_write_result(_run_materialized_view(config, source_view, spark))
        finally:
            spark.catalog.dropTempView(source_view)
        view_full_name = _qualified(catalog, schema, config.get("view_name", ""))
        return {"result": spark.sql(f"SELECT * FROM {view_full_name}")}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            write_result = (
                spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
            )
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            write_result = (
                spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
            )
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            write_result = spark.sql(stmt, args=args) if args else spark.sql(stmt)
        _display_write_result(write_result)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {"result": spark.sql(f"SELECT * FROM {full_name}")}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "materialized_view",
    "catalog": "vertem",
    "schema": "workshop_dataprep",
    "view_name": "g_resgates",
    "cluster_by_mode": "auto"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["combine_14.combined_data"]
}
if config["meta_state"]["is_disabled"]:
    try:
        _lb_disabled_persisted_result = spark.sql("SELECT * FROM vertem.workshop_dataprep.g_resgates").limit(0)
        _lb_disabled_persisted_result.schema
    except Exception as _lb_disabled_persisted_error:
        _lb_disabled_persisted_condition_getter = getattr(_lb_disabled_persisted_error, "getCondition", None)
        _lb_disabled_persisted_error_class = _lb_disabled_persisted_condition_getter() if callable(_lb_disabled_persisted_condition_getter) else None
        if not _lb_disabled_persisted_error_class:
            _lb_disabled_persisted_legacy_error_class_getter = getattr(_lb_disabled_persisted_error, "getErrorClass", None)
            _lb_disabled_persisted_error_class = _lb_disabled_persisted_legacy_error_class_getter() if callable(_lb_disabled_persisted_legacy_error_class_getter) else None
        if _lb_disabled_persisted_error_class not in ("TABLE_OR_VIEW_NOT_FOUND", "PATH_NOT_FOUND"):
            raise
        _lb_disabled_persisted_input = inputs.get("data")
        _lb_disabled_persisted_result = _lb_disabled_persisted_input.limit(0) if hasattr(_lb_disabled_persisted_input, "limit") else _lb_disabled_persisted_input
    out = {"result": _lb_disabled_persisted_result}
else:
    out = run(config, inputs, spark)
ctx["output_16.result"] = out["result"]
if globals().get("ld_display_outputs", False) or "output_16" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["output_16.result"])
if "output_16" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "output_16", "counts": {"result": int(ctx["output_16.result"].count())}}}, raw=True)

In [0]:
"""
id: prepare_21
template: prepare
templateVersion: 2.0.0
name: p-meta
position:
  x: 3600
  y: 155
description:
  text: Create a new column with year-month from a date and clean the client name by trimming spaces and capitalizing words.
  hash: a18b7f9d
codegenHash: d318165b9d528af4
config:
  actions:
    - type: formula
      target: mes_resgate
      expression: date_format(data_resgate, 'yyyy-MM')
    - type: formula
      target: nome_cliente
      expression: trim(initcap(nome_cliente))
input:
  - node: combine_14
    input_port: data
    output_port: combined_data
"""

# generated from the system
from typing import Any, Callable, Dict, List, Optional, Tuple
import json
import re
import pyspark.sql.functions as F
import pyspark.sql.types as T

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

CURRENT_COLUMN_PLACEHOLDER = "{current_column}"

JSON_PARSE_MODES = ("name_value", "typed_value", "flatten_array", "unnest")

JSON_PARSE_OUTPUT_SUFFIXES: Dict[str, List[str]] = {
    "name_value": ["_Name", "_ValueString"],
    "typed_value": [
        "_Name",
        "_ValueString",
        "_ValueInt",
        "_ValueFloat",
        "_ValueBool",
    ],
    "flatten_array": ["_flatten", "_idx"],
}

JSON_PARSE_SEPARATOR = "_"

JSON_VARIANT_DTYPE = "variant"

JSON_UNNEST_MAX_KEYS = 1024

INT64_MIN = -(2 ** 63)
INT64_MAX = 2 ** 63 - 1

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _action_columns(action: Dict[str, Any], action_type: str) -> List[str]:
    """Resolve an action's target columns from either accepted key.

    The "columns" list wins when it names anything, else the scalar
    "column". An empty list therefore can't mask a valid scalar. Empty
    names are dropped and repeats collapsed, since applying an action
    twice to a column is a no-op. Names are used verbatim (not
    stripped): a column identifier can legally carry edge spaces, so
    trimming would look up a different column than the one saved.
    Mirrors readColumnSelection in the notebook parser so the UI and
    the job resolve the same selection.
    """

    def _names(raw: Any) -> List[str]:
        if isinstance(raw, str):
            raw = [raw]
        if not isinstance(raw, (list, tuple)):
            raise ValueError(
                f"{action_type}: 'columns' must be a list of column names, "
                f"got {type(raw).__name__}"
            )
        names: List[str] = []
        for entry in raw:
            if not isinstance(entry, str):
                raise ValueError(
                    f"{action_type}: column names must be strings, got {entry!r}"
                )
            if entry and entry not in names:
                names.append(entry)
        return names

    columns = _names(action.get("columns", []))
    if not columns:
        columns = _names(action.get("column", ""))
    if not columns:
        raise ValueError(f"{action_type}: 'column' or 'columns' is required")
    return columns

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _quoted_or_comment_end(expression: str, start: int) -> int:
    end = len(expression)
    opening = expression[start]
    if opening in ("'", '"', "`"):
        index = start + 1
        while index < end:
            if expression[index] == "\\" and opening != "`":
                index += 2
                continue
            if expression[index] == opening:
                if index + 1 < end and expression[index + 1] == opening:
                    index += 2
                    continue
                return index + 1
            index += 1
        return end
    if expression.startswith("--", start):
        line_end = expression.find("\n", start)
        return end if line_end == -1 else line_end
    if expression.startswith("/*", start):
        comment_end = expression.find("*/", start + 2)
        return end if comment_end == -1 else comment_end + 2
    return start

def _placeholder_positions(expression: str) -> List[int]:
    positions: List[int] = []
    index = 0
    end = len(expression)
    while index < end:
        if expression.startswith(CURRENT_COLUMN_PLACEHOLDER, index):
            positions.append(index)
            index += len(CURRENT_COLUMN_PLACEHOLDER)
            continue
        skip_to = _quoted_or_comment_end(expression, index)
        index = skip_to if skip_to > index else index + 1
    return positions

def _expand_placeholders(
    expression: str, positions: List[int], quoted_column: str
) -> str:
    rendered: List[str] = []
    cursor = 0
    for position in positions:
        rendered.append(expression[cursor:position])
        rendered.append(quoted_column)
        cursor = position + len(CURRENT_COLUMN_PLACEHOLDER)
    rendered.append(expression[cursor:])
    return "".join(rendered)

def _apply_multi_column_formula(df, action: Dict[str, Any]):
    columns = action.get("columns") or []
    expression = action.get("expression", "")
    prefix = str(action.get("prefix") or "").strip()
    suffix = str(action.get("suffix") or "").strip()
    if isinstance(columns, str) or not isinstance(columns, (list, tuple)):
        raise ValueError(
            "multi_column_formula: 'columns' must be a list of column names, "
            f"got {type(columns).__name__}"
        )
    if not columns:
        raise ValueError("multi_column_formula: 'columns' is required")
    if not expression:
        raise ValueError("multi_column_formula: 'expression' is required")
    positions = _placeholder_positions(expression)
    if not positions:
        raise ValueError(
            "multi_column_formula: 'expression' must contain the "
            + CURRENT_COLUMN_PLACEHOLDER
            + " placeholder outside of string literals, quoted identifiers "
            + "and comments, e.g. '"
            + CURRENT_COLUMN_PLACEHOLDER
            + " * 1.1'"
        )
    assignments: Dict[str, Any] = {}
    for raw_column in columns:
        column = "" if raw_column is None else str(raw_column).strip()
        if not column:
            raise ValueError(
                "multi_column_formula: 'columns' must not contain blank entries"
            )
        _require_column(df, column, "multi_column_formula")
        rendered = _expand_placeholders(
            expression, positions, _quoted_ident(column)
        )
        assignments[f"{prefix}{column}{suffix}"] = F.expr(rendered)
    return df.withColumns(assignments)

def _apply_cast(df, action: Dict[str, Any], column: str):
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return F.expr(f"try_cast({_quoted_ident(column)} as {to_type})")
    return _col(column).cast(to_type)

def _apply_replace_value(df, action: Dict[str, Any], column: str):
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return F.when(predicate, replacement).otherwise(_col(column))

def _apply_fill_null(df, action: Dict[str, Any], column: str):
    with_val = action.get("with", "")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return F.when(_col(column).isNull(), replacement).otherwise(_col(column))

def _apply_text_case(df, action: Dict[str, Any], column: str):
    case = action.get("case", "")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_trim(df, action: Dict[str, Any], column: str):
    side = action.get("side", "both")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_regex_replace(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    return F.regexp_replace(_col(column), pattern, replacement)

def _apply_extract(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    group_raw = action.get("group", 0)
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return F.regexp_extract(_col(column), pattern, group_idx)

def _extract_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    """Destination for one extract application.

    Several extractions cannot share one destination, so "target" is
    honoured for a single source and ignored beyond that, where every
    source is rewritten in place.
    """
    if len(columns) > 1:
        return column
    return action.get("target") or column

def _apply_parse_date(df, action: Dict[str, Any], column: str):
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return F.try_to_timestamp(_col(column), F.lit(fmt))
            return F.try_to_timestamp(_col(column))
        if fmt:
            return F.to_date(_col(column), fmt)
        return F.expr(f"try_cast({_quoted_ident(column)} as date)")
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return fn(_col(column), fmt)
    return fn(_col(column))

def _unique_temp_name(df, base: str) -> str:
    """Name for an intermediate column that cannot shadow user data."""
    existing = set(df.columns)
    if base not in existing:
        return base
    suffix = 2
    while base + "_" + str(suffix) in existing:
        suffix += 1
    return base + "_" + str(suffix)

JSON_DECODE_ERRORS = (ValueError, TypeError, RecursionError)

JSON_PARSE_CONTAINERS = {
    "flatten_array": (list, "array"),
    "unnest": (dict, "object"),
}

def _json_container(text: Any, column: str, mode: str, strict: bool):
    """Decode one document that must be the container the mode expects.
    Returns None when the text is missing, undecodable or the wrong shape;
    raises for the latter two when strict."""
    expected, label = JSON_PARSE_CONTAINERS[mode]
    if _json_document_missing(text):
        return None
    try:
        document = json.loads(text)
    except JSON_DECODE_ERRORS as exc:
        if strict:
            raise ValueError(
                "parse_json: invalid JSON in column '" + column + "': " + str(exc)
            ) from exc
        return None
    if isinstance(document, expected):
        return document
    if strict:
        raise ValueError(
            "parse_json: mode '" + mode + "' expects a JSON " + label
            + " in column '" + column + "', got " + type(document).__name__
        )
    return None

def _json_leaf_pairs(text: str) -> List[Tuple[str, Any]]:
    """Depth-first (path, scalar) pairs for one JSON document.

    Paths use dotted notation with array positions as segments
    ("phoneNumber.0.type"). An empty object or array yields a single
    (path, None) pair so the row still records that the key existed.

    The walk is iterative rather than recursive so that traversal depth is
    bounded by memory, not by the interpreter's recursion limit. (json's own
    decoder still recurses, so documents nested deeper than that limit are
    reported as undecodable — see JSON_DECODE_ERRORS.)
    """
    pairs: List[Tuple[str, Any]] = []
    stack: List[Tuple[str, Any]] = [("", json.loads(text))]
    while stack:
        path, node = stack.pop()
        if isinstance(node, dict):
            if not node:
                pairs.append((path, None))
                continue
            for key in reversed(list(node.keys())):
                child = str(key) if not path else path + "." + str(key)
                stack.append((child, node[key]))
        elif isinstance(node, list):
            if not node:
                pairs.append((path, None))
                continue
            for index in range(len(node) - 1, -1, -1):
                child = str(index) if not path else path + "." + str(index)
                stack.append((child, node[index]))
        else:
            pairs.append((path, node))
    return pairs

def _json_text(value: Any) -> Optional[str]:
    """Render a JSON value as text: strings verbatim, everything else as
    compact JSON so booleans read "true"/"false" and containers keep their
    original form."""
    if value is None:
        return None
    if value is True:
        return "true"
    if value is False:
        return "false"
    if isinstance(value, str):
        return value
    return json.dumps(value, separators=(",", ":"), ensure_ascii=False)

def _reject_json_constant(name: str) -> Any:
    raise ValueError("parse_json: refusing to read " + name + " as a number")

JSON_SCALAR_LEADS = "-0123456789tf"

def _json_scalar_from_text(text: str) -> Any:
    """Reinterpret a quoted scalar as the value it spells, or None when it
    spells nothing.

    Plenty of documents quote everything ({"active": "true", "age": "25"}),
    which would otherwise pile every leaf into the string column and leave
    the typed ones permanently empty. The test is JSON's own literal
    grammar, applied to the whole text: "25" is a number because JSON says
    so, while "007", "+5", "1_000" and " 25 " are not, and so keep their
    exact text rather than being silently rewritten. Only bool/int/float
    count -- "null" stays the word, never a missing value, and Infinity/NaN
    are refused outright since JSON has no such literals.
    """
    if not text or text[0] not in JSON_SCALAR_LEADS or text[-1].isspace():
        return None
    try:
        parsed = json.loads(text, parse_constant=_reject_json_constant)
    except JSON_DECODE_ERRORS:
        return None
    if parsed is True or parsed is False or isinstance(parsed, (int, float)):
        return parsed
    return None

def _json_typed(value: Any) -> Tuple[Any, Any, Any, Any]:
    """Spread a JSON scalar across (string, int, float, bool).

    Exactly one slot is filled, picked from the value's type, so an integer
    never also appears as a float. Integers too wide for BIGINT fall back to
    the string slot instead of overflowing. A quoted scalar is routed by
    what it spells rather than by its quoting, so {"a": true} and
    {"a": "true"} land in the same column -- see _json_scalar_from_text.
    """
    if value is None:
        return (None, None, None, None)
    if value is True or value is False:
        return (None, None, None, bool(value))
    if isinstance(value, str):
        spelled = _json_scalar_from_text(value)
        if spelled is None:
            return (value, None, None, None)
        return _json_typed(spelled)
    if isinstance(value, int):
        if INT64_MIN <= value <= INT64_MAX:
            return (None, value, None, None)
        return (str(value), None, None, None)
    if isinstance(value, float):
        return (None, None, value, None)
    return (_json_text(value), None, None, None)

def _json_document_missing(text: Any) -> bool:
    """NULL and blank text mean "no document" rather than "bad document", so
    they yield NULL outputs even when on_error is "error"."""
    return text is None or (isinstance(text, str) and not text.strip())

def _prefixed_output(prefix: str, suffix: str) -> str:
    """Name a generated column. With no prefix the separator goes too, so
    the column is "Name" rather than "_Name"."""
    if prefix:
        return prefix + suffix
    if suffix.startswith(JSON_PARSE_SEPARATOR):
        return suffix[len(JSON_PARSE_SEPARATOR):]
    return suffix

def _json_source_column(df, column: str):
    """The source column, as the JSON text a mode's UDF expects.

    A VARIANT column holds an already-parsed document rather than its text,
    so render it back with to_json. Any other type is handed over untouched:
    string is text already, and nothing else is offered as a source.
    """
    if _column_dtype(df, column) == JSON_VARIANT_DTYPE:
        return F.to_json(_col(column))
    return _col(column)

def _require_free_columns(outputs: List[str], retained: List[str]) -> None:
    collisions = [name for name in outputs if name in retained]
    if collisions:
        raise ValueError(
            f"parse_json: generated column(s) {collisions} would overwrite existing "
            f"columns. Set a different 'prefix'."
        )

def _parse_json_pairs(df, column: str, source, mode: str, outputs: List[str],
                      retained: List[str], strict: bool):
    typed = mode == "typed_value"
    fields = [
        T.StructField("name", T.StringType()),
        T.StructField("value_string", T.StringType()),
    ]
    if typed:
        fields.extend([
            T.StructField("value_int", T.LongType()),
            T.StructField("value_float", T.DoubleType()),
            T.StructField("value_bool", T.BooleanType()),
        ])

    def to_pairs(text):
        if _json_document_missing(text):
            return None
        try:
            pairs = _json_leaf_pairs(text)
        except JSON_DECODE_ERRORS as exc:
            if strict:
                raise ValueError(
                    "parse_json: invalid JSON in column '" + column + "': " + str(exc)
                ) from exc
            return None
        if typed:
            return [(path,) + _json_typed(value) for path, value in pairs]
        return [(path, _json_text(value)) for path, value in pairs]

    pairs_udf = F.udf(to_pairs, T.ArrayType(T.StructType(fields)))
    exploded = _unique_temp_name(df, "__parse_json")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.explode_outer(pairs_udf(source)).alias(exploded),
    ).select(
        *kept,
        *[
            _col(exploded).getField(field.name).alias(output)
            for field, output in zip(fields, outputs)
        ],
    )

def _parse_json_flatten_array(df, column: str, source, outputs: List[str],
                              retained: List[str], strict: bool):
    def to_elements(text):
        document = _json_container(text, column, "flatten_array", strict)
        if document is None:
            return None
        return [_json_text(element) for element in document]

    elements_udf = F.udf(to_elements, T.ArrayType(T.StringType()))
    position = _unique_temp_name(df, "__parse_json_pos")
    element = _unique_temp_name(df, "__parse_json_val")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.posexplode_outer(elements_udf(source)).alias(position, element),
    ).select(
        *kept,
        _col(element).alias(outputs[0]),
        (_col(position) + F.lit(1)).cast("int").alias(outputs[1]),
    )

def _parse_json_unnest(df, column: str, source, prefix: str,
                       retained: List[str], strict: bool):
    """Alteryx 'Unnest JSON field': one column per top-level key, each value
    kept as JSON text. One level only -- unnest again to go deeper.

    Alone among the modes, the output columns come from the data, so this
    runs one extra Spark job to discover the key names before it can build
    the projection. Keys are ordered by their first position within a
    document (ties broken by name) so the resulting schema is document-order
    like Alteryx's, yet stable across runs and partitionings.
    """
    def to_keys(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [(str(key), index) for index, key in enumerate(document.keys())]

    keys_udf = F.udf(
        to_keys,
        T.ArrayType(
            T.StructType([
                T.StructField("key", T.StringType()),
                T.StructField("position", T.IntegerType()),
            ])
        ),
    )
    found = _unique_temp_name(df, "__parse_json_key")
    discovered = (
        df.select(F.explode(keys_udf(source)).alias(found))
        .select(
            _col(found).getField("key").alias("key"),
            _col(found).getField("position").alias("position"),
        )
        .groupBy("key")
        .agg(F.min("position").alias("position"))
        .limit(JSON_UNNEST_MAX_KEYS + 1)
        .collect()
    )
    if len(discovered) > JSON_UNNEST_MAX_KEYS:
        raise ValueError(
            f"parse_json: mode 'unnest' found more than {JSON_UNNEST_MAX_KEYS} "
            f"distinct keys in column '{column}'. Keys that vary per row (ids, "
            f"timestamps) belong in rows, not columns -- use mode 'name_value' "
            f"for those."
        )
    keys = [
        row["key"]
        for row in sorted(discovered, key=lambda row: (row["position"], row["key"]))
    ]
    if not keys:
        raise ValueError(
            f"parse_json: mode 'unnest' found no keys to unnest in column '{column}'. "
            f"Every value was NULL, blank, or not a JSON object."
        )
    outputs = [prefix + "." + key if prefix else key for key in keys]
    _require_free_columns(outputs, retained)

    def to_values(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [_json_text(document.get(key)) for key in keys]

    values_udf = F.udf(to_values, T.ArrayType(T.StringType()))
    values = _unique_temp_name(df, "__parse_json_values")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        values_udf(source).alias(values),
    ).select(
        *kept,
        *[
            _col(values).getItem(index).alias(output)
            for index, output in enumerate(outputs)
        ],
    )

def _apply_parse_json(df, action: Dict[str, Any]):
    """Explode JSON into rows or columns, mirroring Alteryx's JSON Parse
    tool.

    Python UDFs on purpose: from_json / schema_of_json would need the
    document's schema at plan time, and inferring it means eagerly scanning
    the data, which would both cost a job per action and make the operator's
    output schema depend on the current data. The "unnest" mode accepts
    exactly that cost because its column names ARE the data; every other
    mode stays lazy. VARIANT does not change this -- see the parse_json
    design notes at the top of this file for why each mode reads text.
    """
    column = action.get("column", "")
    mode = action.get("mode", "name_value")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_json")
    if mode not in JSON_PARSE_MODES:
        raise ValueError(
            f"parse_json: unsupported mode '{mode}'. "
            f"Choose one of: {sorted(JSON_PARSE_MODES)}"
        )
    if on_error not in ("null", "error"):
        raise ValueError(
            f"parse_json: unsupported on_error '{on_error}'. Choose null or error."
        )
    prefix = (action.get("prefix") or "").strip()
    keep_source = action.get("keep_source", True) is not False
    retained = [name for name in df.columns if keep_source or name != column]
    strict = on_error == "error"
    source = _json_source_column(df, column)
    if mode == "unnest":
        return _parse_json_unnest(df, column, source, prefix, retained, strict)
    outputs = [
        _prefixed_output(prefix, suffix)
        for suffix in JSON_PARSE_OUTPUT_SUFFIXES[mode]
    ]
    _require_free_columns(outputs, retained)
    if mode == "flatten_array":
        return _parse_json_flatten_array(df, column, source, outputs, retained, strict)
    return _parse_json_pairs(df, column, source, mode, outputs, retained, strict)

VALID_RENAME_TRANSFORMS = {
    "add_prefix",
    "add_suffix",
    "lower",
    "upper",
    "trim",
    "remove_whitespace",
    "snake_case",
    "title_case",
    "sentence_case",
    "camel_case",
    "replace",
}

def _rename_words(name: str) -> List[str]:
    return [word for word in re.split(r"[^A-Za-z0-9]+", name) if word]

def _capitalize_word(word: str) -> str:
    return word[:1].upper() + word[1:].lower()

def _renamed(name: str, action: Dict[str, Any]) -> str:
    transform = action.get("transform", "")
    if transform == "add_prefix":
        return action.get("text", "") + name
    if transform == "add_suffix":
        return name + action.get("text", "")
    if transform == "lower":
        return name.lower()
    if transform == "upper":
        return name.upper()
    if transform == "trim":
        return name.strip()
    if transform == "remove_whitespace":
        return re.sub(r"\s+", "", name)
    if transform == "snake_case":
        return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    if transform == "title_case":
        return " ".join(_capitalize_word(word) for word in _rename_words(name))
    if transform == "sentence_case":
        words = _rename_words(name)
        if not words:
            return ""
        return " ".join([_capitalize_word(words[0])] + [word.lower() for word in words[1:]])
    if transform == "camel_case":
        words = _rename_words(name)
        if not words:
            return ""
        return words[0].lower() + "".join(_capitalize_word(word) for word in words[1:])
    if transform == "replace":
        find = action.get("find", "")
        if not find:
            return name
        return name.replace(find, action.get("replace_with", ""))
    return name

def _require_no_merged_columns(
    old_names: List[str], new_names: List[str]
) -> None:
    sources_by_name: Dict[str, List[str]] = {}
    display_by_name: Dict[str, str] = {}
    for old, new in zip(old_names, new_names):
        key = new.lower()
        sources_by_name.setdefault(key, []).append(old)
        display_by_name.setdefault(key, new)
    merged = [
        (display_by_name[key], sorted(sources))
        for key, sources in sources_by_name.items()
        if len({source.lower() for source in sources}) > 1
    ]
    if merged:
        details = "; ".join(
            f"{sources} -> {new!r}" for new, sources in sorted(merged)
        )
        raise ValueError(
            "rename_columns: the rename would give distinct columns the "
            f"same name ({details}). Spark matches column names "
            "case-insensitively. Narrow 'columns' or pick another transform."
        )

def _apply_rename_columns(df, action: Dict[str, Any]):
    transform = action.get("transform", "")
    if transform not in VALID_RENAME_TRANSFORMS:
        raise ValueError(
            f"rename_columns: unsupported transform '{transform}'. "
            f"Choose one of: {sorted(VALID_RENAME_TRANSFORMS)}"
        )
    targets = action.get("columns") or []
    if not isinstance(targets, list):
        raise ValueError(
            "rename_columns: 'columns' must be a list of column names"
        )
    old_names = list(df.columns)
    missing = [name for name in targets if name not in old_names]
    if missing:
        raise ValueError(
            f"rename_columns: column(s) {missing} not found in input data. "
            f"Available columns: {old_names}"
        )
    selected = set(targets)
    new_names = [
        _renamed(name, action) if not selected or name in selected else name
        for name in old_names
    ]
    blank = [old for old, new in zip(old_names, new_names) if not new]
    if blank:
        raise ValueError(
            f"rename_columns: transform '{transform}' leaves column(s) "
            f"{blank} with an empty name."
        )
    _require_no_merged_columns(old_names, new_names)
    if new_names == old_names:
        return df
    return df.toDF(*new_names)

COLUMN_ACTION_DISPATCH: Dict[str, Callable] = {
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def _in_place_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    return column

ACTION_TARGET_RESOLVERS: Dict[str, Callable] = {
    "extract": _extract_target,
}

# formula names its own output, multi_column_formula writes one output per
# selected column, parse_json reshapes rows/columns, and rename_columns
# rewrites the column names, so all four sit outside the single-expression,
# in-place column dispatch and are routed on their own; every other action
# targets one column or several via the shared column dispatch below.
SUPPORTED_ACTION_TYPES = sorted(
    [
        "formula",
        "multi_column_formula",
        "parse_json",
        "rename_columns",
        *COLUMN_ACTION_DISPATCH,
    ]
)

def _apply_column_action(df, action: Dict[str, Any], action_type: str):
    """Apply one column-scoped action across all of its target columns.

    Every expression is built against the same input DataFrame and
    applied in one withColumns pass. Equivalent to chaining a withColumn
    per column, since no builder reads a column another target in the
    same action rewrites, but it adds one Project instead of N.
    """
    columns = _action_columns(action, action_type)
    build_expression = COLUMN_ACTION_DISPATCH[action_type]
    resolve_target = ACTION_TARGET_RESOLVERS.get(action_type, _in_place_target)
    updates: Dict[str, Any] = {}
    for column in columns:
        _require_column(df, column, action_type)
        target = resolve_target(action, column, columns)
        updates[target] = build_expression(df, action, column)
    return df.withColumns(updates)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        if action_type == "formula":
            df = _apply_formula(df, action)
        elif action_type == "multi_column_formula":
            df = _apply_multi_column_formula(df, action)
        elif action_type == "parse_json":
            df = _apply_parse_json(df, action)
        elif action_type == "rename_columns":
            df = _apply_rename_columns(df, action)
        elif action_type in COLUMN_ACTION_DISPATCH:
            df = _apply_column_action(df, action, action_type)
        else:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {SUPPORTED_ACTION_TYPES}"
            )
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "target": "mes_resgate",
            "expression": "date_format(data_resgate, 'yyyy-MM')"
        },
        {
            "type": "formula",
            "target": "nome_cliente",
            "expression": "trim(initcap(nome_cliente))"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["combine_14.combined_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_21.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_21" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_21.prepared_data"])
if "prepare_21" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "prepare_21", "counts": {"prepared_data": int(ctx["prepare_21.prepared_data"].count())}}}, raw=True)

In [0]:
"""
id: aggregate_20
template: aggregate
templateVersion: 2.0.0
name: a-meta
position:
  x: 3900
  y: 0
description:
  text: Group data by customer name and redemption month, then sum redeemed points.
  hash: cf8a3a86
codegenHash: 7e9149ee54d999b7
config:
  group_bys:
    - expr: nome_cliente
      type: column
    - expr: mes_resgate
      type: column
  aggregations:
    - columnExpr:
        expr: pontos_resgatados
        type: column
      fn: SUM
      alias: pontos_resgatados
input:
  - node: prepare_21
    input_port: data
    output_port: prepared_data
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

DEFAULT_PERCENTILE = 0.5

DEFAULT_CONCAT_SEPARATOR = ", "

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
            "FIRST": F.first,
            "LAST": F.last,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = agg_fn(arg)
        elif fn == "-" or fn == "_":
            col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        elif fn == "PERCENTILE":
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        elif fn == "CONCAT":
            raw_sep = agg_def.get("separator")
            sep = raw_sep if isinstance(raw_sep, str) else DEFAULT_CONCAT_SEPARATOR
            concat_arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = F.concat_ws(sep, F.collect_list(concat_arg))
        elif fn == "COUNT_DISTINCT":
            arg = F.expr(raw_expr) if col_expr.get("type") != "column" else raw_expr
            col = F.count_distinct(arg)
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "nome_cliente",
            "type": "column"
        },
        {
            "expr": "mes_resgate",
            "type": "column"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "pontos_resgatados",
                "type": "column"
            },
            "fn": "SUM",
            "alias": "pontos_resgatados",
            "withAsKeyword": None
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["prepare_21.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["aggregate_20.aggregated_data"] = out["aggregated_data"]
if globals().get("ld_display_outputs", False) or "aggregate_20" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["aggregate_20.aggregated_data"])
if "aggregate_20" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "aggregate_20", "counts": {"aggregated_data": int(ctx["aggregate_20.aggregated_data"].count())}}}, raw=True)

In [0]:
"""
id: join_23
template: join
templateVersion: 3.0.0
name: j-realizado-meta
position:
  x: 4200
  y: 155
description:
  text: Perform a left join on two inputs using the specified keys, keeping all rows from the left side and matching rows from the right. Show unmatched rows separately.
  hash: e811884c
codegenHash: ea5821d113ebbb3b
config:
  join_type: left
  match_case: false
  multi_table: false
  conditions:
    - left_input: 0
      right_input: 1
      join_keys:
        - left: nome_cliente
          right: nome_cliente
        - left: mes_resgate
          right: ano_mes
      join_conditions: ""
  columns:
    - edits: []
      ordered: []
    - edits: []
      ordered: []
input:
  - node: aggregate_20
    input_port: left
    output_port: aggregated_data
  - node: source_22
    input_port: right
    output_port: data
"""

# generated from the system
from typing import Any, Dict, List, Optional
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _qualified_col(alias: str, name: str):
    escaped = name.replace("`", "``")
    return F.col("`" + alias + "`.`" + escaped + "`")

def _ordered_names(df_cols: List[str], cfg: Dict[str, Any]) -> List[str]:
    ordered: List[str] = cfg.get("ordered") or []
    col_set = set(df_cols)
    placed = set()
    result: List[str] = []
    for name in ordered:
        if name in placed or name not in col_set:
            continue
        placed.add(name)
        result.append(name)
    for name in df_cols:
        if name in placed:
            continue
        placed.add(name)
        result.append(name)
    return result

def _edits_by_column(cfg: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    by_col: Dict[str, Dict[str, Any]] = {}
    for item in cfg.get("edits", []) or []:
        col = item.get("column")
        if col is not None and col not in by_col:
            by_col[col] = item
    return by_col

def _projection(frames: List[Any]):
    out = []
    seen: Dict[str, int] = {}
    n = len(frames)
    for idx, (alias, df, cfg) in enumerate(frames):
        edits = _edits_by_column(cfg)
        for name in _ordered_names(list(df.columns), cfg):
            edit = edits.get(name)
            if edit is not None and not _is_checked(edit):
                continue
            col = _qualified_col(alias, name)
            user_alias = edit.get("alias") if edit else None
            if user_alias:
                out.append(col.alias(user_alias))
                seen[user_alias.lower()] = seen.get(user_alias.lower(), 0) + 1
                continue
            key = name.lower()
            if key not in seen:
                seen[key] = 1
                out.append(col.alias(name))
                continue
            new_name = ("right_" + name) if (n == 2 and idx == 1) else (name + "_" + str(idx + 1))
            while new_name.lower() in seen:
                new_name = new_name + "_dup"
            seen[new_name.lower()] = 1
            out.append(col.alias(new_name))
    return out

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    dfs: List[Any] = []
    for port in ("left", "right"):
        df = inputs.get(port)
        if df is not None:
            dfs.append(df)
    dfs.extend(inputs.get("data") or [])
    if len(dfs) < 2:
        raise ValueError(
            f"Join requires at least 2 inputs, got {len(dfs)}. "
            "Wire 2 or more upstream operators into this Join."
        )

    n = len(dfs)
    join_type = config.get("join_type") or "split_join"
    match_case = bool(config.get("match_case", False))
    conditions: List[Dict[str, Any]] = config.get("conditions") or []
    columns: List[Dict[str, Any]] = config.get("columns") or []

    aliased = [df.alias("t" + str(i)) for i, df in enumerate(dfs)]
    types = [{f.name.lower(): f.dataType for f in df.schema} for df in aliased]

    def key_col(idx: int, name: str):
        col = _qualified_col("t" + str(idx), name)
        if not match_case and isinstance(types[idx].get(name.lower()), StringType):
            return F.upper(F.trim(col))
        return col

    def edge_predicate(i: int, j: int, edge: Dict[str, Any]) -> Optional[Any]:
        predicates = []
        for key in edge.get("join_keys") or []:
            predicates.append(key_col(i, key["left"]) == key_col(j, key["right"]))
        condition = edge.get("join_conditions") or ""
        if condition:
            predicates.append(F.expr(condition))
        join_expr = None
        for predicate in predicates:
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    edges: List[Any] = [
        (int(edge.get("left_input", 0)), int(edge.get("right_input", 0)), edge)
        for edge in conditions
    ]

    predicates_by_step: Dict[int, List[Any]] = {}
    for i, j, edge in edges:
        predicate = edge_predicate(i, j, edge)
        if predicate is not None:
            predicates_by_step.setdefault(max(i, j), []).append(predicate)

    def predicate_at(step: int) -> Optional[Any]:
        join_expr = None
        for predicate in predicates_by_step.get(step, []):
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    def cfg(i: int) -> Dict[str, Any]:
        return columns[i] if i < len(columns) else {}

    is_two = n == 2
    is_split = is_two and join_type == "split_join"

    if is_split:
        join_expr = predicate_at(1)
        left_df, right_df = aliased[0], aliased[1]
        if join_expr is None:
            matched = left_df.join(right_df, how="inner")
            left_unmatched = left_df.join(right_df, how="left_anti")
            right_unmatched = right_df.join(left_df, how="left_anti")
        else:
            matched = left_df.join(right_df, join_expr, how="inner")
            left_unmatched = left_df.join(right_df, join_expr, how="left_anti")
            right_unmatched = right_df.join(left_df, join_expr, how="left_anti")
        projection = _projection([("t0", left_df, cfg(0)), ("t1", right_df, cfg(1))])
        if projection:
            matched = matched.select(*projection)
        return {
            "joined_data": matched,
            "left_unmatched": left_unmatched,
            "right_unmatched": right_unmatched,
        }

    how = join_type if is_two else "inner"
    if is_two:
        acc = aliased[0]
        predicate = predicate_at(1)
        if predicate is None:
            acc = acc.join(aliased[1], how=how)
        else:
            acc = acc.join(aliased[1], predicate, how=how)
    else:
        adjacency: Dict[int, List[int]] = {}
        for idx, (i, j, _edge) in enumerate(edges):
            adjacency.setdefault(i, []).append(idx)
            adjacency.setdefault(j, []).append(idx)
        visited = [False] * n
        visited[0] = True
        acc = aliased[0]
        used_edges = set()
        queue: List[int] = [0]
        while queue:
            cur = queue.pop(0)
            for idx in adjacency.get(cur, []):
                i, j, edge = edges[idx]
                nxt = j if i == cur else i
                if visited[nxt]:
                    continue
                visited[nxt] = True
                used_edges.add(idx)
                predicate = edge_predicate(i, j, edge)
                if predicate is None:
                    acc = acc.join(aliased[nxt], how=how)
                else:
                    acc = acc.join(aliased[nxt], predicate, how=how)
                queue.append(nxt)
        for idx, (i, j, edge) in enumerate(edges):
            if idx in used_edges:
                continue
            predicate = edge_predicate(i, j, edge)
            if predicate is not None:
                acc = acc.where(predicate)

    projection = _projection([("t" + str(i), aliased[i], cfg(i)) for i in range(n)])
    if projection:
        acc = acc.select(*projection)

    return {
        "joined_data": acc,
        "left_unmatched": spark.createDataFrame([], aliased[0].schema),
        "right_unmatched": spark.createDataFrame([], aliased[n - 1].schema),
    }

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "match_case": False,
    "multi_table": False,
    "conditions": [
        {
            "left_input": 0,
            "right_input": 1,
            "join_keys": [
                {
                    "left": "nome_cliente",
                    "right": "nome_cliente"
                },
                {
                    "left": "mes_resgate",
                    "right": "ano_mes"
                }
            ],
            "join_conditions": ""
        }
    ],
    "columns": [
        {
            "edits": [],
            "ordered": []
        },
        {
            "edits": [],
            "ordered": []
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "left": ctx["aggregate_20.aggregated_data"],
    "right": ctx["source_22.data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["join_23.joined_data"] = out["joined_data"]
ctx["join_23.left_unmatched"] = out["left_unmatched"]
ctx["join_23.right_unmatched"] = out["right_unmatched"]
if globals().get("ld_display_outputs", False) or "join_23" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["join_23.joined_data"])
    display(ctx["join_23.left_unmatched"])
    display(ctx["join_23.right_unmatched"])
if "join_23" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "join_23", "counts": {"joined_data": int(ctx["join_23.joined_data"].count()), "left_unmatched": int(ctx["join_23.left_unmatched"].count()), "right_unmatched": int(ctx["join_23.right_unmatched"].count())}}}, raw=True)

In [0]:
"""
id: select_24
template: transform
templateVersion: 3.0.0
name: s-meta
position:
  x: 4500
  y: 155
description:
  text: Remove two columns and rearrange remaining columns in a specific order.
  hash: 9958d14f
codegenHash: 1a3c17c6f2a1f062
config:
  mode: passthrough
  edits:
    - column: right_nome_cliente
      checked: false
    - column: ano_mes
      checked: false
  ordered:
    - mes_resgate
    - nome_cliente
    - pontos_resgatados
    - meta_pontos
    - meta_resgates
    - responsavel_conta
    - prioridade
    - right_nome_cliente
    - ano_mes
input:
  - node: join_23
    input_port: data
    output_port: joined_data
"""

# generated from the system
from typing import Any, Dict, List, Optional
from pyspark.sql import functions as F
from pyspark.sql import types as _spark_types
from pyspark.sql.types import (
    BinaryType,
    BooleanType,
    DateType,
    DecimalType,
    FractionalType,
    IntegerType,
    IntegralType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

_TIMESTAMP_TYPES = tuple(
    t
    for t in (TimestampType, getattr(_spark_types, "TimestampNTZType", None))
    if t is not None
)

def _col_ref(name: str):
    if "." in name and not (name.startswith("`") and name.endswith("`")):
        return F.col("`" + name.replace("`", "``") + "`")
    return F.col(name)

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _column_for(item: Dict[str, Any]):
    col = _col_ref(item.get("column", ""))
    alias = item.get("alias")
    if alias:
        col = col.alias(alias)
    return col

def _passthrough_column(name: str, rename_map: Dict[str, Dict[str, Any]]):
    entry = rename_map.get(name)
    col = _col_ref(name)
    if entry and entry.get("alias"):
        col = col.alias(entry["alias"])
    return col

def _type_category(data_type) -> Optional[str]:
    if isinstance(data_type, BooleanType):
        return "boolean"
    if isinstance(data_type, DecimalType):
        return "decimal"
    if isinstance(data_type, IntegralType):
        return "integer"
    if isinstance(data_type, FractionalType):
        return "float"
    if isinstance(data_type, StringType):
        return "string"
    if isinstance(data_type, DateType):
        return "date"
    if isinstance(data_type, _TIMESTAMP_TYPES):
        return "timestamp"
    if isinstance(data_type, BinaryType):
        return "binary"
    return None

_METADATA_SCHEMA = StructType(
    [
        StructField("Name", StringType(), False),
        StructField("Type", StringType(), False),
        StructField("Category", StringType(), True),
        StructField("FieldNumber", IntegerType(), False),
        StructField("IsNumeric", BooleanType(), False),
        StructField("IsInteger", BooleanType(), False),
        StructField("IsFloat", BooleanType(), False),
        StructField("IsString", BooleanType(), False),
        StructField("IsDateOrTime", BooleanType(), False),
        StructField("IsBinary", BooleanType(), False),
    ]
)

def _metadata_row(index: int, field):
    data_type = field.dataType
    category = _type_category(data_type)
    return (
        field.name,
        data_type.simpleString(),
        category,
        index + 1,
        category in ("integer", "float", "decimal"),
        isinstance(data_type, IntegralType),
        category == "float",
        isinstance(data_type, StringType),
        isinstance(data_type, (DateType,) + _TIMESTAMP_TYPES),
        isinstance(data_type, BinaryType),
    )

def _sql_string_literal(value: str) -> str:
    return "'" + value.replace("\\", "\\\\").replace("'", "\\'") + "'"

def _predicate_for(dynamic: Dict[str, Any]) -> str:
    kind = dynamic.get("kind")
    if kind == "byType":
        selected = dynamic.get("types") or []
        if not selected:
            return "false"
        quoted = ", ".join(_sql_string_literal(t) for t in selected)
        return "Category IN (" + quoted + ")"
    if kind == "byName":
        pattern = dynamic.get("namePattern")
        if not pattern or not pattern.get("op"):
            raise ValueError(
                "Select: a byName dynamic rule requires namePattern with an 'op' and 'value'"
            )
        op = pattern.get("op")
        value = pattern.get("value", "")
        case_sensitive = pattern.get("caseSensitive", False) is True
        if op == "regex":
            regex = value if case_sensitive else "(?i)" + value
            return "Name rlike " + _sql_string_literal(regex)
        name_expr = "Name" if case_sensitive else "lower(Name)"
        needle = value if case_sensitive else value.lower()
        escaped = needle.replace("\\", "\\\\").replace("%", "\\%").replace("_", "\\_")
        if op == "startsWith":
            like = escaped + "%"
        elif op == "endsWith":
            like = "%" + escaped
        else:
            like = "%" + escaped + "%"
        return name_expr + " like " + _sql_string_literal(like)
    if kind == "byExpression":
        expression = dynamic.get("expression") or ""
        return expression if expression.strip() else "false"
    return "false"

def _resolve_dynamic(df, spark, dynamic: Dict[str, Any]):
    predicate = _predicate_for(dynamic)
    fields = df.schema.fields
    rows = [_metadata_row(i, f) for i, f in enumerate(fields)]
    meta_df = spark.createDataFrame(rows, _METADATA_SCHEMA)
    matched_ordinals = {
        row["FieldNumber"] for row in meta_df.filter(predicate).select("FieldNumber").collect()
    }
    action = dynamic.get("action", "keep")
    return [
        index
        for index in range(len(fields))
        if ((index + 1) in matched_ordinals) != (action == "remove")
    ]

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]

    dynamic = config.get("dynamic")
    if dynamic:
        kept_indexes = _resolve_dynamic(df, spark, dynamic)
        original_names = [field.name for field in df.schema.fields]
        placeholders = ["_c" + str(i) for i in range(len(original_names))]
        projected = df.toDF(*placeholders).select(*(F.col(placeholders[i]) for i in kept_indexes))
        return {
            "transformed_data": projected.toDF(*(original_names[i] for i in kept_indexes))
        }

    mode = config.get("mode", "passthrough")
    edits: List[Dict[str, Any]] = config.get("edits", [])
    ordered: List[str] = config.get("ordered") or []

    if mode == "select":
        checked_edits = [item for item in edits if _is_checked(item)]
        if not checked_edits:
            return {"transformed_data": df}
        order_index = {name: i for i, name in enumerate(ordered)}
        tail = len(order_index)
        ordered_edits = sorted(
            checked_edits,
            key=lambda item: order_index.get(item.get("column", ""), tail),
        )
        return {"transformed_data": df.select(*(_column_for(item) for item in ordered_edits))}

    unchecked_cols = {
        item.get("column", "")
        for item in edits
        if not _is_checked(item)
    }
    rename_map = {
        item.get("column", ""): item
        for item in edits
        if item.get("alias") and _is_checked(item)
    }
    upstream_cols = list(df.columns)
    upstream_set = set(upstream_cols)

    effective_ordered = ordered if ordered else list(upstream_cols)

    placed = set()
    out = []
    for token in effective_ordered:
        if token in placed or token not in upstream_set:
            continue
        placed.add(token)
        if token in unchecked_cols:
            continue
        out.append(_passthrough_column(token, rename_map))

    for col in upstream_cols:
        if col in placed or col in unchecked_cols:
            continue
        out.append(_passthrough_column(col, rename_map))

    if not out:
        return {"transformed_data": df}
    return {"transformed_data": df.select(*out)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "mode": "passthrough",
    "edits": [
        {
            "column": "right_nome_cliente",
            "checked": False
        },
        {
            "column": "ano_mes",
            "checked": False
        }
    ],
    "ordered": [
        "mes_resgate",
        "nome_cliente",
        "pontos_resgatados",
        "meta_pontos",
        "meta_resgates",
        "responsavel_conta",
        "prioridade",
        "right_nome_cliente",
        "ano_mes"
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["join_23.joined_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["select_24.transformed_data"] = out["transformed_data"]
if globals().get("ld_display_outputs", False) or "select_24" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["select_24.transformed_data"])
if "select_24" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "select_24", "counts": {"transformed_data": int(ctx["select_24.transformed_data"].count())}}}, raw=True)

In [0]:
"""
id: output_25_1
template: output
templateVersion: 4.0.0
name: vertem.workshop_dataprep.g_resgate_realizado_meta
position:
  x: 4800
  y: 155
description:
  text: Create or replace a table in the specified catalog and schema with the data provided, using overwrite mode.
  hash: 7c8d80dc
codegenHash: e556653ac6e0f4e0
config:
  output_type: table
  catalog: vertem
  schema: workshop_dataprep
  table_name: g_resgate_realizado_meta
  write_mode: overwrite
input:
  - node: select_24
    input_port: data
    output_port: transformed_data
"""

# generated from the system
import json
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_FAST_CSV_CELLS = 50_000_000

MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES = 2 * 1024 * 1024 * 1024

_FAST_CSV_BYTES_PER_CELL_OVERHEAD = 16

MAX_EXCEL_CELLS = 5_000_000

MAX_SPLIT_GROUPS = 50

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _pandas_line_terminator_parameter(pd) -> str:
    import inspect

    return (
        "lineterminator"
        if "lineterminator" in inspect.signature(pd.DataFrame.to_csv).parameters
        else "line_terminator"
    )

def _write_csv_overwrite(df, path: str, include_header: bool = True) -> bool:
    import os
    import shutil
    import tempfile

    from pyspark.sql import functions as F
    try:
        import pandas as pd
    except ModuleNotFoundError:
        return False

    supported_type_names = {
        "boolean",
        "byte",
        "char",
        "date",
        "decimal",
        "double",
        "float",
        "integer",
        "long",
        "short",
        "string",
        "timestamp",
        "timestamp_ntz",
        "varchar",
        "void",
    }
    normalized_names = [name.casefold() for name in df.columns]
    if (
        not df.columns
        or len(set(normalized_names)) != len(normalized_names)
        or any(field.dataType.typeName() not in supported_type_names for field in df.schema.fields)
    ):
        return False

    spark_output_directory = os.path.isdir(path) and os.path.isfile(
        os.path.join(path, "_SUCCESS")
    )
    if os.path.isdir(path) and not spark_output_directory:
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )

    from databricks.sdk import WorkspaceClient

    client = WorkspaceClient()

    expressions = []
    output_column_names = []
    nan_markers = []
    for index, field in enumerate(df.schema.fields):
        escaped_name = field.name.replace("`", "``")
        column = F.col(f"`{escaped_name}`")
        output_column_name = f"__lb_csv_column_{index}"
        output_column_names.append(field.name)
        type_name = field.dataType.typeName()
        if type_name in {"char", "string", "varchar"}:
            first_character = F.substring(column, 1, 1)
            column = F.when(
                first_character.isin(*_FORMULA_TRIGGERS),
                F.concat(F.lit("'"), column),
            ).otherwise(column)
        elif type_name in {"byte", "integer", "long", "short"}:
            column = column.cast("string")
        elif type_name in {"double", "float"}:
            marker = f"__lb_csv_nan_{index}"
            nan_markers.append((output_column_name, marker))
            expressions.append(F.isnan(column).alias(marker))
        elif type_name == "boolean":
            column = F.when(column, F.lit("True")).when(~column, F.lit("False"))
        elif type_name in {"date", "decimal"}:
            column = column.cast("string")
        elif type_name in {"timestamp", "timestamp_ntz"}:
            column = F.regexp_replace(
                F.date_format(column, "yyyy-MM-dd HH:mm:ss.SSSSSS"),
                r"[.]000000$",
                "",
            )
        expressions.append(column.alias(output_column_name))

    fast_csv_row_cap = min(
        MAX_SINGLE_FILE_ROWS,
        max(1, MAX_FAST_CSV_CELLS // len(df.columns)),
    )
    prepared_df = df.select(*expressions).limit(fast_csv_row_cap + 1).persist()
    try:
        payload_size = F.lit(0)
        for name in prepared_df.columns:
            escaped_name = name.replace("`", "``")
            payload_size = payload_size + F.length(
                F.encode(
                    F.coalesce(F.col(f"`{escaped_name}`").cast("string"), F.lit("")),
                    "UTF-8",
                )
            )
        stats = prepared_df.agg(
            F.count(F.lit(1)).alias("row_count"),
            F.sum(payload_size).alias("payload_bytes"),
        ).collect()[0]
        row_count = stats["row_count"]
        if row_count > fast_csv_row_cap:
            if fast_csv_row_cap < MAX_SINGLE_FILE_ROWS:
                raise ValueError(
                    f"Output: result too wide for safe single-file CSV export "
                    f"(over {MAX_FAST_CSV_CELLS:,} cells) — write to a table instead."
                )
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
        estimated_driver_bytes = (stats["payload_bytes"] or 0) + (
            row_count * len(prepared_df.columns) * _FAST_CSV_BYTES_PER_CELL_OVERHEAD
        )
        if estimated_driver_bytes > MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES:
            raise ValueError(
                f"Output: result too large for safe single-file CSV export "
                f"(estimated driver memory exceeds "
                f"{MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES // (1024 * 1024):,} MiB) — "
                f"write to a table instead."
            )
        pdf = prepared_df.toPandas()
    finally:
        prepared_df.unpersist()
    for column_name, marker in nan_markers:
        nan_mask = pdf[marker].fillna(False)
        finite_mask = ~pdf[column_name].isna()
        pdf[column_name] = pdf[column_name].astype(object)
        pdf.loc[finite_mask, column_name] = pdf.loc[finite_mask, column_name].map(
            lambda value: repr(float(value))
        )
        pdf.loc[nan_mask, column_name] = "nan"
        pdf.loc[~nan_mask & ~finite_mask, column_name] = None
        pdf.drop(columns=[marker], inplace=True)
    pdf.columns = output_column_names
    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        pdf.to_csv(
            local_tmp,
            index=False,
            header=include_header,
            **{_pandas_line_terminator_parameter(pd): "\r\n"},
        )
        if spark_output_directory:
            shutil.rmtree(path)
        with open(local_tmp, "rb") as f:
            client.files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)
    return True

def _file_path(
    catalog: str, schema: str, volume: str, file_name: str, directory_path: str = ""
) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        if directory_path:
            parts = directory_path.split("/")
            if (
                directory_path.startswith("/")
                or directory_path.endswith("/")
                or chr(92) in directory_path
                or any(part in ("", ".", "..") for part in parts)
            ):
                raise ValueError(
                    f"Output: invalid 'directory_path' {directory_path!r}: it must be "
                    "a relative path beneath the selected volume."
                )
        schema_segment = schema.replace(".", "/")
        directory_segment = f"/{directory_path}" if directory_path else ""
        return f"/Volumes/{catalog}/{schema_segment}/{volume}{directory_segment}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _sanitize_split_value(value) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(c for c in text if c not in "/\\").strip()
    cleaned = cleaned.replace("..", ".")
    cleaned = cleaned.strip(". ")
    return cleaned or "none"

def _split_file_name(file_name: str, value, file_type: str, used: set) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = file_name
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    base = f"{stem}_{_sanitize_split_value(value)}"
    candidate = base
    n = 2
    while candidate.lower() in used:
        candidate = f"{base}_{n}"
        n += 1
    used.add(candidate.lower())
    return candidate

_INVALID_SHEET_CHARS = "[]:*?/\\"
_SPLIT_MAP_SHEET = "_lb_split_map"

def _normalize_user_sheet_name(sheet_name: str) -> str:
    if not sheet_name:
        return ""
    cleaned = "".join(
        c for c in sheet_name if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    return cleaned[:31]

def _sanitize_sheet_name(value, used: set) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(
        c for c in text if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    cleaned = cleaned[:31]
    candidate = cleaned
    n = 2
    while candidate.lower() in used:
        tag = f"_{n}"
        candidate = cleaned[: 31 - len(tag)] + tag
        n += 1
    used.add(candidate.lower())
    return candidate

def _split_value_key(value) -> str:
    if value is None:
        return "null"
    if isinstance(value, bool):
        return f"b:{int(value)}"
    if isinstance(value, int):
        return f"i:{value}"
    if isinstance(value, float):
        return f"f:{value}"
    return f"s:{value}"

def _assign_split_names(values, name_for, prior_names=None):
    used: set = set()
    assigned: Dict[int, str] = {}
    priors = list(prior_names) if prior_names is not None else [None] * len(values)
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        prior = priors[i] if i < len(priors) else None
        if prior:
            assigned[i] = prior
            used.add(prior.lower())
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        if i in assigned:
            continue
        assigned[i] = name_for(values[i], used)
    return [assigned[i] for i in range(len(values))]

def _split_sheet_names(values, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(values, _sanitize_sheet_name, prior_names)

def _split_file_names(file_name: str, values, file_type: str, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(
        values,
        lambda value, used: _split_file_name(file_name, value, file_type, used),
        prior_names,
    )

def _load_sheet_split_map(workbook):
    if _SPLIT_MAP_SHEET not in workbook.sheetnames:
        return {}
    mapping = {}
    for row in workbook[_SPLIT_MAP_SHEET].iter_rows(min_row=2, values_only=True):
        if not row or row[0] is None or row[1] is None:
            continue
        mapping[str(row[0])] = str(row[1])
    return mapping

def _write_sheet_split_map(workbook, mapping) -> None:
    if _SPLIT_MAP_SHEET in workbook.sheetnames:
        del workbook[_SPLIT_MAP_SHEET]
    worksheet = workbook.create_sheet(_SPLIT_MAP_SHEET)
    worksheet.sheet_state = "hidden"
    worksheet.append(["value_key", "sheet_name"])
    for key in sorted(mapping):
        worksheet.append([key, mapping[key]])

def _split_manifest_path(directory: str, base_file_name: str, file_type: str) -> str:
    import os

    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    return os.path.join(directory, f"{stem}.lb-split-manifest.json")

def _load_file_split_manifest(directory: str, base_file_name: str, file_type: str):
    import json
    import os

    path = _split_manifest_path(directory, base_file_name, file_type)
    if not os.path.isfile(path):
        return {}
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    groups = payload.get("groups") if isinstance(payload, dict) else None
    if not isinstance(groups, dict):
        return {}
    return {str(k): str(v) for k, v in groups.items()}

def _write_file_split_manifest(directory: str, base_file_name: str, file_type: str, mapping) -> None:
    import json
    import os

    if not directory:
        return
    os.makedirs(directory, exist_ok=True)
    path = _split_manifest_path(directory, base_file_name, file_type)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump({"version": 1, "groups": mapping}, handle, indent=2, sort_keys=True)

def _staged_publisher():
    import os
    import shutil
    import tempfile

    pending: List[Any] = []

    def _discard() -> None:
        for local_tmp, _ in pending:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    def stage_for(dest: str):
        def _stage(write_body) -> None:
            fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
            os.close(fd)
            try:
                write_body(local_tmp)
            except BaseException:
                if os.path.isfile(local_tmp):
                    os.remove(local_tmp)
                raise
            pending.append((local_tmp, dest))

        return _stage

    def publish():
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        remote_staging: List[Any] = []
        try:
            for local_tmp, dest in pending:
                staging_dest = f"{dest}.lb-publishing"
                if os.path.isdir(staging_dest):
                    shutil.rmtree(staging_dest)
                with open(local_tmp, "rb") as f:
                    client.files.upload(staging_dest, f, overwrite=True)
                remote_staging.append((staging_dest, dest))
            for staging_dest, dest in remote_staging:
                if os.path.isdir(dest):
                    shutil.rmtree(dest)
                with open(staging_dest, "rb") as f:
                    client.files.upload(dest, f, overwrite=True)
        finally:
            for staging_dest, _ in remote_staging:
                if os.path.isfile(staging_dest):
                    os.remove(staging_dest)
            _discard()
        return [dest for _, dest in pending]

    return stage_for, publish, _discard

def _remove_stale_split_files(directory: str, base_file_name: str, file_type: str, keep) -> None:
    import os

    if not directory or not os.path.isdir(directory):
        return
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    if not stem:
        return
    prefix = f"{stem}_"
    keep_names = {os.path.basename(p) for p in keep}
    for name in sorted(os.listdir(directory)):
        if name in keep_names:
            continue
        if not name.startswith(prefix) or not name.lower().endswith(suffix.lower()):
            continue
        if len(name) <= len(prefix) + len(suffix):
            continue
        stale = os.path.join(directory, name)
        if os.path.isfile(stale):
            os.remove(stale)

def _align_source_keys(existing_header, columns, safe, where: str):
    pending: Dict[str, List[Any]] = {}
    for c in columns:
        pending.setdefault(str(safe(c)), []).append(c)
    source_keys: List[Any] = []
    for h in existing_header:
        bucket = pending.get(str(h))
        if not bucket:
            raise ValueError(
                f"Output: cannot append to {where}: the existing column {str(h)!r} "
                f"occurs more times than the data provides."
            )
        source_keys.append(bucket.pop(0))
    return source_keys

def _group_rows(rows, split_column: str):
    groups: List[Any] = []
    index: Dict[Any, int] = {}
    for row in rows:
        key = row[split_column]
        if key not in index:
            if len(groups) >= MAX_SPLIT_GROUPS:
                raise ValueError(
                    f"Output: splitting by {split_column!r} would produce more than "
                    f"{MAX_SPLIT_GROUPS} sheets/files. Pick a lower-cardinality column "
                    f"or write to a table instead."
                )
            index[key] = len(groups)
            groups.append((key, []))
        groups[index[key]][1].append(row)
    return groups

def _cell_reference(ref: str):
    from openpyxl.utils.cell import coordinate_to_tuple

    try:
        return coordinate_to_tuple(ref.strip().upper())
    except Exception as exc:
        raise ValueError(
            f"Output: invalid cell reference {ref!r} in the range — use A1 notation."
        ) from exc

def _write_single_file(
    rows, columns, file_type: str, path: str, append: bool, include_header: bool, stage=None
) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    _stage = stage if stage is not None else _upload_stage

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                elif include_header:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        _write_excel_single(
            rows, columns, path, appending, include_header, "", _stage
        )
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _excel_illegal_re():
    import re as _re

    try:
        from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
    except ImportError:
        _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
    return _lb_illegal

def _require_openpyxl():
    try:
        import openpyxl
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Writing Excel files requires the 'openpyxl' package, which "
            "is not installed in this environment. Add "
            "openpyxl==3.1.5 to the notebook environment "
            "and apply it, then run again."
        ) from exc

def _write_excel_single(rows, columns, path, appending, include_header, sheet_name, stage) -> None:
    import io

    _require_openpyxl()
    import openpyxl
    import pandas as pd

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""

    if appending:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            if target_sheet in workbook.sheetnames:
                worksheet = workbook[target_sheet]
            else:
                worksheet = workbook.create_sheet(target_sheet)
        else:
            worksheet = workbook.active

        first = next(worksheet.iter_rows(values_only=True), None)
        if first is not None:
            header = list(first)
            if sorted(str(c) for c in header) != sorted(
                str(_excel_safe(c)) for c in columns
            ):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                    f"file's columns {[str(c) for c in header]}."
                )
            source_keys = _align_source_keys(header, columns, _excel_safe, path)
            existing_rows = max(0, (worksheet.max_row or 0) - 1)
            row_cap = _excel_row_cap(len(header))
            if existing_rows + len(rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(header):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[k])) for k in source_keys]
                )
        else:
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            row_cap = _excel_row_cap(len(columns))
            if len(rows) >= row_cap:
                raise ValueError(
                    f"Output: writing {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(columns):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[c])) for c in columns]
                )

        def _write_excel(tmp_path: str) -> None:
            workbook.save(tmp_path)

        stage(_write_excel)
        return

    header = list(columns)
    source_keys = list(columns)
    combined = [
        [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
    ]
    new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])
    to_excel_kwargs = {"index": False, "engine": "openpyxl", "header": include_header}
    if target_sheet:
        to_excel_kwargs["sheet_name"] = target_sheet

    def _write_excel(tmp_path: str) -> None:
        buffer = io.BytesIO()
        new_df.to_excel(buffer, **to_excel_kwargs)
        with open(tmp_path, "wb") as out:
            out.write(buffer.getvalue())

    stage(_write_excel)

def _write_excel_sheets(groups, columns, path, include_header, append) -> None:
    import os
    import shutil
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    if os.path.isdir(path):
        if os.path.isfile(os.path.join(path, "_SUCCESS")):
            shutil.rmtree(path)
        else:
            raise ValueError(
                f"Output: cannot write file to {path}: a directory already exists at that path."
            )

    appending = append and os.path.isfile(path)

    if appending:
        workbook = openpyxl.load_workbook(path)
    else:
        workbook = openpyxl.Workbook()
        workbook.remove(workbook.active)

    effective_groups = groups if groups else [(None, [])]
    values = [value for value, _ in effective_groups]
    prior_map = _load_sheet_split_map(workbook) if appending else {}
    prior_for_assign = {
        key: name
        for key, name in prior_map.items()
        if name in workbook.sheetnames
    }
    sheet_names = _split_sheet_names(values, prior_for_assign if appending else None)
    updated_map = dict(prior_map) if appending else {}
    for value, target_name in zip(values, sheet_names):
        updated_map[_split_value_key(value)] = target_name
    row_cap = _excel_row_cap(len(columns))
    for (value, group_rows), target_name in zip(effective_groups, sheet_names):
        if appending and target_name in workbook.sheetnames:
            worksheet = workbook[target_name]
            if include_header:
                first = next(worksheet.iter_rows(values_only=True), None)
                if first is not None:
                    existing_header = [str(c) for c in first]
                    if sorted(existing_header) != sorted(str(_excel_safe(c)) for c in columns):
                        raise ValueError(
                            f"Output: cannot append to sheet {target_name!r} in {path}: "
                            f"the data's columns {[str(_excel_safe(c)) for c in columns]} do not "
                            f"match the sheet's columns {existing_header}."
                        )
                    source_keys = _align_source_keys(
                        existing_header, columns, _excel_safe, f"sheet {target_name!r} in {path}"
                    )
                    existing_rows = max(0, (worksheet.max_row or 0) - 1)
                else:
                    source_keys = list(columns)
                    existing_rows = 0
            else:
                source_keys = list(columns)
                existing_rows = worksheet.max_row or 0
            if existing_rows + len(group_rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(group_rows):,} rows would grow sheet "
                    f"{target_name!r} in {path} past the single-file excel export limit "
                    f"({MAX_EXCEL_CELLS:,} cells at {len(columns):,} columns) — write to "
                    f"a table instead."
                )
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[k])) for k in source_keys])
        else:
            worksheet = workbook.create_sheet(target_name)
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[c])) for c in columns])

    if not workbook.sheetnames or workbook.sheetnames == [_SPLIT_MAP_SHEET]:
        workbook.create_sheet("Sheet")
    _write_sheet_split_map(workbook, updated_map)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _write_excel_range(rows, columns, path, sheet_name, range_start, range_end, include_header) -> None:
    import os
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    start_row, start_col = _cell_reference(range_start)
    needed_rows = len(rows) + (1 if include_header else 0)
    needed_cols = len(columns)
    if range_end:
        end_row, end_col = _cell_reference(range_end)
        if end_row < start_row or end_col < start_col:
            raise ValueError(
                f"Output: range end {range_end!r} must be at or after range start {range_start!r}."
            )
        available_rows = end_row - start_row + 1
        available_cols = end_col - start_col + 1
        if needed_rows > available_rows or needed_cols > available_cols:
            raise ValueError(
                f"Output: the data ({len(rows):,} rows x {needed_cols} columns"
                f"{' plus a header row' if include_header else ''}) does not fit the "
                f"range {range_start}:{range_end} ({available_rows} rows x {available_cols} "
                f"columns). Widen the range or write to a table instead."
            )
    else:
        end_row = start_row + max(needed_rows, 1) - 1
        end_col = start_col + max(needed_cols, 1) - 1

    if os.path.isdir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""
    exists = os.path.isfile(path)
    if exists:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            worksheet = (
                workbook[target_sheet]
                if target_sheet in workbook.sheetnames
                else workbook.create_sheet(target_sheet)
            )
        else:
            worksheet = workbook.active
    else:
        workbook = openpyxl.Workbook()
        worksheet = workbook.active
        if target_sheet:
            worksheet.title = target_sheet

    if range_end:
        clear_end_row, clear_end_col = end_row, end_col
    else:
        clear_end_row = max(end_row, worksheet.max_row)
        probe = start_col
        while probe < worksheet.max_column:
            if all(
                worksheet.cell(row=r, column=probe + 1).value is None
                for r in range(start_row, clear_end_row + 1)
            ):
                break
            probe += 1
        clear_end_col = max(end_col, probe)

    from openpyxl.utils.cell import range_boundaries

    removed_merges = []
    for merged in list(worksheet.merged_cells.ranges):
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(str(merged))
        overlaps = (
            m_min_row <= clear_end_row
            and m_max_row >= start_row
            and m_min_col <= clear_end_col
            and m_max_col >= start_col
        )
        if overlaps:
            removed_merges.append(str(merged))
            worksheet.unmerge_cells(str(merged))

    for r in range(start_row, clear_end_row + 1):
        for c in range(start_col, clear_end_col + 1):
            worksheet.cell(row=r, column=c).value = None

    target_row = start_row
    if include_header:
        for offset, column in enumerate(columns):
            worksheet.cell(row=target_row, column=start_col + offset, value=_excel_safe(column))
        target_row += 1
    for row in rows:
        for offset, column in enumerate(columns):
            worksheet.cell(
                row=target_row,
                column=start_col + offset,
                value=_neutralize_formula(_excel_safe(row[column])),
            )
        target_row += 1

    write_end_row = target_row - 1
    write_end_col = start_col + max(needed_cols, 1) - 1
    for merged in removed_merges:
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(merged)
        overlaps_write = (
            m_min_row <= write_end_row
            and m_max_row >= start_row
            and m_min_col <= write_end_col
            and m_max_col >= start_col
        )
        if not overlaps_write:
            worksheet.merge_cells(merged)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _collect_capped(df, file_type: str):
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
        return rows
    rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
    if len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: result too large for single-file export "
            f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
        )
    return rows

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    split_by_column = config.get("split_by_column", "") or ""
    split_mode = config.get("split_mode", "sheets")
    sheet_name = config.get("sheet_name", "") or ""
    include_header = config.get("include_header", True)
    range_start = (config.get("range_start", "") or "").strip()
    range_end = (config.get("range_end", "") or "").strip()
    write_mode = config.get("write_mode", "overwrite")
    columns = df.columns

    if range_end and not range_start:
        raise ValueError(
            "Output: 'range_start' is required to write a cell range "
            "('range_end' alone is not a valid range)."
        )
    if range_start:
        if file_type != "excel":
            raise ValueError("Output: a cell range is only supported for excel file output.")
        if split_by_column:
            raise ValueError(
                "Output: a cell range cannot be combined with split_by_column — "
                "clear one of them."
            )
        if write_mode == "append":
            raise ValueError(
                "Output: a cell range cannot be combined with append — a range writes "
                "into a fixed region, so use write_mode overwrite."
            )

    if split_by_column:
        if split_mode not in ("sheets", "files"):
            raise ValueError(
                f"Output: split_mode {split_mode!r} is not supported. Use 'sheets' or 'files'."
            )
        if split_by_column not in columns:
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} is not a column of the input."
            )
        from pyspark.sql.types import ArrayType, MapType, StructType

        split_type = df.schema[split_by_column].dataType
        if isinstance(split_type, (ArrayType, MapType, StructType)):
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} has type "
                f"{split_type.simpleString()}, which cannot be used to group rows. "
                f"Pick a column with a scalar type (string, number, boolean, date)."
            )
        if split_mode == "sheets" and file_type != "excel":
            raise ValueError(
                "Output: split_mode 'sheets' is only supported for excel file output; "
                "use 'files' for csv/json."
            )

    base_file_name = _with_extension(config.get("file_name", ""), file_type)
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        base_file_name,
        config.get("directory_path", ""),
    )

    if (
        file_type == "csv"
        and not split_by_column
        and write_mode == "overwrite"
        and config.get("__fast_csv_overwrite", False)
        and _write_csv_overwrite(df, path, include_header)
    ):
        return

    rows = _collect_capped(df, file_type)
    append = write_mode == "append"

    if range_start:
        _write_excel_range(
            rows, columns, path, sheet_name, range_start, range_end, include_header
        )
        return

    if split_by_column and split_mode == "sheets":
        groups = _group_rows(rows, split_by_column)
        _write_excel_sheets(groups, columns, path, include_header, append)
        return

    if split_by_column:
        import os

        groups = _group_rows(rows, split_by_column)
        directory = os.path.dirname(path)
        prior_map = (
            _load_file_split_manifest(directory, base_file_name, file_type)
            if append
            else {}
        )
        prior_for_assign = {}
        if append:
            for key, name in prior_map.items():
                candidate = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(name, file_type),
                    config.get("directory_path", ""),
                )
                if os.path.isfile(candidate):
                    prior_for_assign[key] = name
        group_names = _split_file_names(
            config.get("file_name", ""),
            [value for value, _ in groups],
            file_type,
            prior_for_assign if append else None,
        )
        updated_map = dict(prior_map) if append else {}
        for value, group_name in zip([value for value, _ in groups], group_names):
            updated_map[_split_value_key(value)] = group_name
        stage_for, publish, discard = _staged_publisher()
        try:
            for (_, group_rows), group_name in zip(groups, group_names):
                group_path = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(group_name, file_type),
                    config.get("directory_path", ""),
                )
                if file_type == "excel":
                    _write_excel_single_at(
                        group_rows,
                        columns,
                        group_path,
                        append,
                        include_header,
                        sheet_name,
                        stage_for(group_path),
                    )
                else:
                    _write_single_file(
                        group_rows,
                        columns,
                        file_type,
                        group_path,
                        append,
                        include_header,
                        stage_for(group_path),
                    )
        except BaseException:
            discard()
            raise
        written = publish()
        _write_file_split_manifest(directory, base_file_name, file_type, updated_map)
        if not append:
            _remove_stale_split_files(
                directory, base_file_name, file_type, written
            )
        return

    if file_type == "excel":
        _write_excel_single_at(rows, columns, path, append, include_header, sheet_name)
        return
    _write_single_file(rows, columns, file_type, path, append, include_header)

def _write_excel_single_at(rows, columns, path, append, include_header, sheet_name, stage=None) -> None:
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            if os.path.isdir(path) and _is_spark_output_dir(path):
                _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    appending = append and os.path.isfile(path)
    _write_excel_single(
        rows,
        columns,
        path,
        appending,
        include_header,
        sheet_name,
        stage if stage is not None else _upload_stage,
    )

def _read_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    path_literal = json.dumps(path)
    if file_type == "excel":
        sheet_name = (config.get("sheet_name", "") or "").strip()
        range_start = (config.get("range_start", "") or "").strip()
        range_end = (config.get("range_end", "") or "").strip()
        cell_range = (
            f"{range_start}:{range_end}" if range_start and range_end else range_start
        )
        data_address = (
            f"{sheet_name}!{cell_range}" if sheet_name and cell_range else sheet_name or cell_range
        )
        options = ['format=>"excel"']
        if data_address:
            options.append(f"dataAddress=>{json.dumps(data_address)}")
        options.extend(
            [
                f'headerRows=>{1 if config.get("include_header", True) else 0}',
                'schemaEvolutionMode=>"none"',
            ]
        )
        return spark.sql(f"SELECT * FROM read_files({path_literal}, {', '.join(options)})")
    if file_type == "csv":
        header = "true" if config.get("include_header", True) else "false"
        return spark.sql(
            f'SELECT * FROM read_files({path_literal}, format=>"csv", header=>{header})'
        )
    return spark.sql(f'SELECT * FROM read_files({path_literal}, format=>"{file_type}")')

def _display_write_result(write_result) -> None:
    if write_result is not None and globals().get("ld_display_outputs", False):
        display(write_result)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
):
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    return spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        file_type = config.get("file_type", "csv")
        split_col = (config.get("split_by_column") or "").strip()
        if split_col:
            return {"result": df}
        path = _file_path(
            config.get("catalog", ""),
            config.get("schema", ""),
            config.get("volume", ""),
            _with_extension(config.get("file_name", ""), file_type),
            config.get("directory_path", ""),
        )
        return {"result": _read_file_result(path, file_type, config, spark)}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _display_write_result(_run_materialized_view(config, source_view, spark))
        finally:
            spark.catalog.dropTempView(source_view)
        view_full_name = _qualified(catalog, schema, config.get("view_name", ""))
        return {"result": spark.sql(f"SELECT * FROM {view_full_name}")}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            write_result = (
                spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
            )
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            write_result = (
                spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
            )
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            write_result = spark.sql(stmt, args=args) if args else spark.sql(stmt)
        _display_write_result(write_result)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {"result": spark.sql(f"SELECT * FROM {full_name}")}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "vertem",
    "schema": "workshop_dataprep",
    "table_name": "g_resgate_realizado_meta",
    "write_mode": "overwrite"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["select_24.transformed_data"]
}
if config["meta_state"]["is_disabled"]:
    try:
        _lb_disabled_persisted_result = spark.sql("SELECT * FROM vertem.workshop_dataprep.g_resgate_realizado_meta").limit(0)
        _lb_disabled_persisted_result.schema
    except Exception as _lb_disabled_persisted_error:
        _lb_disabled_persisted_condition_getter = getattr(_lb_disabled_persisted_error, "getCondition", None)
        _lb_disabled_persisted_error_class = _lb_disabled_persisted_condition_getter() if callable(_lb_disabled_persisted_condition_getter) else None
        if not _lb_disabled_persisted_error_class:
            _lb_disabled_persisted_legacy_error_class_getter = getattr(_lb_disabled_persisted_error, "getErrorClass", None)
            _lb_disabled_persisted_error_class = _lb_disabled_persisted_legacy_error_class_getter() if callable(_lb_disabled_persisted_legacy_error_class_getter) else None
        if _lb_disabled_persisted_error_class not in ("TABLE_OR_VIEW_NOT_FOUND", "PATH_NOT_FOUND"):
            raise
        _lb_disabled_persisted_input = inputs.get("data")
        _lb_disabled_persisted_result = _lb_disabled_persisted_input.limit(0) if hasattr(_lb_disabled_persisted_input, "limit") else _lb_disabled_persisted_input
    out = {"result": _lb_disabled_persisted_result}
else:
    out = run(config, inputs, spark)
ctx["output_25_1.result"] = out["result"]
if globals().get("ld_display_outputs", False) or "output_25_1" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["output_25_1.result"])
if "output_25_1" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "output_25_1", "counts": {"result": int(ctx["output_25_1.result"].count())}}}, raw=True)